<a href="https://colab.research.google.com/github/worasak2530seed/fast-hybrid-pro/blob/main/Fast_Hybrid_Pro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================================
# FAST HYBRID PRO v2
# CLEAN RESEARCH / BACKTEST NOTEBOOK
# ============================================================

!pip -q install yfinance pandas numpy matplotlib

import warnings
warnings.filterwarnings("ignore")

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from dataclasses import dataclass, asdict

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 100)

print("✅ Setup complete")

In [ ]:

# ============================================================
# 2) CONFIGURATION
# ============================================================

CAPITAL = 100_000

# Maximum risk per trade
MAX_RISK_PCT = 0.02

# Thai board lot
BOARD_LOT = 100

# Minimum score for signal
MIN_SCORE = 3

# Historical data requirement
MIN_BARS = 200

# Backtest warm-up
WARMUP_BARS = 200

# Maximum holding period
MAX_HOLDING_DAYS = 20

# ============================================================
# PRODUCTION STRATEGY — selected from unseen-test validation
# ============================================================
PRODUCTION_MIN_SCORE = 4
PRODUCTION_WEEKLY_CONFIRM = False
PRODUCTION_ATR_MULT = 1.2
PRODUCTION_TARGET2_PCT = 0.15
PRODUCTION_MAX_HOLDING_DAYS = 20

PRODUCTION_SYMBOLS = [
    "TKN.BK",
    "SAPPE.BK",
    "SPA.BK",
    "XO.BK",
    "FORTH.BK",
    "DITTO.BK",
    "ICHI.BK",
    "MASTER.BK",
]


WATCHLIST = sorted(set([
    'DELTA.BK',
    'SNNP.BK',
    'JMART.BK',
    'JMT.BK',
    'FORTH.BK',
    'SPA.BK',
    'AU.BK',
    'SAPPE.BK',
    'ICHI.BK',
    'CBG.BK',
    'KAMART.BK',
    'MOSHI.BK',
    'MEB.BK',
    'MASTER.BK',
    'KLINIQ.BK',
    'TKN.BK',
    'XO.BK',
    'DITTO.BK',
    'PLANB.BK'
]))


@dataclass
class StrategyConfig:

    # --------------------------------------------------------
    # MACD
    # --------------------------------------------------------
    macd_fast: int = 4
    macd_slow: int = 12
    macd_signal: int = 9

    # --------------------------------------------------------
    # STOCHASTIC
    # --------------------------------------------------------
    stoch_k: int = 8
    stoch_smooth: int = 3
    stoch_d: int = 3

    # --------------------------------------------------------
    # DAILY TREND
    # --------------------------------------------------------
    ema_fast: int = 5
    ema_mid: int = 10
    ema_long: int = 75

    # --------------------------------------------------------
    # WEEKLY TREND
    # --------------------------------------------------------
    weekly_ema_fast: int = 5
    weekly_ema_slow: int = 10

    # --------------------------------------------------------
    # ATR
    # --------------------------------------------------------
    atr_length: int = 14
    atr_stop_mult: float = 1.5

    # --------------------------------------------------------
    # BOLLINGER
    # --------------------------------------------------------
    bb_length: int = 20
    bb_mult: float = 2.0

    # --------------------------------------------------------
    # KELTNER
    # --------------------------------------------------------
    kc_mult: float = 1.5

    # --------------------------------------------------------
    # SQUEEZE
    # --------------------------------------------------------
    squeeze_lookback: int = 10

    # --------------------------------------------------------
    # VOLUME
    # --------------------------------------------------------
    volume_ma: int = 5
    volume_ratio_min: float = 1.2

    # --------------------------------------------------------
    # TARGET
    # --------------------------------------------------------
    target1_pct: float = 0.07
    target2_pct: float = 0.15

    # --------------------------------------------------------
    # SCORE
    # --------------------------------------------------------
    min_score: int = 3


# Current canonical research setup
CANONICAL = StrategyConfig(
    macd_fast=4,
    macd_slow=12,
    macd_signal=9
)


# Alternative setup for comparison
FAST_MACD = StrategyConfig(
    macd_fast=6,
    macd_slow=13,
    macd_signal=5
)


print("WATCHLIST:", len(WATCHLIST))

print("\nCANONICAL")
print(asdict(CANONICAL))

print("\nFAST MACD")
print(asdict(FAST_MACD))

In [ ]:

# ============================================================
# 3) DATA ENGINE
# ============================================================

REQUIRED_COLS = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume"
]


def normalize_ohlcv(df):

    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()

    # yfinance can return MultiIndex columns
    if isinstance(out.columns, pd.MultiIndex):
        out.columns = out.columns.get_level_values(0)

    # Remove duplicate columns
    out = out.loc[:, ~out.columns.duplicated()]

    missing = [
        c for c in REQUIRED_COLS
        if c not in out.columns
    ]

    if missing:
        raise ValueError(
            f"Missing OHLCV columns: {missing}"
        )

    out = out[REQUIRED_COLS].copy()

    # Datetime
    out.index = pd.to_datetime(out.index)

    # Remove timezone if present
    if getattr(out.index, "tz", None) is not None:
        out.index = out.index.tz_localize(None)

    # Numeric conversion
    for col in REQUIRED_COLS:
        out[col] = pd.to_numeric(
            out[col],
            errors="coerce"
        )

    # Remove invalid rows
    out = out.dropna(
        subset=[
            "Open",
            "High",
            "Low",
            "Close"
        ]
    )

    # Remove duplicate dates
    out = out[
        ~out.index.duplicated(
            keep="last"
        )
    ]

    # Sort chronologically
    out = out.sort_index()

    return out


def download_stock(
    symbol,
    period="5y"
):

    raw = yf.download(
        symbol,
        period=period,
        interval="1d",
        auto_adjust=False,
        progress=False,
        threads=False
    )

    return normalize_ohlcv(raw)


data = {}
errors = {}


print("🚀 Downloading market data...\n")


for symbol in WATCHLIST:

    try:

        df = download_stock(symbol)

        if len(df) < MIN_BARS:

            errors[symbol] = (
                f"Only {len(df)} bars "
                f"(minimum {MIN_BARS})"
            )

        else:

            data[symbol] = df

            print(
                f"✅ {symbol:<12} "
                f"{len(df):>5} bars"
            )

    except Exception as e:

        errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


print("\n====================================")
print(
    f"Loaded: {len(data)} / "
    f"{len(WATCHLIST)}"
)
print("====================================")


if errors:

    print("\n⚠️ Errors / insufficient data:")

    for symbol, error in errors.items():

        print(
            f"{symbol}: {error}"
        )

In [ ]:

# ============================================================
# 4) DAILY INDICATORS
# ============================================================

def wilder_rma(series, length):
    """
    Wilder RMA
    ใช้แนวคิดเดียวกับ TradingView ta.rma()
    """

    return series.ewm(
        alpha=1 / length,
        adjust=False,
        min_periods=length
    ).mean()


def calculate_atr(df, length=14):

    prev_close = df["Close"].shift(1)

    tr1 = df["High"] - df["Low"]
    tr2 = (df["High"] - prev_close).abs()
    tr3 = (df["Low"] - prev_close).abs()

    true_range = pd.concat(
        [tr1, tr2, tr3],
        axis=1
    ).max(axis=1)

    atr = wilder_rma(
        true_range,
        length
    )

    return true_range, atr


def calculate_daily_indicators(df):

    out = df.copy()

    close = out["Close"]
    high = out["High"]
    low = out["Low"]
    volume = out["Volume"]

    # --------------------------------------------------------
    # EMA 5 / 10 / 75
    # --------------------------------------------------------

    out["EMA5"] = close.ewm(
        span=5,
        adjust=False
    ).mean()

    out["EMA10"] = close.ewm(
        span=10,
        adjust=False
    ).mean()

    out["EMA75"] = close.ewm(
        span=75,
        adjust=False
    ).mean()

    # --------------------------------------------------------
    # MACD 4 / 12 / 9
    # --------------------------------------------------------

    ema_fast = close.ewm(
        span=4,
        adjust=False
    ).mean()

    ema_slow = close.ewm(
        span=12,
        adjust=False
    ).mean()

    out["MACD"] = ema_fast - ema_slow

    out["MACD_SIGNAL"] = out["MACD"].ewm(
        span=9,
        adjust=False
    ).mean()

    out["MACD_HIST"] = (
        out["MACD"] - out["MACD_SIGNAL"]
    )

    # --------------------------------------------------------
    # STOCHASTIC 8 / 3 / 3
    # --------------------------------------------------------

    lowest_8 = low.rolling(
        window=8,
        min_periods=8
    ).min()

    highest_8 = high.rolling(
        window=8,
        min_periods=8
    ).max()

    price_range = highest_8 - lowest_8

    stoch_raw = (
        100
        * (close - lowest_8)
        / price_range.replace(0, np.nan)
    )

    out["STOCH_K"] = stoch_raw.rolling(
        window=3,
        min_periods=3
    ).mean()

    out["STOCH_D"] = out["STOCH_K"].rolling(
        window=3,
        min_periods=3
    ).mean()

    # --------------------------------------------------------
    # ATR 14 — WILDER
    # --------------------------------------------------------

    out["TR"], out["ATR14"] = calculate_atr(
        out,
        length=14
    )

    # --------------------------------------------------------
    # BOLLINGER BAND 20 / 2
    # --------------------------------------------------------

    bb_mid = close.rolling(
        window=20,
        min_periods=20
    ).mean()

    bb_std = close.rolling(
        window=20,
        min_periods=20
    ).std()

    out["BB_MID"] = bb_mid

    out["BB_UPPER"] = (
        bb_mid + (2.0 * bb_std)
    )

    out["BB_LOWER"] = (
        bb_mid - (2.0 * bb_std)
    )

    # --------------------------------------------------------
    # KELTNER CHANNEL
    # --------------------------------------------------------

    out["KC_MID"] = bb_mid

    out["KC_UPPER"] = (
        bb_mid + (1.5 * out["ATR14"])
    )

    out["KC_LOWER"] = (
        bb_mid - (1.5 * out["ATR14"])
    )

    # --------------------------------------------------------
    # SQUEEZE
    # --------------------------------------------------------

    out["SQUEEZE"] = (
        (out["BB_UPPER"] <= out["KC_UPPER"])
        &
        (out["BB_LOWER"] >= out["KC_LOWER"])
    )

    # --------------------------------------------------------
    # SQUEEZE RELEASE
    # --------------------------------------------------------

    out["SQUEEZE_RELEASE"] = (
        (~out["SQUEEZE"])
        &
        (out["SQUEEZE"].shift(1).fillna(False))
    )

    # --------------------------------------------------------
    # BARS SINCE LAST SQUEEZE
    # --------------------------------------------------------

    bars_since_squeeze = []

    last_squeeze_index = None

    for i, is_squeeze in enumerate(
        out["SQUEEZE"].fillna(False)
    ):

        if is_squeeze:

            last_squeeze_index = i
            bars_since_squeeze.append(0)

        elif last_squeeze_index is None:

            bars_since_squeeze.append(np.nan)

        else:

            bars_since_squeeze.append(
                i - last_squeeze_index
            )

    out["BARS_SINCE_SQUEEZE"] = (
        bars_since_squeeze
    )

    out["SQUEEZE_RECENT"] = (
        out["BARS_SINCE_SQUEEZE"] <= 10
    )

    # --------------------------------------------------------
    # VOLUME
    # --------------------------------------------------------

    out["VOLUME_MA5"] = volume.rolling(
        window=5,
        min_periods=5
    ).mean()

    out["VOLUME_RATIO"] = (
        volume
        / out["VOLUME_MA5"].replace(0, np.nan)
    )

    # --------------------------------------------------------
    # CANDLE BODY
    # --------------------------------------------------------

    out["BODY"] = (
        close - out["Open"]
    ).abs()

    out["BODY_ATR_RATIO"] = (
        out["BODY"]
        / out["ATR14"].replace(0, np.nan)
    )

    return out


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

daily_data = {}
indicator_errors = {}

print("📊 Calculating daily indicators...\n")

for symbol, df in data.items():

    try:

        result = calculate_daily_indicators(df)

        daily_data[symbol] = result

        print(
            f"✅ {symbol:<12} "
            f"{len(result):>5} bars"
        )

    except Exception as e:

        indicator_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


print("\n====================================")
print(
    f"Daily indicator data: "
    f"{len(daily_data)} / {len(data)}"
)
print("====================================")


if indicator_errors:

    print("\n⚠️ Indicator errors:")

    for symbol, error in indicator_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# QUICK VALIDATION
# ============================================================

if daily_data:

    sample_symbol = next(iter(daily_data))

    sample = daily_data[sample_symbol]

    print(
        f"\n🔎 Quick validation: {sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "EMA5",
                "EMA10",
                "EMA75",
                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",
                "STOCH_K",
                "STOCH_D",
                "ATR14",
                "SQUEEZE",
                "SQUEEZE_RELEASE",
                "VOLUME_RATIO"
            ]
        ].tail(5)
    )

In [ ]:

# ============================================================
# 5) WEEKLY INDICATORS
# ============================================================

def calculate_weekly_indicators(df):

    daily = df.copy()

    # --------------------------------------------------------
    # Daily -> Weekly
    #
    # Week ends on Friday.
    # --------------------------------------------------------

    weekly = daily.resample("W-FRI").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "Volume": "sum"
    })

    weekly = weekly.dropna(
        subset=[
            "Open",
            "High",
            "Low",
            "Close"
        ]
    )

    close = weekly["Close"]

    # --------------------------------------------------------
    # WEEKLY EMA 5 / 10
    # --------------------------------------------------------

    weekly["W_EMA5"] = close.ewm(
        span=5,
        adjust=False
    ).mean()

    weekly["W_EMA10"] = close.ewm(
        span=10,
        adjust=False
    ).mean()

    # --------------------------------------------------------
    # WEEKLY MACD 4 / 12 / 9
    # --------------------------------------------------------

    weekly_ema_fast = close.ewm(
        span=4,
        adjust=False
    ).mean()

    weekly_ema_slow = close.ewm(
        span=12,
        adjust=False
    ).mean()

    weekly["W_MACD"] = (
        weekly_ema_fast
        - weekly_ema_slow
    )

    weekly["W_MACD_SIGNAL"] = (
        weekly["W_MACD"].ewm(
            span=9,
            adjust=False
        ).mean()
    )

    weekly["W_MACD_HIST"] = (
        weekly["W_MACD"]
        - weekly["W_MACD_SIGNAL"]
    )

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # A weekly candle ending Friday becomes available
    # only from the next trading day.
    #
    # Example:
    #
    # Week ending 2026-09-25
    # -> usable from 2026-09-28
    #
    # This prevents Friday from seeing its own completed
    # weekly candle during the Friday signal.
    # --------------------------------------------------------

    weekly_features = weekly[
        [
            "W_EMA5",
            "W_EMA10",
            "W_MACD",
            "W_MACD_SIGNAL",
            "W_MACD_HIST"
        ]
    ].copy()

    # Move availability date to next Monday.
    #
    # W-FRI + 3 days = Monday
    #
    # If Monday is a market holiday, forward-fill will keep
    # the last completed weekly value until the next trading day.
    weekly_features.index = (
        weekly_features.index
        + pd.Timedelta(days=3)
    )

    # --------------------------------------------------------
    # Map completed weekly values to daily data
    # --------------------------------------------------------

    daily_index = daily.index

    mapped_weekly = weekly_features.reindex(
        daily_index,
        method="ffill"
    )

    # --------------------------------------------------------
    # Combine
    # --------------------------------------------------------

    out = daily.copy()

    for col in mapped_weekly.columns:
        out[col] = mapped_weekly[col]

    return out, weekly


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

weekly_data = {}
combined_data = {}
weekly_errors = {}

print("📅 Calculating weekly indicators...\n")

for symbol, df in daily_data.items():

    try:

        combined, weekly = calculate_weekly_indicators(df)

        combined_data[symbol] = combined
        weekly_data[symbol] = weekly

        print(
            f"✅ {symbol:<12} "
            f"Daily: {len(combined):>5} | "
            f"Weekly: {len(weekly):>4}"
        )

    except Exception as e:

        weekly_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


print("\n====================================")
print(
    f"Weekly indicator data: "
    f"{len(combined_data)} / {len(daily_data)}"
)
print("====================================")


if weekly_errors:

    print("\n⚠️ Weekly indicator errors:")

    for symbol, error in weekly_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# VALIDATION
# ============================================================

if combined_data:

    sample_symbol = next(iter(combined_data))

    sample = combined_data[sample_symbol]

    print(
        f"\n🔎 Weekly validation: {sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "W_EMA5",
                "W_EMA10",
                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST"
            ]
        ].tail(15)
    )

In [ ]:

# ============================================================
# 6) BUILD FEATURE DATASETS
# ============================================================

feature_data = {}

feature_errors = {}


for symbol, df in combined_data.items():

    try:

        out = df.copy()

        # ----------------------------------------------------
        # Basic validity
        # ----------------------------------------------------

        required = [
            "Open",
            "High",
            "Low",
            "Close",
            "Volume",

            "EMA5",
            "EMA10",
            "EMA75",

            "MACD",
            "MACD_SIGNAL",
            "MACD_HIST",

            "STOCH_K",
            "STOCH_D",

            "ATR14",

            "BB_UPPER",
            "BB_LOWER",

            "KC_UPPER",
            "KC_LOWER",

            "SQUEEZE",
            "SQUEEZE_RELEASE",
            "SQUEEZE_RECENT",

            "VOLUME_RATIO",

            "W_EMA5",
            "W_EMA10",

            "W_MACD",
            "W_MACD_SIGNAL",
            "W_MACD_HIST"
        ]

        missing = [
            col
            for col in required
            if col not in out.columns
        ]

        if missing:

            raise ValueError(
                f"Missing columns: {missing}"
            )

        # ----------------------------------------------------
        # Remove rows that cannot produce indicators
        #
        # IMPORTANT:
        # We do NOT drop rows globally before calculating
        # indicators. All indicators have already been
        # calculated using the full historical dataset.
        # ----------------------------------------------------

        out = out.dropna(
            subset=[
                "EMA75",
                "ATR14",
                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",
                "STOCH_K",
                "STOCH_D",
                "W_EMA5",
                "W_EMA10",
                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST",
                "VOLUME_RATIO"
            ]
        )

        # ----------------------------------------------------
        # Ensure chronological order
        # ----------------------------------------------------

        out = out.sort_index()

        # ----------------------------------------------------
        # Ensure unique dates
        # ----------------------------------------------------

        out = out[
            ~out.index.duplicated(
                keep="last"
            )
        ]

        # ----------------------------------------------------
        # Create previous-bar values
        #
        # Used later for signal confirmation.
        # ----------------------------------------------------

        out["PREV_CLOSE"] = (
            out["Close"].shift(1)
        )

        out["PREV_MACD"] = (
            out["MACD"].shift(1)
        )

        out["PREV_MACD_SIGNAL"] = (
            out["MACD_SIGNAL"].shift(1)
        )

        out["PREV_MACD_HIST"] = (
            out["MACD_HIST"].shift(1)
        )

        out["PREV_STOCH_K"] = (
            out["STOCH_K"].shift(1)
        )

        out["PREV_STOCH_D"] = (
            out["STOCH_D"].shift(1)
        )

        # ----------------------------------------------------
        # Daily trend state
        # ----------------------------------------------------

        out["TREND_BULL"] = (
            (out["Close"] > out["EMA5"])
            &
            (out["EMA5"] > out["EMA10"])
            &
            (out["EMA10"] > out["EMA75"])
        )

        # ----------------------------------------------------
        # Weekly trend state
        # ----------------------------------------------------

        out["WEEKLY_TREND_BULL"] = (
            out["W_EMA5"] > out["W_EMA10"]
        )

        # ----------------------------------------------------
        # Daily MACD bullish state
        # ----------------------------------------------------

        out["MACD_BULL"] = (
            out["MACD"] > out["MACD_SIGNAL"]
        )

        out["MACD_HIST_RISING"] = (
            out["MACD_HIST"]
            > out["PREV_MACD_HIST"]
        )

        # ----------------------------------------------------
        # Weekly MACD bullish state
        # ----------------------------------------------------

        out["WEEKLY_MACD_BULL"] = (
            out["W_MACD"] > out["W_MACD_SIGNAL"]
        )

        out["WEEKLY_MACD_HIST_RISING"] = (
            out["W_MACD_HIST"]
            > out["W_MACD_HIST"].shift(1)
        )

        # ----------------------------------------------------
        # Stochastic bullish state
        # ----------------------------------------------------

        out["STOCH_BULL"] = (
            (out["STOCH_K"] > out["STOCH_D"])
            &
            (out["STOCH_K"] < 80)
        )

        # ----------------------------------------------------
        # Volume confirmation
        # ----------------------------------------------------

        out["VOLUME_CONFIRM"] = (
            out["VOLUME_RATIO"]
            >= 1.2
        )

        # ----------------------------------------------------
        # Early-entry / squeeze state
        # ----------------------------------------------------

        out["EARLY_ENTRY_ZONE"] = (
            out["SQUEEZE"]
            |
            out["SQUEEZE_RELEASE"]
            |
            out["SQUEEZE_RECENT"]
        )

        # ----------------------------------------------------
        # Target prices
        # ----------------------------------------------------

        out["TARGET1"] = (
            out["Close"] * 1.07
        )

        out["TARGET2"] = (
            out["Close"] * 1.15
        )

        # ----------------------------------------------------
        # Final dataset
        # ----------------------------------------------------

        feature_data[symbol] = out

    except Exception as e:

        feature_errors[symbol] = str(e)


print("📦 Building feature datasets...\n")

for symbol, df in feature_data.items():

    print(
        f"✅ {symbol:<12} "
        f"{len(df):>5} usable bars"
    )


print("\n====================================")
print(
    f"Feature datasets: "
    f"{len(feature_data)} / {len(combined_data)}"
)
print("====================================")


if feature_errors:

    print("\n⚠️ Feature errors:")

    for symbol, error in feature_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# VALIDATION
# ============================================================

if feature_data:

    sample_symbol = next(iter(feature_data))

    sample = feature_data[sample_symbol]

    print(
        f"\n🔎 Feature validation: {sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "EMA5",
                "EMA10",
                "EMA75",
                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",
                "W_EMA5",
                "W_EMA10",
                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST",
                "TREND_BULL",
                "WEEKLY_TREND_BULL",
                "MACD_BULL",
                "STOCH_BULL",
                "VOLUME_CONFIRM",
                "EARLY_ENTRY_ZONE"
            ]
        ].tail(10)
    )

In [ ]:

# ============================================================
# 7) LEGACY SCORE
# ============================================================
#
# วัตถุประสงค์:
# - เก็บสูตร FAST HYBRID PRO เดิมไว้เป็น Baseline
# - ใช้เปรียบเทียบกับ Canonical Research Score ใน Cell 8
# - ยังไม่ทำ Backtest และยังไม่เปิด Position
#
# LEGACY 5-POINT SCORE
# 1. MACD 6/13/5 + Histogram Rising
# 2. Stochastic 8/3/3 : K > D และ K < 80
# 3. Trend: Close > EMA20 > EMA50
# 4. Candle Body / ATR >= 0.8
# 5. Volatility: BB breakout OR squeeze release
#
# หมายเหตุ:
# - Cell 4 ใช้ EMA5/10/75 และ MACD 4/12/9 เป็น canonical data
# - ดังนั้น Legacy MACD จะคำนวณใหม่ตรงนี้ด้วย 6/13/5
# - EMA20/EMA50 ก็จะคำนวณใหม่เพื่อรักษาสูตร Legacy เดิม
# ============================================================


def calculate_legacy_score(df):
    out = df.copy()

    # --------------------------------------------------------
    # 1) LEGACY MACD 6/13/5
    # --------------------------------------------------------
    close = out["Close"]

    legacy_ema_fast = close.ewm(
        span=6,
        adjust=False
    ).mean()

    legacy_ema_slow = close.ewm(
        span=13,
        adjust=False
    ).mean()

    out["LEGACY_MACD"] = (
        legacy_ema_fast - legacy_ema_slow
    )

    out["LEGACY_MACD_SIGNAL"] = (
        out["LEGACY_MACD"]
        .ewm(span=5, adjust=False)
        .mean()
    )

    out["LEGACY_MACD_HIST"] = (
        out["LEGACY_MACD"]
        - out["LEGACY_MACD_SIGNAL"]
    )

    out["LEGACY_MACD_HIST_PREV"] = (
        out["LEGACY_MACD_HIST"].shift(1)
    )

    out["LEGACY_MACD_BULL"] = (
        (out["LEGACY_MACD"] > out["LEGACY_MACD_SIGNAL"])
        &
        (
            out["LEGACY_MACD_HIST"]
            > out["LEGACY_MACD_HIST_PREV"]
        )
    )


    # --------------------------------------------------------
    # 2) LEGACY STOCHASTIC 8/3/3
    # --------------------------------------------------------
    lowest_8 = out["Low"].rolling(
        window=8,
        min_periods=8
    ).min()

    highest_8 = out["High"].rolling(
        window=8,
        min_periods=8
    ).max()

    price_range = (
        highest_8 - lowest_8
    ).replace(0, np.nan)

    stoch_raw = (
        100
        * (close - lowest_8)
        / price_range
    )

    out["LEGACY_STOCH_K"] = (
        stoch_raw
        .rolling(window=3, min_periods=3)
        .mean()
    )

    out["LEGACY_STOCH_D"] = (
        out["LEGACY_STOCH_K"]
        .rolling(window=3, min_periods=3)
        .mean()
    )

    out["LEGACY_STOCH_BULL"] = (
        (out["LEGACY_STOCH_K"] > out["LEGACY_STOCH_D"])
        &
        (out["LEGACY_STOCH_K"] < 80)
    )


    # --------------------------------------------------------
    # 3) LEGACY TREND
    # Close > EMA20 > EMA50
    # --------------------------------------------------------
    out["LEGACY_EMA20"] = (
        close.ewm(
            span=20,
            adjust=False
        ).mean()
    )

    out["LEGACY_EMA50"] = (
        close.ewm(
            span=50,
            adjust=False
        ).mean()
    )

    out["LEGACY_TREND_BULL"] = (
        (close > out["LEGACY_EMA20"])
        &
        (out["LEGACY_EMA20"] > out["LEGACY_EMA50"])
    )


    # --------------------------------------------------------
    # 4) CANDLE BODY / ATR >= 0.8
    # --------------------------------------------------------
    #
    # ใช้ ATR14 จาก Cell 4
    # Body = abs(Close - Open)
    #
    # เงื่อนไขเดิม:
    # Body / ATR >= 0.8
    # --------------------------------------------------------

    out["LEGACY_BODY_ATR_RATIO"] = (
        (out["Close"] - out["Open"]).abs()
        / out["ATR14"].replace(0, np.nan)
    )

    out["LEGACY_CANDLE_BULL"] = (
        out["LEGACY_BODY_ATR_RATIO"] >= 0.8
    )


    # --------------------------------------------------------
    # 5) VOLATILITY
    # BB Breakout OR Squeeze Release
    # --------------------------------------------------------
    #
    # BB breakout:
    # Close > Upper BB
    #
    # Squeeze release:
    # SQUEEZE วันนี้ False
    # และเมื่อวานเป็น True
    # --------------------------------------------------------

    out["LEGACY_BB_BREAKOUT"] = (
        out["Close"] > out["BB_UPPER"]
    )

    out["LEGACY_SQUEEZE_RELEASE"] = (
        (~out["SQUEEZE"])
        &
        (out["SQUEEZE"].shift(1).fillna(False))
    )

    out["LEGACY_VOLATILITY_BULL"] = (
        out["LEGACY_BB_BREAKOUT"]
        |
        out["LEGACY_SQUEEZE_RELEASE"]
    )


    # --------------------------------------------------------
    # LEGACY SCORE
    # --------------------------------------------------------

    score_components = [
        "LEGACY_MACD_BULL",
        "LEGACY_STOCH_BULL",
        "LEGACY_TREND_BULL",
        "LEGACY_CANDLE_BULL",
        "LEGACY_VOLATILITY_BULL",
    ]

    out["LEGACY_SCORE"] = (
        out[score_components]
        .fillna(False)
        .astype(int)
        .sum(axis=1)
    )


    # --------------------------------------------------------
    # Signal classification
    # --------------------------------------------------------
    #
    # Legacy เดิมใช้ 5 คะแนนเต็ม
    #
    # >= 3 = ผ่าน baseline
    # < 3  = ไม่ผ่าน
    # --------------------------------------------------------

    out["LEGACY_SIGNAL"] = (
        out["LEGACY_SCORE"] >= 3
    )


    return out


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

legacy_data = {}
legacy_errors = {}

print("🧮 Calculating Legacy Score...\n")

for symbol, df in feature_data.items():
    try:
        result = calculate_legacy_score(df)

        legacy_data[symbol] = result

        print(
            f"✅ {symbol:<12} "
            f"{len(result):>5} bars"
        )

    except Exception as e:
        legacy_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print(
    f"Legacy datasets: "
    f"{len(legacy_data)} / {len(feature_data)}"
)
print("====================================")


if legacy_errors:
    print("\n⚠️ Legacy calculation errors:")

    for symbol, error in legacy_errors.items():
        print(f"{symbol}: {error}")


# ============================================================
# VALIDATION
# ============================================================

if legacy_data:

    sample_symbol = next(iter(legacy_data))
    sample = legacy_data[sample_symbol]

    print(
        f"\n🔎 Legacy validation: "
        f"{sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "LEGACY_EMA20",
                "LEGACY_EMA50",

                "LEGACY_MACD",
                "LEGACY_MACD_SIGNAL",
                "LEGACY_MACD_HIST",

                "LEGACY_STOCH_K",
                "LEGACY_STOCH_D",

                "LEGACY_BODY_ATR_RATIO",

                "LEGACY_BB_BREAKOUT",
                "LEGACY_SQUEEZE_RELEASE",

                "LEGACY_MACD_BULL",
                "LEGACY_STOCH_BULL",
                "LEGACY_TREND_BULL",
                "LEGACY_CANDLE_BULL",
                "LEGACY_VOLATILITY_BULL",

                "LEGACY_SCORE",
                "LEGACY_SIGNAL",
            ]
        ].tail(10)
    )


# ============================================================
# SCORE DISTRIBUTION
# ============================================================

if legacy_data:

    all_scores = pd.concat(
        [
            df["LEGACY_SCORE"]
            for df in legacy_data.values()
        ]
    )

    print("\n📊 Legacy Score Distribution")

    display(
        all_scores
        .value_counts()
        .sort_index()
        .rename("bars")
        .to_frame()
    )

    print(
        f"\nBars with Legacy Score >= 3: "
        f"{int((all_scores >= 3).sum()):,}"
    )

    print(
        f"Total usable bars: "
        f"{len(all_scores):,}"
    )

    print(
        f"Signal rate: "
        f"{(all_scores >= 3).mean() * 100:.2f}%"
    )

In [ ]:

# ============================================================
# 8) CANONICAL RESEARCH SCORE
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
# สร้างคะแนนระบบใหม่เพื่อใช้เปรียบเทียบกับ Legacy Score
#
# CANONICAL INDICATORS
#   DAY MACD      = 4 / 12 / 9
#   STOCHASTIC    = 8 / 3 / 3
#   DAY EMA       = 5 / 10 / 75
#   WEEK EMA      = 5 / 10
#   WEEK MACD     = 4 / 12 / 9
#
# RESEARCH SCORE = 5 POINTS
#
# 1) MACD
#    DAY MACD > Signal
#    AND Histogram > 0
#
# 2) STOCHASTIC
#    K > D
#    AND K < 80
#
# 3) DAY TREND
#    Close > EMA5 > EMA10 > EMA75
#
# 4) EARLY ENTRY
#    Squeeze OR Squeeze Release
#
# 5) VOLUME
#    Volume Ratio >= 1.2
#
# PASS:
#    Score >= 3
#
# IMPORTANT
# ------------------------------------------------------------
# Cell นี้เป็น Research Baseline
# ไม่ใช่ Production Weighted Score
# ============================================================


def calculate_canonical_research_score(df):

    out = df.copy()

    # --------------------------------------------------------
    # 1) DAY MACD 4/12/9
    # --------------------------------------------------------

    out["CANONICAL_MACD_BULL"] = (
        (out["MACD"] > out["MACD_SIGNAL"])
        &
        (out["MACD_HIST"] > 0)
    )


    # --------------------------------------------------------
    # 2) STOCHASTIC 8/3/3
    # --------------------------------------------------------

    out["CANONICAL_STOCH_BULL"] = (
        (out["STOCH_K"] > out["STOCH_D"])
        &
        (out["STOCH_K"] < 80)
    )


    # --------------------------------------------------------
    # 3) DAY TREND
    #
    # Close > EMA5 > EMA10 > EMA75
    # --------------------------------------------------------

    out["CANONICAL_TREND_BULL"] = (
        (out["Close"] > out["EMA5"])
        &
        (out["EMA5"] > out["EMA10"])
        &
        (out["EMA10"] > out["EMA75"])
    )


    # --------------------------------------------------------
    # 4) EARLY ENTRY
    #
    # ใช้ Squeeze หรือ Squeeze Release
    #
    # ไม่ใช้ SQUEEZE_RECENT ในคะแนนหลัก
    # เพื่อไม่ให้ signal window กว้างเกินไป
    # --------------------------------------------------------

    out["CANONICAL_EARLY_ENTRY"] = (
        out["SQUEEZE"]
        |
        out["SQUEEZE_RELEASE"]
    )


    # --------------------------------------------------------
    # 5) VOLUME CONFIRMATION
    # --------------------------------------------------------

    out["CANONICAL_VOLUME_CONFIRM"] = (
        out["VOLUME_RATIO"] >= 1.2
    )


    # --------------------------------------------------------
    # SCORE COMPONENTS
    # --------------------------------------------------------

    canonical_components = [
        "CANONICAL_MACD_BULL",
        "CANONICAL_STOCH_BULL",
        "CANONICAL_TREND_BULL",
        "CANONICAL_EARLY_ENTRY",
        "CANONICAL_VOLUME_CONFIRM",
    ]

    out["CANONICAL_SCORE"] = (
        out[canonical_components]
        .fillna(False)
        .astype(int)
        .sum(axis=1)
    )


    # --------------------------------------------------------
    # SIGNAL
    # --------------------------------------------------------

    out["CANONICAL_SIGNAL"] = (
        out["CANONICAL_SCORE"] >= 3
    )


    # --------------------------------------------------------
    # ADDITIONAL INFORMATION
    # --------------------------------------------------------
    #
    # Weekly confirmation ไม่ถูกนับเป็นคะแนนเพิ่ม
    # ใน Research Score
    #
    # เก็บไว้เพื่อใช้วิเคราะห์ภายหลัง
    # --------------------------------------------------------

    out["CANONICAL_WEEKLY_TREND"] = (
        out["W_EMA5"] > out["W_EMA10"]
    )

    out["CANONICAL_WEEKLY_MACD"] = (
        out["W_MACD"] > out["W_MACD_SIGNAL"]
    )

    out["CANONICAL_WEEKLY_CONFIRM"] = (
        out["CANONICAL_WEEKLY_TREND"]
        &
        out["CANONICAL_WEEKLY_MACD"]
    )


    return out


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

canonical_data = {}
canonical_errors = {}

print("🧮 Calculating Canonical Research Score...\n")

for symbol, df in feature_data.items():

    try:

        result = calculate_canonical_research_score(df)

        canonical_data[symbol] = result

        print(
            f"✅ {symbol:<12} "
            f"{len(result):>5} bars"
        )

    except Exception as e:

        canonical_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print(
    f"Canonical datasets: "
    f"{len(canonical_data)} / {len(feature_data)}"
)
print("====================================")


if canonical_errors:

    print("\n⚠️ Canonical calculation errors:")

    for symbol, error in canonical_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# VALIDATION
# ============================================================

if canonical_data:

    sample_symbol = next(iter(canonical_data))

    sample = canonical_data[sample_symbol]

    print(
        f"\n🔎 Canonical validation: "
        f"{sample_symbol}"
    )

    display(
        sample[
            [
                "Close",

                "EMA5",
                "EMA10",
                "EMA75",

                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",

                "STOCH_K",
                "STOCH_D",

                "W_EMA5",
                "W_EMA10",

                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST",

                "VOLUME_RATIO",

                "SQUEEZE",
                "SQUEEZE_RELEASE",

                "CANONICAL_MACD_BULL",
                "CANONICAL_STOCH_BULL",
                "CANONICAL_TREND_BULL",
                "CANONICAL_EARLY_ENTRY",
                "CANONICAL_VOLUME_CONFIRM",

                "CANONICAL_SCORE",
                "CANONICAL_SIGNAL",

                "CANONICAL_WEEKLY_TREND",
                "CANONICAL_WEEKLY_MACD",
                "CANONICAL_WEEKLY_CONFIRM",
            ]
        ].tail(10)
    )


# ============================================================
# SCORE DISTRIBUTION
# ============================================================

if canonical_data:

    all_canonical_scores = pd.concat(
        [
            df["CANONICAL_SCORE"]
            for df in canonical_data.values()
        ]
    )

    print("\n📊 Canonical Score Distribution")

    display(
        all_canonical_scores
        .value_counts()
        .sort_index()
        .rename("bars")
        .to_frame()
    )


    signal_count = int(
        (all_canonical_scores >= 3).sum()
    )

    total_bars = len(
        all_canonical_scores
    )

    signal_rate = (
        signal_count / total_bars * 100
        if total_bars > 0
        else 0
    )


    print(
        f"\nBars with Canonical Score >= 3: "
        f"{signal_count:,}"
    )

    print(
        f"Total usable bars: "
        f"{total_bars:,}"
    )

    print(
        f"Canonical signal rate: "
        f"{signal_rate:.2f}%"
    )


# ============================================================
# LEGACY vs CANONICAL QUICK COMPARISON
# ============================================================

if legacy_data and canonical_data:

    legacy_scores = pd.concat(
        [
            df["LEGACY_SCORE"]
            for df in legacy_data.values()
        ]
    )

    canonical_scores = pd.concat(
        [
            df["CANONICAL_SCORE"]
            for df in canonical_data.values()
        ]
    )

    print("\n====================================")
    print("LEGACY vs CANONICAL")
    print("====================================")

    print(
        f"Legacy signal rate:     "
        f"{(legacy_scores >= 3).mean() * 100:.2f}%"
    )

    print(
        f"Canonical signal rate:  "
        f"{(canonical_scores >= 3).mean() * 100:.2f}%"
    )

In [ ]:

# ============================================================
# 9) MONEY MANAGEMENT — FINAL
# ============================================================

INITIAL_CAPITAL = 100_000.0
MAX_RISK_PCT = 0.02
RISK_PER_TRADE = INITIAL_CAPITAL * MAX_RISK_PCT

LOT_SIZE = 100
ATR_STOP_MULT = 1.5
RECENT_LOW_LOOKBACK = 5


print("💰 Money Management Configuration")
print("------------------------------------")
print(f"Initial Capital : {INITIAL_CAPITAL:,.2f} THB")
print(f"Risk / Trade    : {MAX_RISK_PCT * 100:.2f}%")
print(f"Risk Amount     : {RISK_PER_TRADE:,.2f} THB")
print(f"Lot Size        : {LOT_SIZE:,} shares")
print(f"ATR Stop Mult   : {ATR_STOP_MULT:.2f}x")
print(f"Recent Low      : {RECENT_LOW_LOOKBACK} bars")


# ============================================================
# STOP PRICE
# ============================================================

def calculate_stop_price(
    entry_price,
    atr,
    recent_low,
    atr_multiplier=ATR_STOP_MULT,
):
    if pd.isna(entry_price) or entry_price <= 0:
        return np.nan

    if pd.isna(atr) or atr <= 0:
        return np.nan

    if pd.isna(recent_low) or recent_low <= 0:
        return np.nan

    atr_stop = entry_price - atr * atr_multiplier

    # Conservative stop:
    # choose the LOWER of ATR stop and recent low
    stop_price = min(atr_stop, recent_low)

    if stop_price <= 0:
        return np.nan

    if stop_price >= entry_price:
        return np.nan

    return float(stop_price)


# ============================================================
# POSITION SIZE
# ============================================================

def calculate_position_size(
    capital,
    entry_price,
    stop_price,
    risk_pct=MAX_RISK_PCT,
    lot_size=LOT_SIZE,
):
    if pd.isna(capital) or capital <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "INVALID_CAPITAL",
        }

    if pd.isna(entry_price) or entry_price <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "INVALID_ENTRY",
        }

    if pd.isna(stop_price) or stop_price <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "INVALID_STOP",
        }

    if stop_price >= entry_price:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "STOP_NOT_BELOW_ENTRY",
        }

    max_risk = capital * risk_pct
    risk_per_share = entry_price - stop_price

    if risk_per_share <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": float(risk_per_share),
            "position_value": 0.0,
            "status": "INVALID_RISK_PER_SHARE",
        }

    # --------------------------------------------------------
    # Limit #1: risk-based position size
    # --------------------------------------------------------

    risk_lots = int(
        (max_risk / risk_per_share) // lot_size
    )

    # --------------------------------------------------------
    # Limit #2: capital-based position size
    # --------------------------------------------------------

    capital_lots = int(
        (capital / entry_price) // lot_size
    )

    # Final lots must satisfy BOTH constraints
    lots = min(risk_lots, capital_lots)

    shares = lots * lot_size

    if shares < lot_size:
        if capital_lots < 1:
            status = "INSUFFICIENT_CAPITAL_FOR_ONE_LOT"
        else:
            status = "ONE_LOT_EXCEEDS_RISK"

        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": float(risk_per_share),
            "position_value": 0.0,
            "status": status,
        }

    actual_risk = shares * risk_per_share
    position_value = shares * entry_price

    if capital_lots < risk_lots:
        status = "CAPITAL_LIMITED"
    else:
        status = "OK"

    return {
        "shares": int(shares),
        "lots": int(lots),
        "risk_amount": float(actual_risk),
        "risk_per_share": float(risk_per_share),
        "position_value": float(position_value),
        "status": status,
    }


# ============================================================
# COMPLETE TRADE PLAN
# ============================================================

def build_trade_plan(
    capital,
    entry_price,
    atr,
    recent_low,
    risk_pct=MAX_RISK_PCT,
    atr_multiplier=ATR_STOP_MULT,
    lot_size=LOT_SIZE,
):
    stop_price = calculate_stop_price(
        entry_price,
        atr,
        recent_low,
        atr_multiplier,
    )

    if pd.isna(stop_price):
        return {
            "entry_price": (
                float(entry_price)
                if pd.notna(entry_price)
                else np.nan
            ),
            "stop_price": np.nan,
            "shares": 0,
            "lots": 0,
            "risk_per_share": np.nan,
            "risk_amount": 0.0,
            "position_value": 0.0,
            "status": "INVALID_STOP",
        }

    position = calculate_position_size(
        capital,
        entry_price,
        stop_price,
        risk_pct,
        lot_size,
    )

    return {
        "entry_price": float(entry_price),
        "stop_price": float(stop_price),
        "shares": position["shares"],
        "lots": position["lots"],
        "risk_per_share": position["risk_per_share"],
        "risk_amount": position["risk_amount"],
        "position_value": position["position_value"],
        "status": position["status"],
    }


# ============================================================
# RECENT LOW
# ============================================================

for symbol, df in canonical_data.items():
    df["RECENT_LOW_5"] = (
        df["Low"]
        .rolling(
            window=RECENT_LOW_LOOKBACK,
            min_periods=RECENT_LOW_LOOKBACK,
        )
        .min()
    )


# ============================================================
# VALIDATION
# ============================================================

print("\n====================================")
print("🧪 Money Management Validation")
print("====================================")


# ------------------------------------------------------------
# Test 1: Normal trade
# ------------------------------------------------------------

test_entry = 50.00
test_atr = 1.50
test_recent_low = 47.00

test_plan = build_trade_plan(
    INITIAL_CAPITAL,
    test_entry,
    test_atr,
    test_recent_low,
)

print("\nTest 1: Normal Trade")
print("------------------------------------")

for key, value in test_plan.items():
    if isinstance(value, float):
        print(f"{key:<20}: {value:,.4f}")
    else:
        print(f"{key:<20}: {value}")


# ------------------------------------------------------------
# Test 2: Stop validation
# ------------------------------------------------------------

test_entry = 10.00
test_atr = 0.50
test_recent_low = 9.80

test_plan_2 = build_trade_plan(
    INITIAL_CAPITAL,
    test_entry,
    test_atr,
    test_recent_low,
)

print("\nTest 2: Stop Validation")
print("------------------------------------")
print(f"Entry : {test_plan_2['entry_price']:.4f}")
print(f"Stop  : {test_plan_2['stop_price']:.4f}")

if pd.notna(test_plan_2["stop_price"]):
    print(
        "Valid : "
        f"{test_plan_2['stop_price'] < test_plan_2['entry_price']}"
    )
else:
    print("Valid : False")


# ------------------------------------------------------------
# Test 3: One lot risk check
# ------------------------------------------------------------

test_entry = 80.00
test_stop = 60.00

test_position = calculate_position_size(
    INITIAL_CAPITAL,
    test_entry,
    test_stop,
)

print("\nTest 3: One Lot Risk Check")
print("------------------------------------")
print(f"Entry       : {test_entry:.2f}")
print(f"Stop        : {test_stop:.2f}")
print(f"Risk / Share: {test_position['risk_per_share']:.2f}")
print(f"Shares      : {test_position['shares']:,}")
print(f"Lots        : {test_position['lots']:,}")
print(f"Risk Amount : {test_position['risk_amount']:,.2f}")
print(f"Position    : {test_position['position_value']:,.2f}")
print(f"Status      : {test_position['status']}")


# ------------------------------------------------------------
# Test 4: Capital limit
# ------------------------------------------------------------

test_entry = 1.00
test_stop = 0.99

test_position_4 = calculate_position_size(
    INITIAL_CAPITAL,
    test_entry,
    test_stop,
)

print("\nTest 4: Capital Limit Check")
print("------------------------------------")
print(f"Entry       : {test_entry:.2f}")
print(f"Stop        : {test_stop:.2f}")
print(f"Risk / Share: {test_position_4['risk_per_share']:.4f}")
print(f"Shares      : {test_position_4['shares']:,}")
print(f"Lots        : {test_position_4['lots']:,}")
print(f"Risk Amount : {test_position_4['risk_amount']:,.2f}")
print(f"Position    : {test_position_4['position_value']:,.2f}")
print(f"Status      : {test_position_4['status']}")

assert (
    test_position_4["position_value"]
    <= INITIAL_CAPITAL + 1e-9
), "Position value exceeds capital"


# ============================================================
# REAL DATA SANITY CHECK
# ============================================================

print("\n====================================")
print("🔎 Real Data Sanity Check")
print("====================================")

if canonical_data:
    sample_symbol = next(iter(canonical_data))
    sample = canonical_data[sample_symbol]

    valid_rows = (
        sample[
            ["Close", "ATR14", "RECENT_LOW_5"]
        ]
        .dropna()
        .tail(3)
    )

    print(f"\nSample: {sample_symbol}")
    display(valid_rows)


# ============================================================
# FINAL CHECKS
# ============================================================

assert INITIAL_CAPITAL > 0
assert 0 < MAX_RISK_PCT <= 0.05
assert LOT_SIZE > 0
assert ATR_STOP_MULT > 0
assert RECENT_LOW_LOOKBACK > 0

print("\n====================================")
print("✅ Money Management Cell Ready")
print("====================================")
print(f"Maximum risk per trade: {RISK_PER_TRADE:,.2f} THB")
print("Stop rule: min(Entry - ATR × 1.5, Recent Low)")
print("Position size: whole 100-share lots, rounded DOWN")
print("Position limit: Risk limit + Capital limit")
print("Target prices: calculated later from actual Entry")

In [ ]:

# ============================================================
# 10) CURRENT SCANNER
# ============================================================

print("🔎 CURRENT SCANNER")
print("=" * 90)


def build_current_scan_row(symbol, df):
    if df is None or df.empty:
        return {
            "Symbol": symbol,
            "Date": pd.NaT,
            "Close": np.nan,
            "Score": np.nan,
            "Signal": False,
            "WeeklyConfirm": False,
            "MACD": np.nan,
            "MACD_Signal": np.nan,
            "MACD_Hist": np.nan,
            "EMA5": np.nan,
            "EMA10": np.nan,
            "EMA75": np.nan,
            "W_EMA5": np.nan,
            "W_EMA10": np.nan,
            "W_MACD": np.nan,
            "W_MACD_Signal": np.nan,
            "ATR14": np.nan,
            "RecentLow5": np.nan,
            "Stop": np.nan,
            "RiskPerShare": np.nan,
            "Shares": 0,
            "Lots": 0,
            "RiskAmount": 0.0,
            "PositionValue": 0.0,
            "Target1": np.nan,
            "Target2": np.nan,
            "Status": "NO_DATA",
        }

    row = df.iloc[-1]

    entry_price = row["Close"]
    atr = row["ATR14"]
    recent_low = row["RECENT_LOW_5"]

    trade_plan = build_trade_plan(
        INITIAL_CAPITAL,
        entry_price,
        atr,
        recent_low,
        atr_multiplier=PRODUCTION_ATR_MULT,
    )

    score = row.get("CANONICAL_SCORE", np.nan)
    signal = (
        symbol in PRODUCTION_SYMBOLS
        and float(row.get("CANONICAL_SCORE", 0)) >= PRODUCTION_MIN_SCORE
        and (
            not PRODUCTION_WEEKLY_CONFIRM
            or bool(row.get("CANONICAL_WEEKLY_CONFIRM", False))
        )
    )

    weekly_trend = bool(
        row.get("CANONICAL_WEEKLY_TREND", False)
    )

    weekly_macd = bool(
        row.get("CANONICAL_WEEKLY_MACD", False)
    )

    weekly_confirm = bool(
        row.get("CANONICAL_WEEKLY_CONFIRM", False)
    )

    target1 = (
        float(entry_price * 1.07)
        if pd.notna(entry_price)
        else np.nan
    )

    target2 = (
        float(entry_price * (1 + PRODUCTION_TARGET2_PCT))
        if pd.notna(entry_price)
        else np.nan
    )

    return {
        "Symbol": symbol,
        "Date": df.index[-1],
        "Close": float(entry_price),
        "Score": score,
        "Signal": signal,
        "WeeklyTrend": weekly_trend,
        "WeeklyMACD": weekly_macd,
        "WeeklyConfirm": weekly_confirm,
        "MACD": row["MACD"],
        "MACD_Signal": row["MACD_SIGNAL"],
        "MACD_Hist": row["MACD_HIST"],
        "EMA5": row["EMA5"],
        "EMA10": row["EMA10"],
        "EMA75": row["EMA75"],
        "W_EMA5": row["W_EMA5"],
        "W_EMA10": row["W_EMA10"],
        "W_MACD": row["W_MACD"],
        "W_MACD_Signal": row["W_MACD_SIGNAL"],
        "ATR14": row["ATR14"],
        "RecentLow5": row["RECENT_LOW_5"],
        "Stop": trade_plan["stop_price"],
        "RiskPerShare": trade_plan["risk_per_share"],
        "Shares": trade_plan["shares"],
        "Lots": trade_plan["lots"],
        "RiskAmount": trade_plan["risk_amount"],
        "PositionValue": trade_plan["position_value"],
        "Target1": target1,
        "Target2": target2,
        "Status": trade_plan["status"],
    }


# ============================================================
# BUILD CURRENT SCAN
# ============================================================

current_scan_rows = []

for symbol, df in canonical_data.items():
    current_scan_rows.append(
        build_current_scan_row(symbol, df)
    )

current_scan = pd.DataFrame(current_scan_rows)


# ============================================================
# SORT
# ============================================================

current_scan = current_scan.sort_values(
    by=["Signal", "Score", "WeeklyConfirm"],
    ascending=[False, False, False],
    na_position="last",
).reset_index(drop=True)


# ============================================================
# DISPLAY
# ============================================================

print("\n📊 Current Market Snapshot")
print("-" * 90)

display(
    current_scan[
        [
            "Symbol",
            "Date",
            "Close",
            "Score",
            "Signal",
            "WeeklyConfirm",
            "ATR14",
            "Stop",
            "Shares",
            "RiskAmount",
            "Target1",
            "Target2",
            "Status",
        ]
    ]
)


# ============================================================
# SIGNAL SUMMARY
# ============================================================

signal_scan = current_scan[
    current_scan["Signal"] == True
].copy()

weekly_confirmed = current_scan[
    current_scan["WeeklyConfirm"] == True
].copy()

print("\n====================================")
print("📈 CURRENT SCANNER SUMMARY")
print("====================================")

print(f"Total symbols       : {len(current_scan)}")
print(f"Production universe : {len(PRODUCTION_SYMBOLS)}")
print(f"Production min score: {PRODUCTION_MIN_SCORE}")
print(f"Production ATR mult : {PRODUCTION_ATR_MULT:.2f}x")
print(f"Production target2  : {PRODUCTION_TARGET2_PCT * 100:.1f}%")
print(f"Canonical signals   : {len(signal_scan)}")
print(f"Weekly confirmed    : {len(weekly_confirmed)}")

if len(current_scan) > 0:
    signal_rate = (
        len(signal_scan) / len(current_scan) * 100
    )
else:
    signal_rate = 0.0

print(f"Signal rate         : {signal_rate:.2f}%")


# ============================================================
# TOP SIGNALS
# ============================================================

print("\n====================================")
print("🏆 TOP CURRENT SIGNALS")
print("====================================")

if signal_scan.empty:
    print("No current Canonical signals.")
else:
    display(
        signal_scan[
            [
                "Symbol",
                "Date",
                "Close",
                "Score",
                "WeeklyConfirm",
                "ATR14",
                "Stop",
                "Shares",
                "RiskAmount",
                "Target1",
                "Target2",
            ]
        ].head(10)
    )


# ============================================================
# SANITY CHECKS
# ============================================================

assert len(current_scan) == len(canonical_data)

assert current_scan["Score"].dropna().between(0, 5).all()

assert (
    current_scan["RiskAmount"].fillna(0)
    <= RISK_PER_TRADE + 1e-9
).all()

assert (
    current_scan["PositionValue"].fillna(0)
    <= INITIAL_CAPITAL + 1e-9
).all()

print("\n====================================")
print("✅ Current Scanner Cell Ready")
print("====================================")

In [ ]:

# ============================================================
# 11) HISTORICAL SIGNAL GENERATION
# ============================================================

print("📡 HISTORICAL SIGNAL GENERATION")
print("=" * 90)


SIGNAL_MIN_SCORE = 3


def generate_historical_signals(
    symbol,
    df,
    min_score=SIGNAL_MIN_SCORE,
):
    if df is None or df.empty:
        return pd.DataFrame()

    data = df.copy()

    # --------------------------------------------------------
    # Required columns
    # --------------------------------------------------------

    required = [
        "Close",
        "Open",
        "High",
        "Low",
        "ATR14",
        "RECENT_LOW_5",
        "CANONICAL_SCORE",
        "CANONICAL_SIGNAL",
        "CANONICAL_WEEKLY_TREND",
        "CANONICAL_WEEKLY_MACD",
        "CANONICAL_WEEKLY_CONFIRM",
    ]

    missing = [
        col for col in required
        if col not in data.columns
    ]

    if missing:
        raise ValueError(
            f"{symbol}: missing columns {missing}"
        )

    # --------------------------------------------------------
    # Signal rows
    # --------------------------------------------------------

    signal_mask = (
        data["CANONICAL_SIGNAL"].fillna(False)
        & (data["CANONICAL_SCORE"] >= min_score)
    )

    signal_dates = data.index[signal_mask]

    rows = []

    for signal_date in signal_dates:
        signal_pos = data.index.get_loc(signal_date)

        # Need next trading day for entry
        if signal_pos + 1 >= len(data):
            continue

        entry_date = data.index[signal_pos + 1]

        signal_row = data.iloc[signal_pos]
        entry_row = data.iloc[signal_pos + 1]

        signal_close = signal_row["Close"]
        signal_atr = signal_row["ATR14"]
        recent_low = signal_row["RECENT_LOW_5"]

        # ----------------------------------------------------
        # Stop is calculated from information available
        # at SIGNAL DATE.
        # ----------------------------------------------------

        stop_price = calculate_stop_price(
            signal_close,
            signal_atr,
            recent_low,
        )

        # ----------------------------------------------------
        # Entry is NEXT DAY OPEN.
        # ----------------------------------------------------

        entry_price = entry_row["Open"]

        # ----------------------------------------------------
        # If entry is invalid, skip.
        # ----------------------------------------------------

        if pd.isna(entry_price) or entry_price <= 0:
            continue

        if pd.isna(stop_price) or stop_price <= 0:
            continue

        # ----------------------------------------------------
        # Position sizing uses actual next-day entry.
        # ----------------------------------------------------

        trade_plan = build_trade_plan(
            INITIAL_CAPITAL,
            entry_price,
            signal_atr,
            recent_low,
        )

        # ----------------------------------------------------
        # Targets based on ACTUAL entry.
        # ----------------------------------------------------

        target1 = entry_price * 1.07
        target2 = entry_price * 1.15

        rows.append({
            "Symbol": symbol,

            "SignalDate": signal_date,
            "EntryDate": entry_date,

            "SignalClose": float(signal_close),
            "EntryPrice": float(entry_price),

            "Score": int(signal_row["CANONICAL_SCORE"]),

            "WeeklyTrend": bool(
                signal_row["CANONICAL_WEEKLY_TREND"]
            ),

            "WeeklyMACD": bool(
                signal_row["CANONICAL_WEEKLY_MACD"]
            ),

            "WeeklyConfirm": bool(
                signal_row["CANONICAL_WEEKLY_CONFIRM"]
            ),

            "ATR14": float(signal_atr),

            "RecentLow5": float(recent_low),

            "StopPrice": float(
                trade_plan["stop_price"]
            ),

            "RiskPerShare": float(
                trade_plan["risk_per_share"]
            ),

            "Shares": int(
                trade_plan["shares"]
            ),

            "Lots": int(
                trade_plan["lots"]
            ),

            "RiskAmount": float(
                trade_plan["risk_amount"]
            ),

            "PositionValue": float(
                trade_plan["position_value"]
            ),

            "Target1": float(target1),
            "Target2": float(target2),

            "EntryOpen": float(entry_row["Open"]),
            "EntryHigh": float(entry_row["High"]),
            "EntryLow": float(entry_row["Low"]),
            "EntryClose": float(entry_row["Close"]),
        })

    return pd.DataFrame(rows)


# ============================================================
# GENERATE SIGNALS FOR ALL SYMBOLS
# ============================================================

historical_signals_list = []

for symbol, df in canonical_data.items():

    signals = generate_historical_signals(
        symbol,
        df,
        min_score=SIGNAL_MIN_SCORE,
    )

    if not signals.empty:
        historical_signals_list.append(signals)


if historical_signals_list:
    historical_signals = pd.concat(
        historical_signals_list,
        ignore_index=True,
    )
else:
    historical_signals = pd.DataFrame()


# ============================================================
# SORT
# ============================================================

if not historical_signals.empty:
    historical_signals = historical_signals.sort_values(
        ["SignalDate", "Score"],
        ascending=[True, False],
    ).reset_index(drop=True)


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print("📊 HISTORICAL SIGNAL SUMMARY")
print("====================================")

print(
    f"Total signals : {len(historical_signals):,}"
)

if not historical_signals.empty:

    print(
        f"Symbols       : "
        f"{historical_signals['Symbol'].nunique()}"
    )

    print(
        f"Score >= 3    : "
        f"{(historical_signals['Score'] >= 3).sum():,}"
    )

    print(
        f"Weekly confirm: "
        f"{historical_signals['WeeklyConfirm'].sum():,}"
    )

    print(
        f"Date range    : "
        f"{historical_signals['SignalDate'].min().date()} "
        f"→ "
        f"{historical_signals['SignalDate'].max().date()}"
    )


# ============================================================
# LOOKAHEAD SANITY CHECK
# ============================================================

print("\n====================================")
print("🔬 LOOKAHEAD SANITY CHECK")
print("====================================")

if not historical_signals.empty:

    invalid_entry = (
        historical_signals["EntryDate"]
        <= historical_signals["SignalDate"]
    ).sum()

    invalid_price = (
        historical_signals["EntryPrice"] <= 0
    ).sum()

    invalid_stop = (
        historical_signals["StopPrice"]
        >= historical_signals["SignalClose"]
    ).sum()

    print(
        f"EntryDate <= SignalDate : {invalid_entry}"
    )

    print(
        f"Invalid EntryPrice      : {invalid_price}"
    )

    print(
        f"Invalid Stop             : {invalid_stop}"
    )

    assert invalid_entry == 0
    assert invalid_price == 0
    assert invalid_stop == 0


# ============================================================
# SAMPLE SIGNALS
# ============================================================

print("\n====================================")
print("🔎 SAMPLE HISTORICAL SIGNALS")
print("====================================")

if historical_signals.empty:
    print("No historical signals found.")
else:
    display(
        historical_signals[
            [
                "Symbol",
                "SignalDate",
                "EntryDate",
                "SignalClose",
                "EntryPrice",
                "Score",
                "WeeklyConfirm",
                "ATR14",
                "StopPrice",
                "Shares",
                "RiskAmount",
                "Target1",
                "Target2",
            ]
        ].tail(15)
    )


# ============================================================
# FINAL CHECK
# ============================================================

print("\n====================================")
print("✅ Historical Signal Generation Ready")
print("====================================")

In [ ]:

# ============================================================
# 11.2) FIX HISTORICAL SIGNALS
# PRESERVE ORIGINAL SIGNAL SET
# FIX STOP FROM SIGNAL-DATE DATA
# ============================================================

print("🔧 FIXING HISTORICAL SIGNALS")
print("=" * 90)

old_signals = historical_signals.copy()

print(f"Original signals : {len(old_signals):,}")

fixed_rows = []
excluded_invalid_stop = 0

for _, old in old_signals.iterrows():

    symbol = old["Symbol"]
    signal_date = pd.Timestamp(old["SignalDate"])
    entry_date = pd.Timestamp(old["EntryDate"])

    if symbol not in feature_data:
        continue

    df = feature_data[symbol].sort_index()

    if signal_date not in df.index:
        continue

    if entry_date not in df.index:
        continue

    # RECENT_LOW_5 is created on canonical_data in Cell 9.
    # feature_data may not carry that derived column, so rebuild it
    # here from signal-date history if it is missing.
    if "RECENT_LOW_5" not in df.columns:
        df = df.copy()
        df["RECENT_LOW_5"] = (
            df["Low"]
            .rolling(
                window=RECENT_LOW_LOOKBACK,
                min_periods=RECENT_LOW_LOOKBACK,
            )
            .min()
        )

    signal_row = df.loc[signal_date]
    entry_row = df.loc[entry_date]

    signal_close = float(signal_row["Close"])
    signal_atr = float(signal_row["ATR14"])
    recent_low_value = signal_row["RECENT_LOW_5"]

    if not np.isfinite(recent_low_value) or recent_low_value <= 0:
        continue

    recent_low = float(recent_low_value)
    entry_price = float(entry_row["Open"])

    # --------------------------------------------------------
    # FIXED STOP FROM SIGNAL DATE
    # --------------------------------------------------------

    stop_price = calculate_stop_price(
        entry_price=signal_close,
        atr=signal_atr,
        recent_low=recent_low,
    )

    if not np.isfinite(stop_price) or stop_price <= 0:
        excluded_invalid_stop += 1
        continue

    # A long trade cannot have its protective stop at or above
    # the actual next-day entry price. This can happen after a
    # gap-down between signal date and entry date. Exclude that
    # trade from the executable historical set rather than
    # manufacturing a stop or silently changing the strategy.
    if entry_price <= stop_price:
        excluded_invalid_stop += 1
        continue

    # --------------------------------------------------------
    # POSITION SIZE
    # Uses actual entry price + fixed stop
    # --------------------------------------------------------

    risk_per_share = entry_price - stop_price

    if risk_per_share <= 0:
        shares = 0
        lots = 0
        risk_amount = 0.0
        position_value = 0.0
        status = "INVALID_STOP"

    else:
        max_risk = INITIAL_CAPITAL * MAX_RISK_PCT

        risk_shares = int(max_risk / risk_per_share)
        capital_shares = int(INITIAL_CAPITAL / entry_price)

        shares = min(risk_shares, capital_shares)

        shares = (shares // LOT_SIZE) * LOT_SIZE
        lots = shares // LOT_SIZE

        risk_amount = risk_per_share * shares
        position_value = entry_price * shares

        status = "OK" if shares > 0 else "CAPITAL_LIMITED"

    # --------------------------------------------------------
    # TARGETS
    # --------------------------------------------------------

    target1 = entry_price * 1.07
    target2 = entry_price * 1.15

    fixed_rows.append({
        "Symbol": symbol,
        "SignalDate": signal_date,
        "EntryDate": entry_date,
        "SignalClose": signal_close,
        "EntryPrice": entry_price,
        "Score": int(old["Score"]),
        "WeeklyConfirm": bool(old["WeeklyConfirm"]),
        "ATR14": signal_atr,
        "RecentLow5": recent_low,
        "StopPrice": stop_price,
        "Shares": shares,
        "Lots": lots,
        "RiskAmount": risk_amount,
        "PositionValue": position_value,
        "Target1": target1,
        "Target2": target2,
        "Status": status,
        "EntryOpen": float(entry_row["Open"]),
        "EntryHigh": float(entry_row["High"]),
        "EntryLow": float(entry_row["Low"]),
        "EntryClose": float(entry_row["Close"]),
    })


# ------------------------------------------------------------
# SAFETY CHECK
# ------------------------------------------------------------

if not fixed_rows:
    raise RuntimeError(
        "No signals were rebuilt. Stop here and inspect Cell 11."
    )


historical_signals = pd.DataFrame(fixed_rows)

historical_signals = (
    historical_signals
    .sort_values(["SignalDate", "Symbol"])
    .reset_index(drop=True)
)


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print("📊 REBUILD SUMMARY")
print("====================================")

print(f"Original signals : {len(old_signals):,}")
print(f"Fixed signals    : {len(historical_signals):,}")
print(
    f"Difference       : "
    f"{len(historical_signals) - len(old_signals):+,}"
)


# ============================================================
# VALIDATION
# ============================================================

entry_before_signal = (
    historical_signals["EntryDate"]
    < historical_signals["SignalDate"]
).sum()

invalid_entry = (
    ~np.isfinite(historical_signals["EntryPrice"])
    | (historical_signals["EntryPrice"] <= 0)
).sum()

invalid_stop = (
    ~np.isfinite(historical_signals["StopPrice"])
    | (historical_signals["StopPrice"] <= 0)
).sum()

stop_ge_entry = (
    historical_signals["StopPrice"]
    >= historical_signals["EntryPrice"]
).sum()

risk_over_limit = (
    historical_signals["RiskAmount"]
    > INITIAL_CAPITAL * MAX_RISK_PCT + 1e-9
).sum()

capital_over_limit = (
    historical_signals["PositionValue"]
    > INITIAL_CAPITAL + 1e-9
).sum()


print("\n====================================")
print("🔍 VALIDATION")
print("====================================")

print(f"Entry before Signal : {entry_before_signal:,}")
print(f"Invalid EntryPrice  : {invalid_entry:,}")
print(f"Invalid Stop        : {invalid_stop:,}")
print(f"Stop >= Entry       : {stop_ge_entry:,}")
print(f"Risk > 2%           : {risk_over_limit:,}")
print(f"Capital > 100k      : {capital_over_limit:,}")


# ============================================================
# STOP CONSISTENCY
# ============================================================

check = historical_signals.head(10).copy()

check["ExpectedStop"] = check.apply(
    lambda r: calculate_stop_price(
        r["SignalClose"],
        r["ATR14"],
        r["RecentLow5"],
    ),
    axis=1,
)

check["StopMatch"] = np.isclose(
    check["StopPrice"],
    check["ExpectedStop"],
    rtol=1e-10,
    atol=1e-10,
)

print("\n====================================")
print("🔒 STOP CONSISTENCY")
print("====================================")

display(
    check[
        [
            "Symbol",
            "SignalDate",
            "SignalClose",
            "EntryPrice",
            "ATR14",
            "RecentLow5",
            "StopPrice",
            "ExpectedStop",
            "StopMatch",
        ]
    ]
)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n====================================")

assert len(historical_signals) + excluded_invalid_stop == len(old_signals), (
    "Signal accounting mismatch during rebuild."
)
assert entry_before_signal == 0, "EntryDate is before SignalDate."
assert invalid_entry == 0, "Invalid EntryPrice found."
assert invalid_stop == 0, "Invalid StopPrice found."
assert stop_ge_entry == 0, "StopPrice >= EntryPrice found."
assert risk_over_limit == 0, "Risk exceeds configured limit."
assert capital_over_limit == 0, "Position value exceeds initial capital."
assert check["StopMatch"].all(), "Stop-price consistency check failed."

print("✅ HISTORICAL SIGNALS FIXED")
print("CELL 11.2 PASSED")
print("====================================")

In [ ]:

# ============================================================
# 12.1) SIGNAL CLUSTER ANALYSIS
# NO TRADE-ENGINE DEPENDENCY
# ============================================================

print("🔍 SIGNAL CLUSTER ANALYSIS")
print("=" * 90)

assert not historical_signals.empty, "historical_signals is empty."

analysis = historical_signals.copy()
analysis["SignalDate"] = pd.to_datetime(analysis["SignalDate"])
analysis["EntryDate"] = pd.to_datetime(analysis["EntryDate"])

analysis = analysis.sort_values(
    ["Symbol", "SignalDate"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Days since previous signal for the same symbol
# ------------------------------------------------------------

analysis["PrevSignalDate"] = (
    analysis.groupby("Symbol")["SignalDate"].shift(1)
)

analysis["DaysSincePrevSignal"] = (
    analysis["SignalDate"] - analysis["PrevSignalDate"]
).dt.days

analysis["ClusteredSignal"] = (
    analysis["DaysSincePrevSignal"].notna()
    & (analysis["DaysSincePrevSignal"] <= 3)
)

clustered = analysis[analysis["ClusteredSignal"]].copy()

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

total_signals = len(analysis)
clustered_count = len(clustered)
cluster_rate = clustered_count / total_signals * 100

print("\n====================================")
print("📊 SIGNAL CLUSTER SUMMARY")
print("====================================")
print(f"Total signals     : {total_signals:,}")
print(f"Clustered signals : {clustered_count:,}")
print(f"Cluster rate      : {cluster_rate:.2f}%")

# ------------------------------------------------------------
# Symbol-level summary
# ------------------------------------------------------------

symbol_cluster = (
    analysis.groupby("Symbol")
    .agg(
        Signals=("SignalDate", "count"),
        Clustered=("ClusteredSignal", "sum"),
    )
)

symbol_cluster["ClusterRate"] = (
    symbol_cluster["Clustered"]
    / symbol_cluster["Signals"]
    * 100
)

symbol_cluster = symbol_cluster.sort_values(
    ["Signals", "ClusterRate"],
    ascending=[False, False],
)

print("\n====================================")
print("📈 SIGNALS BY SYMBOL")
print("====================================")
display(symbol_cluster)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

same_symbol_order_ok = (
    analysis.groupby("Symbol")["SignalDate"]
    .apply(lambda s: s.is_monotonic_increasing)
    .all()
)

invalid_gap = (
    analysis["DaysSincePrevSignal"].notna()
    & (analysis["DaysSincePrevSignal"] < 0)
).sum()

assert same_symbol_order_ok, "Signal dates are not chronological by symbol."
assert invalid_gap == 0, "Negative signal-date gap found."
assert clustered_count <= total_signals

print("\n====================================")
print("🔎 RECENT CLUSTERED SIGNALS")
print("====================================")

if clustered.empty:
    print("No clustered signals found.")
else:
    display(
        clustered[
            [
                "Symbol",
                "SignalDate",
                "EntryDate",
                "Score",
                "WeeklyConfirm",
                "DaysSincePrevSignal",
            ]
        ].tail(30)
    )

print("\n====================================")
print("✅ CELL 12.1 PASSED")
print("====================================")

In [ ]:
# ============================================================
# CELL 12.2 — NON-OVERLAPPING BACKTEST
# ============================================================

print("=" * 90)
print("🔬 NON-OVERLAPPING HISTORICAL BACKTEST")
print("=" * 90)

# ------------------------------------------------------------
# 1) Conservative trade simulator
# ------------------------------------------------------------

def simulate_trade_fixed_stop(signal, price_data):
    """
    Simulate one long trade with:
      - next-day entry
      - conservative gap/stop handling
      - target 1 tracking
      - target 2 exit
      - hard MAX_HOLDING_DAYS time exit
      - MAE/MFE measured only while the trade is actually open

    MAX_HOLDING_DAYS is interpreted as calendar days from EntryDate.
    On the maximum-holding-date bar, STOP/TARGET rules have priority;
    otherwise the position is closed at that day's Close.
    """

    entry_date = pd.Timestamp(signal["EntryDate"])
    entry_price = float(signal["EntryPrice"])
    stop_price = float(signal["StopPrice"])
    shares = int(signal["Shares"])

    if entry_price <= 0 or stop_price <= 0 or shares <= 0:
        return None

    target1 = entry_price * 1.07
    target2 = entry_price * 1.15

    future = price_data.loc[
        price_data.index >= entry_date
    ].copy()

    if future.empty:
        return None

    max_hold_date = entry_date + pd.Timedelta(
        days=MAX_HOLDING_DAYS
    )

    future = future.loc[
        future.index <= max_hold_date
    ].copy()

    if future.empty:
        return None

    target1_hit = False
    exit_date = pd.NaT
    exit_price = np.nan
    exit_reason = None

    for current_date, bar in future.iterrows():

        day_open = float(bar["Open"])
        day_high = float(bar["High"])
        day_low = float(bar["Low"])
        day_close = float(bar["Close"])

        # ----------------------------------------------------
        # Gap handling
        # ----------------------------------------------------
        # If market opens below stop, actual exit is OPEN.
        # This is conservative and avoids assuming execution
        # at a better price than the market provided.
        if day_open <= stop_price:
            exit_date = current_date
            exit_price = day_open
            exit_reason = "STOP_GAP"
            break

        # ----------------------------------------------------
        # Normal intraday stop
        # ----------------------------------------------------
        if day_low <= stop_price:
            exit_date = current_date
            exit_price = stop_price
            exit_reason = "STOP"
            break

        # ----------------------------------------------------
        # Target 2
        # ----------------------------------------------------
        if day_open >= target2:
            target1_hit = True
            exit_date = current_date
            exit_price = day_open
            exit_reason = "TARGET2_GAP"
            break

        if day_high >= target2:
            target1_hit = True
            exit_date = current_date
            exit_price = target2
            exit_reason = "TARGET2"
            break

        # ----------------------------------------------------
        # Target 1 — tracking only
        # ----------------------------------------------------
        if day_high >= target1:
            target1_hit = True

        # ----------------------------------------------------
        # Hard time exit
        # ----------------------------------------------------
        # STOP/TARGET checks above always have priority on this
        # date. If neither occurs, exit at the closing price.
        if current_date >= max_hold_date:
            exit_date = current_date
            exit_price = day_close
            exit_reason = "MAX_HOLD"
            break

    # --------------------------------------------------------
    # End of data
    # --------------------------------------------------------

    if pd.isna(exit_date):
        # This can only happen when the available dataset ends
        # before MAX_HOLDING_DAYS. Close at the final available bar.
        exit_date = future.index[-1]
        exit_price = float(future.iloc[-1]["Close"])
        exit_reason = "EOD"

    holding_days = (
        pd.Timestamp(exit_date) - entry_date
    ).days

    if holding_days > MAX_HOLDING_DAYS:
        raise AssertionError(
            f"MAX_HOLDING_DAYS violated: "
            f"{holding_days} > {MAX_HOLDING_DAYS}"
        )

    pnl = (exit_price - entry_price) * shares
    return_pct = (
        (exit_price - entry_price)
        / entry_price
        * 100
    )

    # IMPORTANT:
    # MAE/MFE must be calculated only over the period in which
    # the position was actually open. Using all future bars would
    # introduce look-ahead leakage into the diagnostics.
    held_future = price_data.loc[
        (price_data.index >= entry_date)
        & (price_data.index <= exit_date)
    ].copy()

    min_low = float(held_future["Low"].min())
    max_high = float(held_future["High"].max())

    mae_pct = (
        (min_low - entry_price)
        / entry_price
        * 100
    )

    mfe_pct = (
        (max_high - entry_price)
        / entry_price
        * 100
    )

    return {
        "Symbol": signal["Symbol"],
        "SignalDate": signal["SignalDate"],
        "EntryDate": entry_date,
        "EntryPrice": entry_price,
        "StopPrice": stop_price,
        "Shares": shares,
        "Target1": target1,
        "Target2": target2,
        "Target1Hit": target1_hit,
        "ExitDate": exit_date,
        "ExitPrice": exit_price,
        "ExitReason": exit_reason,
        "HoldingDays": holding_days,
        "PnL": pnl,
        "ReturnPct": return_pct,
        "MAE_Pct": mae_pct,
        "MFE_Pct": mfe_pct,
        "Score": signal["Score"],
        "WeeklyConfirm": signal["WeeklyConfirm"],
    }


# ------------------------------------------------------------
# 2) Backtest one symbol without overlapping trades
# ------------------------------------------------------------

def backtest_one_symbol_nonoverlap(symbol):

    signals = historical_signals[
        historical_signals["Symbol"] == symbol
    ].copy()

    signals = signals.sort_values(
        ["EntryDate", "SignalDate"]
    ).reset_index(drop=True)

    if symbol not in feature_data:
        return pd.DataFrame()

    price_data = feature_data[symbol].copy()

    trades = []
    previous_exit_date = pd.NaT
    skipped_overlap = 0

    for _, signal in signals.iterrows():

        entry_date = pd.Timestamp(signal["EntryDate"])

        # ----------------------------------------------------
        # Prevent overlapping trades
        # ----------------------------------------------------
        if not pd.isna(previous_exit_date):
            if entry_date <= previous_exit_date:
                skipped_overlap += 1
                continue

        trade = simulate_trade_fixed_stop(
            signal,
            price_data
        )

        if trade is None:
            continue

        trades.append(trade)
        previous_exit_date = trade["ExitDate"]

    result = pd.DataFrame(trades)

    return result


# ------------------------------------------------------------
# 3) Test AU.BK first
# ------------------------------------------------------------

TEST_SYMBOL = "AU.BK"

au_trades = backtest_one_symbol_nonoverlap(TEST_SYMBOL)

print("\n====================================")
print(f"📊 {TEST_SYMBOL} BACKTEST")
print("====================================")

print(
    f"Signals available : "
    f"{len(historical_signals[historical_signals['Symbol'] == TEST_SYMBOL]):,}"
)

print(f"Trades executed   : {len(au_trades):,}")

if not au_trades.empty:

    print(
        f"Total P&L         : "
        f"{au_trades['PnL'].sum():,.2f} THB"
    )

    print(
        f"Average Return    : "
        f"{au_trades['ReturnPct'].mean():.2f}%"
    )

    print(
        f"Median Return     : "
        f"{au_trades['ReturnPct'].median():.2f}%"
    )

    print(
        f"Win Rate          : "
        f"{(au_trades['ReturnPct'] > 0).mean() * 100:.2f}%"
    )

    print("\n====================================")
    print("🔎 TRADE SAMPLE")
    print("====================================")

    display(
        au_trades[
            [
                "Symbol",
                "SignalDate",
                "EntryDate",
                "EntryPrice",
                "StopPrice",
                "ExitDate",
                "ExitPrice",
                "ExitReason",
                "HoldingDays",
                "Shares",
                "PnL",
                "ReturnPct",
                "Target1Hit",
            ]
        ].head(15)
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    invalid_exit = (
        au_trades["ExitDate"] < au_trades["EntryDate"]
    ).sum()

    invalid_return = (
        ~pd.to_numeric(
            au_trades["ReturnPct"],
            errors="coerce"
        ).notna()
    ).sum()

    invalid_holding = (
        au_trades["HoldingDays"] > MAX_HOLDING_DAYS
    ).sum()

    print("\n====================================")
    print("🛡️ BACKTEST VALIDATION")
    print("====================================")
    print(f"Exit before Entry : {invalid_exit}")
    print(f"Invalid Return    : {invalid_return}")
    print(f"Over Max Holding  : {invalid_holding}")

    assert invalid_exit == 0
    assert invalid_return == 0
    assert invalid_holding == 0

    print("\n====================================")
    print("✅ CELL 12.2 PASSED")
    print("====================================")
    print(
        "Non-overlapping backtest is valid for the test symbol."
    )

else:
    print(
        "\n⚠️ No executable trades found for",
        TEST_SYMBOL
    )



In [ ]:

# ============================================================
# CELL 12.3 — TRADE ENGINE DIAGNOSTICS
# ============================================================

print("=" * 90)
print("🔎 TRADE ENGINE DIAGNOSTICS — AU.BK")
print("=" * 90)

if au_trades.empty:
    print("⚠️ No trades available.")
else:

    df = au_trades.copy()

    # --------------------------------------------------------
    # 1) Exit distribution
    # --------------------------------------------------------

    exit_counts = (
        df["ExitReason"]
        .value_counts()
        .rename_axis("ExitReason")
        .reset_index(name="Trades")
    )

    exit_counts["Pct"] = (
        exit_counts["Trades"] / len(df) * 100
    )

    print("\n====================================")
    print("📊 EXIT DISTRIBUTION")
    print("====================================")

    display(exit_counts)

    # --------------------------------------------------------
    # 2) Return distribution
    # --------------------------------------------------------

    print("\n====================================")
    print("📈 RETURN DISTRIBUTION")
    print("====================================")

    print(f"Best trade        : {df['ReturnPct'].max():.2f}%")
    print(f"Worst trade       : {df['ReturnPct'].min():.2f}%")
    print(f"Average           : {df['ReturnPct'].mean():.2f}%")
    print(f"Median            : {df['ReturnPct'].median():.2f}%")
    print(f"Win rate          : {(df['ReturnPct'] > 0).mean() * 100:.2f}%")
    print(f"Loss rate         : {(df['ReturnPct'] < 0).mean() * 100:.2f}%")
    print(f"Flat              : {(df['ReturnPct'] == 0).mean() * 100:.2f}%")

    # --------------------------------------------------------
    # 3) Risk characteristics
    # --------------------------------------------------------

    df["RiskPct"] = (
        (df["EntryPrice"] - df["StopPrice"])
        / df["EntryPrice"]
        * 100
    )

    print("\n====================================")
    print("🛡️ STOP / RISK DISTRIBUTION")
    print("====================================")

    print(f"Average Stop Risk : {df['RiskPct'].mean():.2f}%")
    print(f"Median Stop Risk  : {df['RiskPct'].median():.2f}%")
    print(f"Minimum Stop Risk : {df['RiskPct'].min():.2f}%")
    print(f"Maximum Stop Risk : {df['RiskPct'].max():.2f}%")

    # --------------------------------------------------------
    # 4) Target1 reached but eventually lost
    # --------------------------------------------------------

    target1_then_loss = df[
        (df["Target1Hit"] == True) &
        (df["ReturnPct"] < 0)
    ].copy()

    print("\n====================================")
    print("⚠️ TARGET1 HIT → EVENTUAL LOSS")
    print("====================================")

    print(
        f"Count: {len(target1_then_loss):,} "
        f"({len(target1_then_loss) / len(df) * 100:.2f}%)"
    )

    if not target1_then_loss.empty:
        display(
            target1_then_loss[
                [
                    "SignalDate",
                    "EntryDate",
                    "EntryPrice",
                    "StopPrice",
                    "ExitDate",
                    "ExitPrice",
                    "ExitReason",
                    "ReturnPct",
                    "PnL",
                ]
            ].sort_values("ReturnPct").head(10)
        )

    # --------------------------------------------------------
    # 5) Large stop-distance trades
    # --------------------------------------------------------

    print("\n====================================")
    print("⚠️ LARGEST STOP DISTANCES")
    print("====================================")

    display(
        df[
            [
                "SignalDate",
                "EntryDate",
                "EntryPrice",
                "StopPrice",
                "RiskPct",
                "ExitReason",
                "ReturnPct",
                "PnL",
            ]
        ]
        .sort_values("RiskPct", ascending=False)
        .head(10)
    )

    # --------------------------------------------------------
    # 6) Trade duration
    # --------------------------------------------------------

    df["HoldingDays"] = (
        pd.to_datetime(df["ExitDate"])
        - pd.to_datetime(df["EntryDate"])
    ).dt.days

    print("\n====================================")
    print("⏱️ HOLDING PERIOD")
    print("====================================")

    print(f"Average days     : {df['HoldingDays'].mean():.1f}")
    print(f"Median days      : {df['HoldingDays'].median():.1f}")
    print(f"Maximum days     : {df['HoldingDays'].max():.0f}")
    print(f"MAX_HOLDING_DAYS : {MAX_HOLDING_DAYS}")

    over_limit = int(
        (df["HoldingDays"] > MAX_HOLDING_DAYS).sum()
    )

    print(f"Over max holding : {over_limit}")

    # --------------------------------------------------------
    # 7) Basic assertions
    # --------------------------------------------------------

    assert (df["EntryDate"] <= df["ExitDate"]).all()

    assert (df["HoldingDays"] >= 0).all()
    assert (df["HoldingDays"] <= MAX_HOLDING_DAYS).all()
    assert over_limit == 0
    assert df["RiskPct"].notna().all()
    assert df["ReturnPct"].notna().all()

    print("\n====================================")
    print("✅ CELL 12.3 PASSED")
    print("====================================")
    print("Trade-engine diagnostics completed.")


In [ ]:
# ============================================================
# CELL 12.4 — ALL-SYMBOL NON-OVERLAPPING BACKTEST
# Research validation across the complete watchlist
# ============================================================

print("=" * 90)
print("📊 ALL-SYMBOL NON-OVERLAPPING BACKTEST")
print("=" * 90)

all_trades = []
symbol_summary = []

for symbol in WATCHLIST:
    trades = backtest_one_symbol_nonoverlap(symbol)

    signal_count = int(
        (historical_signals["Symbol"] == symbol).sum()
    )

    trade_count = len(trades)

    if not trades.empty:
        all_trades.append(trades)

        symbol_summary.append({
            "Symbol": symbol,
            "Signals": signal_count,
            "Trades": trade_count,
            "PnL": float(trades["PnL"].sum()),
            "AvgReturnPct": float(trades["ReturnPct"].mean()),
            "WinRatePct": float((trades["ReturnPct"] > 0).mean() * 100),
            "MaxHoldingDays": int(trades["HoldingDays"].max()),
        })
    else:
        symbol_summary.append({
            "Symbol": symbol,
            "Signals": signal_count,
            "Trades": 0,
            "PnL": 0.0,
            "AvgReturnPct": np.nan,
            "WinRatePct": np.nan,
            "MaxHoldingDays": 0,
        })

if all_trades:
    all_trades_df = (
        pd.concat(all_trades, ignore_index=True)
        .sort_values(["EntryDate", "Symbol"])
        .reset_index(drop=True)
    )
else:
    all_trades_df = pd.DataFrame()

symbol_summary_df = pd.DataFrame(symbol_summary)

print("\n====================================")
print("📈 PORTFOLIO-LEVEL SUMMARY")
print("====================================")

print(f"Symbols tested     : {len(WATCHLIST)}")
print(f"Total trades       : {len(all_trades_df):,}")

if not all_trades_df.empty:
    print(f"Total P&L          : {all_trades_df['PnL'].sum():,.2f} THB")
    print(f"Average return     : {all_trades_df['ReturnPct'].mean():.2f}%")
    print(f"Median return      : {all_trades_df['ReturnPct'].median():.2f}%")
    print(f"Win rate           : {(all_trades_df['ReturnPct'] > 0).mean() * 100:.2f}%")
    print(f"Max holding days   : {all_trades_df['HoldingDays'].max():.0f}")

    over_limit = int(
        (all_trades_df["HoldingDays"] > MAX_HOLDING_DAYS).sum()
    )

    invalid_dates = int(
        (all_trades_df["ExitDate"] < all_trades_df["EntryDate"]).sum()
    )

    invalid_returns = int(
        (~pd.to_numeric(
            all_trades_df["ReturnPct"],
            errors="coerce"
        ).notna()).sum()
    )

    print(f"Over max holding   : {over_limit}")
    print(f"Invalid exit dates : {invalid_dates}")
    print(f"Invalid returns    : {invalid_returns}")

    print("\n====================================")
    print("🚪 EXIT REASONS")
    print("====================================")
    display(
        all_trades_df["ExitReason"]
        .value_counts()
        .rename_axis("ExitReason")
        .reset_index(name="Trades")
    )

    print("\n====================================")
    print("🏆 BY SYMBOL")
    print("====================================")
    display(
        symbol_summary_df
        .sort_values(["PnL", "WinRatePct"], ascending=[False, False])
        .reset_index(drop=True)
    )

    assert over_limit == 0
    assert invalid_dates == 0
    assert invalid_returns == 0
    assert all_trades_df["HoldingDays"].ge(0).all()
    assert all_trades_df["HoldingDays"].le(MAX_HOLDING_DAYS).all()

    print("\n====================================")
    print("✅ CELL 12.4 PASSED")
    print("====================================")
    print("All-symbol non-overlapping backtest passed validation.")
else:
    print("⚠️ No executable trades found across the watchlist.")


## CHECKPOINT — 12.2 PASSED

Cells 12.1 and 12.2 have been validated.

Run **Cell 12.3 only** next.

If Cell 12.3 prints `CELL 12.3 PASSED`, the next stage can be added.

In [ ]:
# ============================================================
# CELL 12.5 — WALK-FORWARD STRATEGY OPTIMIZATION
# ============================================================

print("=" * 90)
print("🧪 WALK-FORWARD STRATEGY OPTIMIZATION")
print("=" * 90)

# Fast research grid. Parameters are selected on TRAIN only,
# filtered on VALIDATION, then evaluated once on unseen TEST.
OPT_GRID = {
    "min_score": [3, 4, 5],
    "weekly_confirm": [False, True],
    "atr_mult": [1.2, 1.5],
    "target2_pct": [0.10, 0.15],
    "max_hold_days": [15, 20],
}
OPT_MIN_TRADES = 25

# ------------------------------------------------------------
# Chronological split
# ------------------------------------------------------------
dates = pd.to_datetime(
    historical_signals["SignalDate"]
).sort_values().unique()

assert len(dates) >= 100, "Not enough historical signal dates."

n = len(dates)
train_end = dates[int(n * 0.60) - 1]
valid_end = dates[int(n * 0.80) - 1]
test_start = dates[int(n * 0.80)]

print(f"Train end      : {pd.Timestamp(train_end).date()}")
print(f"Validation end : {pd.Timestamp(valid_end).date()}")
print(f"Test start     : {pd.Timestamp(test_start).date()}")

# ------------------------------------------------------------
# Cache signal sets. This avoids recalculating indicators and
# signal rows for every target/holding combination.
# ------------------------------------------------------------
signal_cache = {}

for min_score in OPT_GRID["min_score"]:
    for weekly_confirm in OPT_GRID["weekly_confirm"]:
        for atr_mult in OPT_GRID["atr_mult"]:

            key = (min_score, weekly_confirm, atr_mult)
            rows = []

            for symbol, df in canonical_data.items():

                mask = df["CANONICAL_SCORE"].ge(min_score)

                if weekly_confirm:
                    mask &= df[
                        "CANONICAL_WEEKLY_CONFIRM"
                    ].fillna(False)

                signal_positions = np.flatnonzero(
                    mask.to_numpy()
                )

                for pos in signal_positions:

                    if pos + 1 >= len(df):
                        continue

                    sig = df.iloc[pos]
                    ent = df.iloc[pos + 1]

                    entry = float(ent["Open"])

                    stop = calculate_stop_price(
                        float(sig["Close"]),
                        float(sig["ATR14"]),
                        float(sig["RECENT_LOW_5"]),
                        atr_mult,
                    )

                    if (
                        not np.isfinite(entry)
                        or entry <= 0
                        or not np.isfinite(stop)
                        or stop <= 0
                        or stop >= entry
                    ):
                        continue

                    position = calculate_position_size(
                        INITIAL_CAPITAL,
                        entry,
                        stop,
                    )

                    if position["shares"] <= 0:
                        continue

                    rows.append({
                        "Symbol": symbol,
                        "SignalDate": df.index[pos],
                        "EntryDate": df.index[pos + 1],
                        "EntryPrice": entry,
                        "StopPrice": float(stop),
                        "Shares": int(position["shares"]),
                    })

            signal_cache[key] = (
                pd.DataFrame(rows)
                .sort_values(
                    ["EntryDate", "SignalDate", "Symbol"]
                )
                .reset_index(drop=True)
                if rows
                else pd.DataFrame()
            )

print(
    f"Cached signal sets: {len(signal_cache)}"
)

# ------------------------------------------------------------
# Fast trade simulation
# ------------------------------------------------------------
def _opt_run(
    signals,
    target2_pct,
    max_hold_days,
    start_date,
    end_date,
):
    if signals.empty:
        return {
            "PnL": 0.0,
            "Trades": 0,
            "WinRatePct": np.nan,
        }

    selected = signals[
        (signals["SignalDate"] >= start_date)
        & (signals["SignalDate"] <= end_date)
    ].sort_values(
        ["EntryDate", "SignalDate", "Symbol"]
    )

    pnl = 0.0
    trades = 0
    wins = 0
    previous_exit = {}

    for _, sig in selected.iterrows():

        symbol = sig["Symbol"]
        entry_date = pd.Timestamp(sig["EntryDate"])

        if (
            symbol in previous_exit
            and entry_date <= previous_exit[symbol]
        ):
            continue

        df = canonical_data[symbol]

        end_date_hold = (
            entry_date
            + pd.Timedelta(days=max_hold_days)
        )

        future = df.loc[
            (df.index >= entry_date)
            & (df.index <= end_date_hold)
        ]

        if future.empty:
            continue

        entry = float(sig["EntryPrice"])
        stop = float(sig["StopPrice"])
        shares = int(sig["Shares"])
        target2 = entry * (1 + target2_pct)

        exit_date = pd.Timestamp(future.index[-1])
        exit_price = float(future.iloc[-1]["Close"])

        for current_date, bar in future.iterrows():

            op = float(bar["Open"])
            hi = float(bar["High"])
            lo = float(bar["Low"])
            cl = float(bar["Close"])

            if op <= stop:
                exit_date = pd.Timestamp(current_date)
                exit_price = op
                break

            if lo <= stop:
                exit_date = pd.Timestamp(current_date)
                exit_price = stop
                break

            if op >= target2:
                exit_date = pd.Timestamp(current_date)
                exit_price = op
                break

            if hi >= target2:
                exit_date = pd.Timestamp(current_date)
                exit_price = target2
                break

            if current_date >= end_date_hold:
                exit_date = pd.Timestamp(current_date)
                exit_price = cl
                break

        trade_pnl = (
            exit_price - entry
        ) * shares

        pnl += trade_pnl
        trades += 1
        wins += int(trade_pnl > 0)

        previous_exit[symbol] = exit_date

    return {
        "PnL": float(pnl),
        "Trades": int(trades),
        "WinRatePct": (
            float(wins / trades * 100)
            if trades
            else np.nan
        ),
    }

# ------------------------------------------------------------
# Evaluate grid
# ------------------------------------------------------------
import itertools

variants = [
    dict(zip(OPT_GRID.keys(), values))
    for values in itertools.product(
        *OPT_GRID.values()
    )
]

train_rows = []
valid_rows = []

for v in variants:

    key = (
        v["min_score"],
        v["weekly_confirm"],
        v["atr_mult"],
    )

    train_result = _opt_run(
        signal_cache[key],
        v["target2_pct"],
        v["max_hold_days"],
        dates[0],
        train_end,
    )

    valid_result = _opt_run(
        signal_cache[key],
        v["target2_pct"],
        v["max_hold_days"],
        train_end + np.timedelta64(1, "D"),
        valid_end,
    )

    train_rows.append({
        **v,
        **{
            f"Train_{k}": value
            for k, value in train_result.items()
        },
    })

    valid_rows.append({
        **v,
        **{
            f"Valid_{k}": value
            for k, value in valid_result.items()
        },
    })

train_df = pd.DataFrame(train_rows)
valid_df = pd.DataFrame(valid_rows)

results = train_df.merge(
    valid_df,
    on=list(OPT_GRID.keys()),
)

eligible = results[
    (results["Train_Trades"] >= OPT_MIN_TRADES)
    & (results["Valid_Trades"] >= 10)
    & (results["Valid_PnL"] > 0)
].copy()

if eligible.empty:
    print(
        "⚠️ No candidate passed the positive-validation filter."
    )

    eligible = results[
        (results["Train_Trades"] >= OPT_MIN_TRADES)
        & (results["Valid_Trades"] >= 10)
    ].copy()

assert not eligible.empty, (
    "No viable strategy candidate found."
)

best = eligible.sort_values(
    ["Train_PnL", "Valid_PnL"],
    ascending=[False, False],
).iloc[0]

BEST_STRATEGY = {
    "min_score": int(best["min_score"]),
    "weekly_confirm": bool(best["weekly_confirm"]),
    "atr_mult": float(best["atr_mult"]),
    "target2_pct": float(best["target2_pct"]),
    "max_hold_days": int(best["max_hold_days"]),
}

print("\n====================================")
print("🏆 SELECTED STRATEGY")
print("====================================")
print(BEST_STRATEGY)
print(
    f"Train P&L      : "
    f"{best['Train_PnL']:,.2f} THB"
)
print(
    f"Validation P&L : "
    f"{best['Valid_PnL']:,.2f} THB"
)

# ------------------------------------------------------------
# FINAL UNSEEN TEST — run exactly once
# ------------------------------------------------------------
best_key = (
    BEST_STRATEGY["min_score"],
    BEST_STRATEGY["weekly_confirm"],
    BEST_STRATEGY["atr_mult"],
)

optimized_test = _opt_run(
    signal_cache[best_key],
    BEST_STRATEGY["target2_pct"],
    BEST_STRATEGY["max_hold_days"],
    test_start,
    dates[-1],
)

baseline = _opt_run(
    signal_cache[(3, False, 1.5)],
    0.15,
    20,
    test_start,
    dates[-1],
)

print("\n====================================")
print("🧪 FINAL UNSEEN TEST")
print("====================================")
print(
    f"Baseline P&L   : "
    f"{baseline['PnL']:,.2f} THB"
)
print(
    f"Optimized P&L  : "
    f"{optimized_test['PnL']:,.2f} THB"
)
print(
    f"Delta P&L      : "
    f"{optimized_test['PnL'] - baseline['PnL']:,.2f} THB"
)
print(
    f"Optimized trades: "
    f"{optimized_test['Trades']}"
)
print(
    f"Optimized win rate: "
    f"{optimized_test['WinRatePct']:.2f}%"
)

if optimized_test["PnL"] > baseline["PnL"]:
    print(
        "\n✅ OPTIMIZED STRATEGY BEATS BASELINE "
        "ON UNSEEN TEST DATA"
    )
else:
    print(
        "\n⚠️ BASELINE REMAINS BETTER "
        "ON UNSEEN TEST DATA"
    )

print("\nTop validation candidates:")
display(
    eligible.sort_values(
        ["Valid_PnL", "Train_PnL"],
        ascending=[False, False],
    )[
        list(OPT_GRID.keys())
        + ["Train_PnL", "Valid_PnL",
           "Train_Trades", "Valid_Trades"]
    ].head(10)
)


In [ ]:
# ============================================================
# CELL 12.6 — SYMBOL FILTER OPTIMIZATION
# ============================================================

print("=" * 90)
print("🎯 SYMBOL FILTER OPTIMIZATION")
print("=" * 90)

# Use the already-selected strategy from Cell 12.5.
assert "BEST_STRATEGY" in globals()

base_key = (
    BEST_STRATEGY["min_score"],
    BEST_STRATEGY["weekly_confirm"],
    BEST_STRATEGY["atr_mult"],
)

base_signals = signal_cache[base_key]

def _symbol_pnl(
    signals,
    target2_pct,
    max_hold_days,
    start_date,
    end_date,
):
    rows = []

    if signals.empty:
        return pd.DataFrame()

    for symbol in sorted(signals["Symbol"].unique()):

        result = _opt_run(
            signals[signals["Symbol"] == symbol],
            target2_pct,
            max_hold_days,
            start_date,
            end_date,
        )

        rows.append({
            "Symbol": symbol,
            **result,
        })

    return pd.DataFrame(rows)

train_symbol = _symbol_pnl(
    base_signals,
    BEST_STRATEGY["target2_pct"],
    BEST_STRATEGY["max_hold_days"],
    dates[0],
    train_end,
)

valid_symbol = _symbol_pnl(
    base_signals,
    BEST_STRATEGY["target2_pct"],
    BEST_STRATEGY["max_hold_days"],
    train_end + np.timedelta64(1, "D"),
    valid_end,
)

symbol_eval = train_symbol.merge(
    valid_symbol,
    on="Symbol",
    suffixes=("_Train", "_Valid"),
)

# Rank symbols by TRAIN P&L only.
symbol_eval = symbol_eval.sort_values(
    "PnL_Train",
    ascending=False,
).reset_index(drop=True)

print("\n====================================")
print("📊 SYMBOL TRAIN / VALIDATION")
print("====================================")
display(symbol_eval)

# Test several portfolio widths.
K_GRID = [5, 8, 10, 12, 15, 19]

filter_results = []

for k in K_GRID:

    symbols = symbol_eval.head(k)["Symbol"].tolist()

    train_pnl = symbol_eval.head(k)["PnL_Train"].sum()
    valid_pnl = symbol_eval.head(k)["PnL_Valid"].sum()

    filter_results.append({
        "K": k,
        "Symbols": symbols,
        "TrainPnL": float(train_pnl),
        "ValidPnL": float(valid_pnl),
        "ValidTrades": int(
            symbol_eval.head(k)["Trades_Valid"].sum()
        ),
    })

filter_df = pd.DataFrame(filter_results)

print("\n====================================")
print("🏆 SYMBOL FILTER CANDIDATES")
print("====================================")
display(
    filter_df[
        ["K", "TrainPnL", "ValidPnL", "ValidTrades"]
    ].sort_values(
        "ValidPnL",
        ascending=False,
    )
)

eligible_filter = filter_df[
    (filter_df["TrainPnL"] > 0)
    & (filter_df["ValidTrades"] >= 20)
].copy()

if eligible_filter.empty:
    eligible_filter = filter_df[
        filter_df["ValidTrades"] >= 20
    ].copy()

assert not eligible_filter.empty

selected_filter = eligible_filter.sort_values(
    ["ValidPnL", "TrainPnL"],
    ascending=[False, False],
).iloc[0]

SELECTED_SYMBOLS = list(
    selected_filter["Symbols"]
)

print("\n====================================")
print("🎯 SELECTED SYMBOL UNIVERSE")
print("====================================")
print(
    f"K = {int(selected_filter['K'])}"
)
print(
    f"Symbols = {SELECTED_SYMBOLS}"
)
print(
    f"Validation P&L = "
    f"{selected_filter['ValidPnL']:,.2f} THB"
)

# ------------------------------------------------------------
# Final unseen test
# ------------------------------------------------------------
test_signals = base_signals[
    base_signals["Symbol"].isin(SELECTED_SYMBOLS)
]

filtered_test = _opt_run(
    test_signals,
    BEST_STRATEGY["target2_pct"],
    BEST_STRATEGY["max_hold_days"],
    test_start,
    dates[-1],
)

unfiltered_test = _opt_run(
    base_signals,
    BEST_STRATEGY["target2_pct"],
    BEST_STRATEGY["max_hold_days"],
    test_start,
    dates[-1],
)

print("\n====================================")
print("🧪 FINAL TEST — SYMBOL FILTER")
print("====================================")
print(
    f"Unfiltered P&L : "
    f"{unfiltered_test['PnL']:,.2f} THB"
)
print(
    f"Filtered P&L   : "
    f"{filtered_test['PnL']:,.2f} THB"
)
print(
    f"Delta P&L      : "
    f"{filtered_test['PnL'] - unfiltered_test['PnL']:,.2f} THB"
)
print(
    f"Filtered trades: "
    f"{filtered_test['Trades']}"
)
print(
    f"Filtered win rate: "
    f"{filtered_test['WinRatePct']:.2f}%"
)

if filtered_test["PnL"] > unfiltered_test["PnL"]:
    print(
        "\n✅ SYMBOL FILTER IMPROVES UNSEEN TEST P&L"
    )
else:
    print(
        "\n⚠️ SYMBOL FILTER DOES NOT IMPROVE UNSEEN TEST P&L"
    )


## PRODUCTION STRATEGY — OPTIMIZED

The current production configuration is frozen from the walk-forward research:

- Minimum score: **4**
- Weekly confirmation: **OFF**
- ATR stop multiplier: **1.2×**
- Target 2: **15%**
- Maximum holding: **20 calendar days**
- Production universe: **8 symbols**
  - TKN.BK
  - SAPPE.BK
  - SPA.BK
  - XO.BK
  - FORTH.BK
  - DITTO.BK
  - ICHI.BK
  - MASTER.BK

Unseen test result for this configuration + selected symbol universe:
**+25,242.06 THB**, versus **-170.21 THB** without the symbol filter and **-68,350.12 THB** for the original baseline on the same unseen period.

This configuration should remain frozen until a new out-of-sample research cycle is completed.

In [ ]:
# ============================================================
# CELL 12.7 — PRODUCTION ROBUSTNESS / RISK DIAGNOSTICS
# ============================================================

print("=" * 90)
print("🛡️ PRODUCTION ROBUSTNESS & RISK DIAGNOSTICS")
print("=" * 90)

assert "BEST_STRATEGY" in globals()
assert "SELECTED_SYMBOLS" in globals()

prod_signals = signal_cache[
    (
        BEST_STRATEGY["min_score"],
        BEST_STRATEGY["weekly_confirm"],
        BEST_STRATEGY["atr_mult"],
    )
]
prod_signals = prod_signals[
    prod_signals["Symbol"].isin(SELECTED_SYMBOLS)
].copy()

def _trade_records(signals, start_date, end_date):
    rows=[]
    previous_exit={}
    selected=signals[
        (signals["SignalDate"]>=start_date)&
        (signals["SignalDate"]<=end_date)
    ].sort_values(["EntryDate","SignalDate","Symbol"])

    for _,sig in selected.iterrows():
        symbol=sig["Symbol"]
        entry_date=pd.Timestamp(sig["EntryDate"])
        if symbol in previous_exit and entry_date<=previous_exit[symbol]:
            continue
        df=canonical_data[symbol]
        end_hold=entry_date+pd.Timedelta(days=BEST_STRATEGY["max_hold_days"])
        future=df.loc[(df.index>=entry_date)&(df.index<=end_hold)]
        if future.empty: continue
        entry=float(sig["EntryPrice"]); stop=float(sig["StopPrice"])
        shares=int(sig["Shares"]); target2=entry*(1+BEST_STRATEGY["target2_pct"])
        exit_date=pd.Timestamp(future.index[-1]); exit_price=float(future.iloc[-1]["Close"]); reason="EOD"
        for dt,bar in future.iterrows():
            op,hi,lo,cl=map(float,[bar["Open"],bar["High"],bar["Low"],bar["Close"]])
            if op<=stop: exit_date=pd.Timestamp(dt); exit_price=op; reason="STOP_GAP"; break
            if lo<=stop: exit_date=pd.Timestamp(dt); exit_price=stop; reason="STOP"; break
            if op>=target2: exit_date=pd.Timestamp(dt); exit_price=op; reason="TARGET2_GAP"; break
            if hi>=target2: exit_date=pd.Timestamp(dt); exit_price=target2; reason="TARGET2"; break
            if dt>=end_hold: exit_date=pd.Timestamp(dt); exit_price=cl; reason="MAX_HOLD"; break
        pnl=(exit_price-entry)*shares
        rows.append({
            "Symbol":symbol,"SignalDate":pd.Timestamp(sig["SignalDate"]),
            "EntryDate":entry_date,"ExitDate":exit_date,
            "EntryPrice":entry,"ExitPrice":exit_price,"StopPrice":stop,
            "Shares":shares,"PnL":pnl,
            "ReturnPct":(exit_price/entry-1)*100,
            "HoldDays":(exit_date-entry_date).days,
            "Reason":reason,
            "RiskPct":max(0,(entry-stop)/entry*100),
        })
        previous_exit[symbol]=exit_date
    return pd.DataFrame(rows)

prod_trades=_trade_records(prod_signals,dates[0],dates[-1])
assert not prod_trades.empty

# Sequential equity using initial capital. This is a trade-level risk diagnostic,
# not yet the final multi-symbol capital allocator.
equity=INITIAL_CAPITAL
curve=[]
peak=equity
max_dd=0.0
for _,t in prod_trades.sort_values("EntryDate").iterrows():
    equity += float(t.PnL)
    peak=max(peak,equity)
    dd=(equity-peak)/peak*100
    max_dd=min(max_dd,dd)
    curve.append((t.EntryDate,equity,dd))
equity_curve=pd.DataFrame(curve,columns=["Date","Equity","DrawdownPct"])

gross_profit=prod_trades.loc[prod_trades.PnL>0,"PnL"].sum()
gross_loss=-prod_trades.loc[prod_trades.PnL<0,"PnL"].sum()
profit_factor=(gross_profit/gross_loss) if gross_loss>0 else np.inf
expectancy=prod_trades.PnL.mean()
max_loss=prod_trades.PnL.min()
max_gain=prod_trades.PnL.max()
avg_win=prod_trades.loc[prod_trades.PnL>0,"PnL"].mean()
avg_loss=prod_trades.loc[prod_trades.PnL<0,"PnL"].mean()

print(f"Trades              : {len(prod_trades):,}")
print(f"Net P&L              : {prod_trades.PnL.sum():,.2f} THB")
print(f"Win rate             : {(prod_trades.PnL>0).mean()*100:.2f}%")
print(f"Profit factor        : {profit_factor:.3f}")
print(f"Expectancy / trade   : {expectancy:,.2f} THB")
print(f"Average winner       : {avg_win:,.2f} THB")
print(f"Average loser        : {avg_loss:,.2f} THB")
print(f"Best trade           : {max_gain:,.2f} THB")
print(f"Worst trade          : {max_loss:,.2f} THB")
print(f"Max drawdown         : {max_dd:.2f}%")
print(f"Ending equity        : {equity:,.2f} THB")
print(f"Average holding      : {prod_trades.HoldDays.mean():.2f} days")
print(f"Median holding       : {prod_trades.HoldDays.median():.2f} days")
print(f"Max holding          : {prod_trades.HoldDays.max():.0f} days")
print(f"Average planned stop : {prod_trades.RiskPct.mean():.2f}%")
print(f"Max planned stop     : {prod_trades.RiskPct.max():.2f}%")

print("\nExit reasons:")
print(prod_trades.Reason.value_counts().to_string())

# Yearly stability.
prod_trades["Year"]=prod_trades.EntryDate.dt.year
yearly=prod_trades.groupby("Year").agg(
    Trades=("PnL","size"),
    PnL=("PnL","sum"),
    WinRatePct=("PnL",lambda x:(x>0).mean()*100),
)
print("\nYearly stability:")
print(yearly.to_string())

# Stress test: degrade every exit by 0.5% and 1.0% to model slippage/fees.
for friction in [0.005,0.010]:
    stressed=(prod_trades.ExitPrice*(1-friction)-prod_trades.EntryPrice)*prod_trades.Shares
    print(f"Stress {friction*100:.1f}% exit friction P&L: {stressed.sum():,.2f} THB")

print("\n🛡️ ROBUSTNESS DIAGNOSTICS COMPLETE")


In [ ]:
# ============================================================
# CELL 12.8 — CAPITAL-AWARE PORTFOLIO BACKTEST
# ============================================================

print("=" * 90)
print("💰 CAPITAL-AWARE PORTFOLIO BACKTEST")
print("=" * 90)

# This is stricter than the independent-symbol test:
# all production symbols share the same 100,000 THB account.
# A new trade is accepted only when its required entry capital
# fits available cash. Positions may coexist across symbols.

assert "SELECTED_SYMBOLS" in globals()

portfolio_signals = prod_signals.copy()
portfolio_signals = portfolio_signals.sort_values(
    ["EntryDate","SignalDate","Symbol"]
).reset_index(drop=True)

capital = float(INITIAL_CAPITAL)
cash = capital
open_positions = []
closed = []

for _, sig in portfolio_signals.iterrows():

    entry_date = pd.Timestamp(sig["EntryDate"])

    # Realize positions that have already exited before this entry.
    still_open = []
    for pos in open_positions:
        if pos["ExitDate"] < entry_date:
            cash += pos["ExitValue"]
            closed.append(pos)
        else:
            still_open.append(pos)
    open_positions = still_open

    entry = float(sig["EntryPrice"])
    shares = int(sig["Shares"])
    required = entry * shares

    # Respect actual available cash.
    if required > cash:
        continue

    symbol = sig["Symbol"]
    df = canonical_data[symbol]
    stop = float(sig["StopPrice"])
    target2 = entry * (1 + BEST_STRATEGY["target2_pct"])
    end_hold = entry_date + pd.Timedelta(
        days=BEST_STRATEGY["max_hold_days"]
    )

    future = df.loc[
        (df.index >= entry_date)
        & (df.index <= end_hold)
    ]
    if future.empty:
        continue

    exit_date = pd.Timestamp(future.index[-1])
    exit_price = float(future.iloc[-1]["Close"])
    reason = "EOD"

    for dt, bar in future.iterrows():
        op, hi, lo, cl = map(
            float,
            [bar["Open"],bar["High"],bar["Low"],bar["Close"]]
        )

        if op <= stop:
            exit_date = pd.Timestamp(dt)
            exit_price = op
            reason = "STOP_GAP"
            break
        if lo <= stop:
            exit_date = pd.Timestamp(dt)
            exit_price = stop
            reason = "STOP"
            break
        if op >= target2:
            exit_date = pd.Timestamp(dt)
            exit_price = op
            reason = "TARGET2_GAP"
            break
        if hi >= target2:
            exit_date = pd.Timestamp(dt)
            exit_price = target2
            reason = "TARGET2"
            break
        if dt >= end_hold:
            exit_date = pd.Timestamp(dt)
            exit_price = cl
            reason = "MAX_HOLD"
            break

    exit_value = exit_price * shares
    cash -= required

    open_positions.append({
        "Symbol":symbol,
        "EntryDate":entry_date,
        "ExitDate":exit_date,
        "EntryPrice":entry,
        "ExitPrice":exit_price,
        "Shares":shares,
        "RequiredCapital":required,
        "ExitValue":exit_value,
        "PnL":exit_value-required,
        "Reason":reason,
    })

# Close any positions still open in the dataset.
for pos in open_positions:
    cash += pos["ExitValue"]
    closed.append(pos)

portfolio_trades = pd.DataFrame(closed)

assert not portfolio_trades.empty

net_pnl = float(portfolio_trades["PnL"].sum())
ending_cash = float(cash)

print(f"Accepted trades       : {len(portfolio_trades):,}")
print(f"Rejected for capital  : {len(portfolio_signals)-len(portfolio_trades):,}")
print(f"Net P&L               : {net_pnl:,.2f} THB")
print(f"Ending capital        : {ending_cash:,.2f} THB")
print(f"Return on initial cap : {net_pnl/INITIAL_CAPITAL*100:.2f}%")
print(
    f"Win rate              : "
    f"{(portfolio_trades.PnL>0).mean()*100:.2f}%"
)

print("\nExit reasons:")
print(
    portfolio_trades["Reason"]
    .value_counts()
    .to_string()
)

print("\nSymbol contribution:")
print(
    portfolio_trades.groupby("Symbol")["PnL"]
    .sum()
    .sort_values(ascending=False)
    .to_string()
)

print("\n💰 CAPITAL-AWARE BACKTEST COMPLETE")


In [ ]:
# ============================================================
# CELL 12.9 — OUT-OF-SAMPLE ROBUSTNESS / CONCENTRATION AUDIT
# ============================================================

print("=" * 90)
print("OUT-OF-SAMPLE ROBUSTNESS & CONCENTRATION AUDIT")
print("=" * 90)

assert "BEST_STRATEGY" in globals()
assert "SELECTED_SYMBOLS" in globals()
assert "prod_signals" in globals()
assert "test_start" in globals()

test_end = dates[-1]
base_test = _trade_records(prod_signals, test_start, test_end)
assert not base_test.empty
base_test_pnl = float(base_test["PnL"].sum())

loo_rows = []
for symbol in SELECTED_SYMBOLS:
    trades = _trade_records(
        prod_signals[prod_signals["Symbol"] != symbol].copy(),
        test_start,
        test_end,
    )
    pnl = float(trades["PnL"].sum()) if not trades.empty else 0.0
    loo_rows.append({
        "Removed": symbol,
        "TestPnL": pnl,
        "DeltaVsAll": pnl - base_test_pnl,
        "Trades": int(len(trades)),
        "WinRatePct": float((trades["PnL"] > 0).mean() * 100) if not trades.empty else np.nan,
    })

loo_df = pd.DataFrame(loo_rows).sort_values("DeltaVsAll", ascending=False)
print("\nLEAVE-ONE-SYMBOL-OUT — FINAL UNSEEN TEST")
print(f"All-symbol test P&L: {base_test_pnl:,.2f} THB")
display(loo_df)

symbol_test = base_test.groupby("Symbol")["PnL"].sum().sort_values(ascending=False)
positive_total = float(symbol_test.clip(lower=0).sum())
top2_total = float(symbol_test.head(2).clip(lower=0).sum())
top2_share = top2_total / positive_total * 100 if positive_total > 0 else np.nan

print("\nTEST PROFIT CONCENTRATION")
print(symbol_test.to_string())
print(f"Top-2 share of positive symbol P&L: {top2_share:.2f}%")

all_prod_dates = pd.to_datetime(prod_signals["SignalDate"]).sort_values().unique()
edges = np.linspace(0, len(all_prod_dates), 5, dtype=int)
rolling_rows = []
for i in range(4):
    start_i = all_prod_dates[edges[i]]
    end_i = all_prod_dates[edges[i + 1] - 1]
    trades = _trade_records(prod_signals, start_i, end_i)
    pnl = float(trades["PnL"].sum()) if not trades.empty else 0.0
    rolling_rows.append({
        "Block": i + 1,
        "Start": pd.Timestamp(start_i).date(),
        "End": pd.Timestamp(end_i).date(),
        "Trades": int(len(trades)),
        "PnL": pnl,
        "WinRatePct": float((trades["PnL"] > 0).mean() * 100) if not trades.empty else np.nan,
    })

rolling_df = pd.DataFrame(rolling_rows)
print("\nFIXED-PRODUCTION ROLLING BLOCKS")
display(rolling_df)
positive_blocks = int((rolling_df["PnL"] > 0).sum())

ditto = loo_df[loo_df["Removed"] == "DITTO.BK"]
ditto_improves = bool(not ditto.empty and ditto.iloc[0]["TestPnL"] > base_test_pnl)

if ditto_improves and positive_blocks >= 3 and top2_share < 80:
    decision = "RESEARCH_DITTO_REMOVAL"
else:
    decision = "KEEP_PRODUCTION_FROZEN"

print("\nROBUSTNESS DECISION")
print(f"DITTO removal improves test: {ditto_improves}")
print(f"Positive rolling blocks: {positive_blocks}/4")
print(f"Top-2 concentration: {top2_share:.2f}%")
print(f"Decision: {decision}")
print("\nOOS ROBUSTNESS AUDIT COMPLETE")



In [ ]:
import base64
exec(base64.b64decode("""cHJpbnQoIj0iICogOTApCnByaW50KCJORVNURUQgV0FMSy1GT1JXQVJEIFNZTUJPTCBTRUxFQ1RJT04iKQpwcmludCgiPSIgKiA5MCkKYXNzZXJ0ICJCRVNUX1NUUkFURUdZIiBpbiBnbG9iYWxzKCkKYXNzZXJ0ICJzaWduYWxfY2FjaGUiIGluIGdsb2JhbHMoKQpiYXNlX2tleT0oQkVTVF9TVFJBVEVHWVsibWluX3Njb3JlIl0sQkVTVF9TVFJBVEVHWVsid2Vla2x5X2NvbmZpcm0iXSxCRVNUX1NUUkFURUdZWyJhdHJfbXVsdCJdKQpuZXN0ZWRfc2lnbmFscz1zaWduYWxfY2FjaGVbYmFzZV9rZXldLmNvcHkoKQphbGxfZGF0ZXM9cGQudG9fZGF0ZXRpbWUobmVzdGVkX3NpZ25hbHNbIlNpZ25hbERhdGUiXSkuc29ydF92YWx1ZXMoKS51bmlxdWUoKQpuPWxlbihhbGxfZGF0ZXMpCmZvbGRfc3BlY3M9WygwLjAwLDAuNTAsMC42NSwwLjgwKSwoMC4wMCwwLjY1LDAuNzcsMC44OSksKDAuMDAsMC43NSwwLjg1LDEuMDApXQpLX0dSSURfTkVTVEVEPVs1LDcsOCwxMCwxMiwxNSwxOV0KbmVzdGVkX3Jvd3M9W10KZm9yIGZvbGRfbm8sKGEsYixjLGQpIGluIGVudW1lcmF0ZShmb2xkX3NwZWNzLHN0YXJ0PTEpOgogICAgdHJhaW5fc3RhcnQ9YWxsX2RhdGVzW2ludChuKmEpXTsgdHJhaW5fZW5kX2Y9YWxsX2RhdGVzW2ludChuKmIpLTFdCiAgICB2YWxpZF9zdGFydD1hbGxfZGF0ZXNbaW50KG4qYildOyB2YWxpZF9lbmRfZj1hbGxfZGF0ZXNbaW50KG4qYyktMV0KICAgIHRlc3Rfc3RhcnRfZj1hbGxfZGF0ZXNbaW50KG4qYyldOyB0ZXN0X2VuZF9mPWFsbF9kYXRlc1tpbnQobipkKS0xXQogICAgdHI9X3N5bWJvbF9wbmwobmVzdGVkX3NpZ25hbHMsQkVTVF9TVFJBVEVHWVsidGFyZ2V0Ml9wY3QiXSxCRVNUX1NUUkFURUdZWyJtYXhfaG9sZF9kYXlzIl0sdHJhaW5fc3RhcnQsdHJhaW5fZW5kX2YpCiAgICB2YT1fc3ltYm9sX3BubChuZXN0ZWRfc2lnbmFscyxCRVNUX1NUUkFURUdZWyJ0YXJnZXQyX3BjdCJdLEJFU1RfU1RSQVRFR1lbIm1heF9ob2xkX2RheXMiXSx2YWxpZF9zdGFydCx2YWxpZF9lbmRfZikKICAgIGV2PXRyLm1lcmdlKHZhLG9uPSJTeW1ib2wiLHN1ZmZpeGVzPSgiX1RyYWluIiwiX1ZhbGlkIikpLnNvcnRfdmFsdWVzKCJQbkxfVHJhaW4iLGFzY2VuZGluZz1GYWxzZSkucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQogICAgcm93cz1bXQogICAgZm9yIGsgaW4gS19HUklEX05FU1RFRDoKICAgICAgICB6PWV2LmhlYWQoaykKICAgICAgICByb3dzLmFwcGVuZCh7IksiOmssIlN5bWJvbHMiOnpbIlN5bWJvbCJdLnRvbGlzdCgpLCJUcmFpblBuTCI6ZmxvYXQoelsiUG5MX1RyYWluIl0uc3VtKCkpLCJWYWxpZFBuTCI6ZmxvYXQoelsiUG5MX1ZhbGlkIl0uc3VtKCkpLCJWYWxpZFRyYWRlcyI6aW50KHpbIlRyYWRlc19WYWxpZCJdLnN1bSgpKX0pCiAgICBjYW5kPXBkLkRhdGFGcmFtZShyb3dzKQogICAgb2s9Y2FuZFsoY2FuZFsiVHJhaW5QbkwiXT4wKSYoY2FuZFsiVmFsaWRUcmFkZXMiXT49MTUpXS5jb3B5KCkKICAgIGlmIG9rLmVtcHR5OiBvaz1jYW5kW2NhbmRbIlZhbGlkVHJhZGVzIl0+PTE1XS5jb3B5KCkKICAgIHNlbD1vay5zb3J0X3ZhbHVlcyhbIlZhbGlkUG5MIiwiVHJhaW5QbkwiXSxhc2NlbmRpbmc9W0ZhbHNlLEZhbHNlXSkuaWxvY1swXQogICAgc3ltYm9scz1saXN0KHNlbFsiU3ltYm9scyJdKQogICAgdGU9X29wdF9ydW4obmVzdGVkX3NpZ25hbHNbbmVzdGVkX3NpZ25hbHNbIlN5bWJvbCJdLmlzaW4oc3ltYm9scyldLEJFU1RfU1RSQVRFR1lbInRhcmdldDJfcGN0Il0sQkVTVF9TVFJBVEVHWVsibWF4X2hvbGRfZGF5cyJdLHRlc3Rfc3RhcnRfZix0ZXN0X2VuZF9mKQogICAgbmVzdGVkX3Jvd3MuYXBwZW5kKHsiRm9sZCI6Zm9sZF9ubywiVHJhaW5FbmQiOnBkLlRpbWVzdGFtcCh0cmFpbl9lbmRfZikuZGF0ZSgpLCJWYWxpZEVuZCI6cGQuVGltZXN0YW1wKHZhbGlkX2VuZF9mKS5kYXRlKCksIlRlc3RTdGFydCI6cGQuVGltZXN0YW1wKHRlc3Rfc3RhcnRfZikuZGF0ZSgpLCJUZXN0RW5kIjpwZC5UaW1lc3RhbXAodGVzdF9lbmRfZikuZGF0ZSgpLCJLIjppbnQoc2VsWyJLIl0pLCJUZXN0UG5MIjpmbG9hdCh0ZVsiUG5MIl0pLCJUZXN0VHJhZGVzIjppbnQodGVbIlRyYWRlcyJdKSwiVGVzdFdpblJhdGUiOmZsb2F0KHRlWyJXaW5SYXRlUGN0Il0pLCJESVRUT19TZWxlY3RlZCI6IkRJVFRPLkJLIiBpbiBzeW1ib2xzLCJTeW1ib2xzIjoiLCAiLmpvaW4oc3ltYm9scyl9KQpuZXN0ZWRfZGY9cGQuRGF0YUZyYW1lKG5lc3RlZF9yb3dzKQpwcmludCgiXG5ORVNURUQgV0FMSy1GT1JXQVJEIFJFU1VMVFMiKQpkaXNwbGF5KG5lc3RlZF9kZikKcHJpbnQoIlxuRElUVE8gU0VMRUNUSU9OIEZSRVFVRU5DWSIpCnByaW50KGYiRElUVE8gc2VsZWN0ZWQgaW4ge2ludChuZXN0ZWRfZGZbJ0RJVFRPX1NlbGVjdGVkJ10uc3VtKCkpfS97bGVuKG5lc3RlZF9kZil9IGZvbGRzIikKcHJpbnQoIlxuVEVTVCBQJkwgU1VNTUFSWSIpCnByaW50KGYiUG9zaXRpdmUgT09TIGZvbGRzOiB7aW50KChuZXN0ZWRfZGZbJ1Rlc3RQbkwnXT4wKS5zdW0oKSl9L3tsZW4obmVzdGVkX2RmKX0iKQpwcmludChmIlRvdGFsIE9PUyBQJkw6IHtuZXN0ZWRfZGZbJ1Rlc3RQbkwnXS5zdW0oKTosLjJmfSBUSEIiKQppZiBpbnQoKG5lc3RlZF9kZlsiVGVzdFBuTCJdPjApLnN1bSgpKT49MiBhbmQgaW50KG5lc3RlZF9kZlsiRElUVE9fU2VsZWN0ZWQiXS5zdW0oKSk8PTE6CiAgICBuZXN0ZWRfZGVjaXNpb249IkRJVFRPX1JFTU9WQUxfSEFTX1JPQlVTVF9TVVBQT1JUIgplbHNlOgogICAgbmVzdGVkX2RlY2lzaW9uPSJLRUVQX0NVUlJFTlRfVU5JVkVSU0UiCnByaW50KGYiXG5ORVNURUQgREVDSVNJT046IHtuZXN0ZWRfZGVjaXNpb259IikKcHJpbnQoIk5FU1RFRCBXQUxLLUZPUldBUkQgQ09NUExFVEUiKQo=""").decode("utf-8"))


In [ ]:
# ============================================================
# CELL 12.11 — RESEARCH ONLY: ENTRY/EXIT VARIANT AUDIT
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — ENTRY / EXIT VARIANT AUDIT")
print("=" * 90)

assert "SELECTED_SYMBOLS" in globals()
assert "signal_cache" in globals()
assert "BEST_STRATEGY" in globals()

key = (BEST_STRATEGY["min_score"], BEST_STRATEGY["weekly_confirm"], BEST_STRATEGY["atr_mult"])
rs = signal_cache[key]
rs = rs[rs["Symbol"].isin(SELECTED_SYMBOLS)].copy()

# Deliberately small grid: change one execution dimension at a time.
variants = [
    {"name":"BASE","target2_pct":0.15,"max_hold_days":20},
    {"name":"TARGET10","target2_pct":0.10,"max_hold_days":20},
    {"name":"HOLD15","target2_pct":0.15,"max_hold_days":15},
    {"name":"TARGET10_HOLD15","target2_pct":0.10,"max_hold_days":15},
]

dates_r = pd.to_datetime(rs["SignalDate"]).sort_values().unique()
nr = len(dates_r)
folds = [
    (0.00,0.50,0.65,0.80),
    (0.15,0.65,0.77,0.90),
    (0.25,0.75,0.85,1.00),
]

rows = []
for fold_no,(a,b,c,d) in enumerate(folds,1):
    train_start = dates_r[int(nr*a)]
    train_end = dates_r[int(nr*b)-1]
    valid_start = dates_r[int(nr*b)]
    valid_end = dates_r[int(nr*c)-1]
    test_start_r = dates_r[int(nr*c)]
    test_end_r = dates_r[int(nr*d)-1]

    vals=[]
    for v in variants:
        vr=_opt_run(rs,v["target2_pct"],v["max_hold_days"],valid_start,valid_end)
        vals.append((v,vr))
    eligible=[x for x in vals if x[1]["Trades"]>=10]
    if not eligible:
        eligible=vals
    chosen=sorted(eligible,key=lambda x:x[1]["PnL"],reverse=True)[0][0]

    te=_opt_run(rs,chosen["target2_pct"],chosen["max_hold_days"],test_start_r,test_end_r)
    base=_opt_run(rs,0.15,20,test_start_r,test_end_r)

    rows.append({
        "Fold":fold_no,
        "TrainEnd":pd.Timestamp(train_end).date(),
        "ValidEnd":pd.Timestamp(valid_end).date(),
        "TestStart":pd.Timestamp(test_start_r).date(),
        "TestEnd":pd.Timestamp(test_end_r).date(),
        "Selected":chosen["name"],
        "TestPnL":te["PnL"],
        "TestTrades":te["Trades"],
        "TestWinRate":te["WinRatePct"],
        "BaseTestPnL":base["PnL"],
        "DeltaVsBase":te["PnL"]-base["PnL"],
    })

exit_nested_df=pd.DataFrame(rows)
display(exit_nested_df)

print("\nEXIT NESTED SUMMARY")
print(f"Positive OOS folds: {int((exit_nested_df['TestPnL']>0).sum())}/{len(exit_nested_df)}")
print(f"Selected total OOS P&L: {exit_nested_df['TestPnL'].sum():,.2f} THB")
print(f"Frozen BASE total OOS P&L: {exit_nested_df['BaseTestPnL'].sum():,.2f} THB")
print(f"Total delta vs BASE: {exit_nested_df['DeltaVsBase'].sum():,.2f} THB")
print(f"Selected variants: {', '.join(exit_nested_df['Selected'].tolist())}")

if int((exit_nested_df["TestPnL"]>0).sum())>=2 and exit_nested_df["DeltaVsBase"].sum()>0:
    print("RESEARCH DECISION: EXIT_VARIANT_CANDIDATE")
else:
    print("RESEARCH DECISION: KEEP_CURRENT_EXIT")

print("PRODUCTION CONFIGURATION UNCHANGED")
print("ENTRY / EXIT VARIANT AUDIT COMPLETE")



In [ ]:
# ============================================================
# CELL 12.12 — RESEARCH ONLY: TRUE GLOBAL PORTFOLIO BACKTEST
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — TRUE GLOBAL PORTFOLIO BACKTEST")
print("=" * 90)

# This cell does NOT modify production. It fixes the 8-symbol universe,
# shares one account, reserves cash for open positions, sizes from current
# equity, processes exits before entries, and records gap losses.

assert "SELECTED_SYMBOLS" in globals()
assert "signal_cache" in globals()
assert "BEST_STRATEGY" in globals()

pkey=(BEST_STRATEGY["min_score"],BEST_STRATEGY["weekly_confirm"],BEST_STRATEGY["atr_mult"])
ps=signal_cache[pkey].copy()
ps=ps[ps["Symbol"].isin(SELECTED_SYMBOLS)].copy()

def _true_portfolio(signals,start_date,end_date,risk_pct,target2_pct,max_hold_days,max_positions):
    start_date=pd.Timestamp(start_date); end_date=pd.Timestamp(end_date)
    s=signals[(signals["SignalDate"]>=start_date)&(signals["SignalDate"]<=end_date)].copy()
    if s.empty:
        return {"PnL":0.0,"Trades":0,"WinRate":np.nan,"MaxDD":0.0,
                "EndEquity":INITIAL_CAPITAL,"PF":np.nan,"GapRate":np.nan,
                "ExposurePct":0.0}

    # Every market date in the selected symbols is processed, so exits can
    # occur even when there are no new signals on that date.
    market_dates=set()
    for sym in SELECTED_SYMBOLS:
        market_dates.update(pd.Timestamp(x) for x in canonical_data[sym].index
                            if start_date<=pd.Timestamp(x)<=end_date)
    market_dates=sorted(market_dates)

    entries=s.sort_values(["EntryDate","SignalDate","Symbol"],
                          ascending=[True,False,True])
    by_entry={d:g for d,g in entries.groupby("EntryDate",sort=False)}

    cash=float(INITIAL_CAPITAL)
    open_pos=[]
    trades=[]
    equity_points=[]
    skipped=0

    for dt in market_dates:
        dt=pd.Timestamp(dt)

        # 1) EXIT FIRST
        remaining=[]
        for p in open_pos:
            if dt < p["EntryDate"]:
                remaining.append(p); continue
            df=canonical_data[p["Symbol"]]
            if dt not in df.index:
                remaining.append(p); continue
            bar=df.loc[dt]
            op,hi,lo,cl=map(float,[bar["Open"],bar["High"],bar["Low"],bar["Close"]])
            px=None; reason=None
            if op<=p["Stop"]:
                px,reason=op,"STOP_GAP"
            elif lo<=p["Stop"]:
                px,reason=p["Stop"],"STOP"
            elif op>=p["Target"]:
                px,reason=op,"TARGET2_GAP"
            elif hi>=p["Target"]:
                px,reason=p["Target"],"TARGET2"
            elif dt>=p["MaxDate"]:
                px,reason=cl,"MAX_HOLD"
            if px is None:
                remaining.append(p); continue

            cash += px*p["Shares"]
            pnl=(px-p["Entry"])*p["Shares"]
            trades.append({
                "Symbol":p["Symbol"],"SignalDate":p["SignalDate"],
                "EntryDate":p["EntryDate"],"ExitDate":dt,
                "EntryPrice":p["Entry"],"ExitPrice":px,"StopPrice":p["Stop"],
                "Shares":p["Shares"],"PnL":pnl,"Reason":reason,
                "GapLoss":bool(reason=="STOP_GAP" and pnl<0),
                "EntryValue":p["Entry"]*p["Shares"],
            })
        open_pos=remaining

        # 2) ENTRIES
        todays=by_entry.get(dt)
        if todays is not None:
            for _,sig in todays.iterrows():
                if len(open_pos)>=max_positions:
                    skipped+=1; continue
                sym=sig["Symbol"]
                if any(p["Symbol"]==sym for p in open_pos):
                    skipped+=1; continue
                entry=float(sig["EntryPrice"]); stop=float(sig["StopPrice"])
                if not(np.isfinite(entry) and entry>0 and np.isfinite(stop) and 0<stop<entry):
                    skipped+=1; continue

                # Mark open positions at today's close to get current equity.
                marked=sum(
                    p["Shares"]*float(canonical_data[p["Symbol"]].loc[dt,"Close"])
                    for p in open_pos
                    if dt in canonical_data[p["Symbol"]].index
                )
                current_equity=cash+marked
                risk_budget=current_equity*risk_pct
                rps=entry-stop
                risk_lots=int((risk_budget/rps)//LOT_SIZE)
                cash_lots=int((cash/entry)//LOT_SIZE)
                lots=min(risk_lots,cash_lots)
                shares=lots*LOT_SIZE
                if shares<LOT_SIZE:
                    skipped+=1; continue

                cash-=shares*entry
                open_pos.append({
                    "Symbol":sym,"SignalDate":pd.Timestamp(sig["SignalDate"]),
                    "EntryDate":dt,"Entry":entry,"Stop":stop,
                    "Target":entry*(1+target2_pct),"Shares":int(shares),
                    "MaxDate":dt+pd.Timedelta(days=max_hold_days),
                })

        # 3) MARK-TO-MARKET EQUITY FOR DRAWNDOWN / EXPOSURE
        marked=sum(
            p["Shares"]*float(canonical_data[p["Symbol"]].loc[dt,"Close"])
            for p in open_pos if dt in canonical_data[p["Symbol"]].index
        )
        equity=cash+marked
        equity_points.append({
            "Date":dt,"Equity":equity,
            "Exposure":marked,
            "OpenPositions":len(open_pos),
        })

    # Positions still open at end_date are explicitly censored at last
    # available close; this avoids silently dropping capital.
    for p in list(open_pos):
        df=canonical_data[p["Symbol"]]
        fut=df[(df.index>=p["EntryDate"])&(df.index<=end_date)]
        if fut.empty: continue
        dt=pd.Timestamp(fut.index[-1]); px=float(fut.iloc[-1]["Close"])
        cash+=px*p["Shares"]
        pnl=(px-p["Entry"])*p["Shares"]
        trades.append({
            "Symbol":p["Symbol"],"SignalDate":p["SignalDate"],
            "EntryDate":p["EntryDate"],"ExitDate":dt,
            "EntryPrice":p["Entry"],"ExitPrice":px,"StopPrice":p["Stop"],
            "Shares":p["Shares"],"PnL":pnl,"Reason":"EOD",
            "GapLoss":False,"EntryValue":p["Entry"]*p["Shares"],
        })

    td=pd.DataFrame(trades)
    ed=pd.DataFrame(equity_points)

    if td.empty:
        return {"PnL":0.0,"Trades":0,"WinRate":np.nan,"MaxDD":0.0,
                "EndEquity":cash,"PF":np.nan,"GapRate":np.nan,
                "ExposurePct":float(ed["Exposure"].mean()/INITIAL_CAPITAL*100) if not ed.empty else 0.0}

    gross_profit=td.loc[td.PnL>0,"PnL"].sum()
    gross_loss=-td.loc[td.PnL<0,"PnL"].sum()
    peak=ed["Equity"].cummax()
    dd=(ed["Equity"]/peak-1)*100
    gap_rate=td["GapLoss"].mean()*100
    return {
        "PnL":float(td.PnL.sum()),"Trades":int(len(td)),
        "WinRate":float((td.PnL>0).mean()*100),
        "MaxDD":float(dd.min()),"EndEquity":float(ed.iloc[-1]["Equity"]),
        "PF":float(gross_profit/gross_loss) if gross_loss>0 else np.inf,
        "GapRate":float(gap_rate),
        "ExposurePct":float(ed["Exposure"].mean()/INITIAL_CAPITAL*100),
        "Skipped":int(skipped),
    }

PORTFOLIO_CONFIGS=[
    {"name":"BASE_2PCT_5","risk":0.020,"target":0.15,"hold":20,"maxpos":5},
    {"name":"RISK_1PCT_5","risk":0.010,"target":0.15,"hold":20,"maxpos":5},
    {"name":"RISK_1PCT_8","risk":0.010,"target":0.15,"hold":20,"maxpos":8},
    {"name":"RISK_1PCT_3","risk":0.010,"target":0.15,"hold":20,"maxpos":3},
    {"name":"RISK_1.5PCT_5","risk":0.015,"target":0.15,"hold":20,"maxpos":5},
]

portfolio_dates=pd.to_datetime(ps["SignalDate"]).sort_values().unique()
npd=len(portfolio_dates)
pf_folds=[
    (0.00,0.50,0.65,0.80),
    (0.15,0.65,0.77,0.90),
    (0.25,0.75,0.85,1.00),
]

pf_rows=[]
for fold_no,(a,b,c,d) in enumerate(pf_folds,1):
    tr0=portfolio_dates[int(npd*a)]; tr1=portfolio_dates[int(npd*b)-1]
    va0=portfolio_dates[int(npd*b)]; va1=portfolio_dates[int(npd*c)-1]
    te0=portfolio_dates[int(npd*c)]; te1=portfolio_dates[int(npd*d)-1]

    val=[]
    for cfg in PORTFOLIO_CONFIGS:
        vr=_true_portfolio(ps,va0,va1,cfg["risk"],cfg["target"],cfg["hold"],cfg["maxpos"])
        val.append((cfg,vr))
    eligible=[x for x in val if x[1]["Trades"]>=10]
    pool=eligible if eligible else val
    chosen=max(pool,key=lambda x:x[1]["PnL"])[0]
    te=_true_portfolio(ps,te0,te1,chosen["risk"],chosen["target"],chosen["hold"],chosen["maxpos"])
    base=_true_portfolio(ps,te0,te1,0.020,0.15,20,5)
    pf_rows.append({
        "Fold":fold_no,"TrainEnd":pd.Timestamp(tr1).date(),
        "ValidEnd":pd.Timestamp(va1).date(),"TestStart":pd.Timestamp(te0).date(),
        "TestEnd":pd.Timestamp(te1).date(),"Selected":chosen["name"],
        "TestPnL":te["PnL"],"Trades":te["Trades"],"WinRate":te["WinRate"],
        "MaxDD":te["MaxDD"],"PF":te["PF"],"GapRate":te["GapRate"],
        "ExposurePct":te["ExposurePct"],"BasePnL":base["PnL"],
        "DeltaVsBase":te["PnL"]-base["PnL"],
    })

true_pf_df=pd.DataFrame(pf_rows)
display(true_pf_df)
print("\nTRUE GLOBAL PORTFOLIO SUMMARY")
print(f"Positive OOS folds: {int((true_pf_df.TestPnL>0).sum())}/{len(true_pf_df)}")
print(f"Selected OOS P&L: {true_pf_df.TestPnL.sum():,.2f} THB")
print(f"Frozen BASE OOS P&L: {true_pf_df.BasePnL.sum():,.2f} THB")
print(f"Delta vs BASE: {true_pf_df.DeltaVsBase.sum():,.2f} THB")
print(f"Variants: {', '.join(true_pf_df.Selected.tolist())}")

if int((true_pf_df.TestPnL>0).sum())>=2 and true_pf_df.DeltaVsBase.sum()>0:
    print("RESEARCH DECISION: PORTFOLIO_CANDIDATE_FOR_FURTHER_VALIDATION")
else:
    print("RESEARCH DECISION: KEEP_CURRENT_PORTFOLIO_RULES")
print("PRODUCTION CONFIGURATION UNCHANGED")
print("TRUE GLOBAL PORTFOLIO BACKTEST COMPLETE")



In [ ]:
# ============================================================
# CELL 12.13 — RESEARCH ONLY: TRANSACTION COST + SLIPPAGE + GAP STRESS v2
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — TRUE GLOBAL PORTFOLIO EXECUTION STRESS TEST")
print("=" * 90)

assert "ps" in globals()
assert "SELECTED_SYMBOLS" in globals()

def _true_portfolio_stress(
    signals, start_date, end_date,
    risk_pct=0.020, target2_pct=0.15,
    max_hold_days=20, max_positions=5,
    commission_pct=0.0015, slippage_pct=0.0010,
    gap_extra_pct=0.0025,
):
    """Same global portfolio accounting as 12.12, with execution friction.

    commission_pct and slippage_pct are applied per side.
    gap_extra_pct is an additional adverse exit shock only on STOP_GAP exits.
    Position sizing is based on the pre-friction signal entry/stop, while cash
    affordability uses the slippage-adjusted entry execution price.
    """
    start_date=pd.Timestamp(start_date); end_date=pd.Timestamp(end_date)
    s=signals[(signals["SignalDate"]>=start_date)&(signals["SignalDate"]<=end_date)].copy()
    if s.empty:
        return {"PnL":0.0,"Trades":0,"WinRate":np.nan,"MaxDD":0.0,
                "EndEquity":INITIAL_CAPITAL,"PF":np.nan,"GapRate":np.nan,
                "ExposurePct":0.0,"Skipped":0}

    market_dates=set()
    for sym in SELECTED_SYMBOLS:
        market_dates.update(
            pd.Timestamp(x) for x in canonical_data[sym].index
            if start_date<=pd.Timestamp(x)<=end_date
        )
    market_dates=sorted(market_dates)

    entries=s.sort_values(
        ["EntryDate","SignalDate","Symbol"],
        ascending=[True,False,True]
    )
    by_entry={d:g for d,g in entries.groupby("EntryDate",sort=False)}

    cash=float(INITIAL_CAPITAL)
    open_pos=[]
    trades=[]
    equity_points=[]
    skipped=0

    for dt in market_dates:
        dt=pd.Timestamp(dt)

        # EXIT FIRST
        remaining=[]
        for p in open_pos:
            if dt < p["EntryDate"]:
                remaining.append(p); continue
            df=canonical_data[p["Symbol"]]
            if dt not in df.index:
                remaining.append(p); continue

            bar=df.loc[dt]
            op,hi,lo,cl=map(float,[bar["Open"],bar["High"],bar["Low"],bar["Close"]])
            raw_px=None; reason=None

            if op<=p["Stop"]:
                raw_px,reason=op,"STOP_GAP"
            elif lo<=p["Stop"]:
                raw_px,reason=p["Stop"],"STOP"
            elif op>=p["Target"]:
                raw_px,reason=op,"TARGET2_GAP"
            elif hi>=p["Target"]:
                raw_px,reason=p["Target"],"TARGET2"
            elif dt>=p["MaxDate"]:
                raw_px,reason=cl,"MAX_HOLD"

            if raw_px is None:
                remaining.append(p); continue

            # Adverse execution slippage on a sell.
            extra_gap = gap_extra_pct if reason=="STOP_GAP" else 0.0
            px = raw_px * (1.0 - slippage_pct - extra_gap)
            exit_value = px * p["Shares"]
            exit_fee = exit_value * commission_pct
            cash += exit_value - exit_fee

            gross_pnl=(raw_px-p["EntryRaw"])*p["Shares"]
            net_pnl=(px-p["EntryExec"])*p["Shares"] - p["EntryFee"] - exit_fee

            trades.append({
                "Symbol":p["Symbol"],"SignalDate":p["SignalDate"],
                "EntryDate":p["EntryDate"],"ExitDate":dt,
                "EntryPrice":p["EntryExec"],"ExitPrice":px,
                "Shares":p["Shares"],"PnL":net_pnl,"GrossPnL":gross_pnl,
                "Reason":reason,"GapLoss":bool(reason=="STOP_GAP" and net_pnl<0),
                "EntryValue":p["EntryExec"]*p["Shares"],
            })

        open_pos=remaining

        # ENTRIES
        todays=by_entry.get(dt)
        if todays is not None:
            for _,sig in todays.iterrows():
                if len(open_pos)>=max_positions:
                    skipped+=1; continue
                sym=sig["Symbol"]
                if any(p["Symbol"]==sym for p in open_pos):
                    skipped+=1; continue

                entry_raw=float(sig["EntryPrice"])
                stop=float(sig["StopPrice"])
                if not(np.isfinite(entry_raw) and entry_raw>0 and
                       np.isfinite(stop) and 0<stop<entry_raw):
                    skipped+=1; continue

                marked=sum(
                    p["Shares"]*float(canonical_data[p["Symbol"]].loc[dt,"Close"])
                    for p in open_pos
                    if dt in canonical_data[p["Symbol"]].index
                )
                current_equity=cash+marked
                risk_budget=current_equity*risk_pct
                rps=entry_raw-stop
                risk_lots=int((risk_budget/rps)//LOT_SIZE)

                entry_exec=entry_raw*(1.0+slippage_pct)
                cash_lots=int((cash/entry_exec)//LOT_SIZE)
                lots=min(risk_lots,cash_lots)
                shares=lots*LOT_SIZE
                if shares<LOT_SIZE:
                    skipped+=1; continue

                entry_value=entry_exec*shares
                entry_fee=entry_value*commission_pct
                total_entry_cash=entry_value+entry_fee
                if total_entry_cash>cash+1e-9:
                    skipped+=1; continue

                cash-=total_entry_cash
                open_pos.append({
                    "Symbol":sym,"SignalDate":pd.Timestamp(sig["SignalDate"]),
                    "EntryDate":dt,"EntryRaw":entry_raw,"EntryExec":entry_exec,
                    "Stop":stop,
                    "Target":entry_raw*(1+target2_pct),
                    "Shares":int(shares),"MaxDate":dt+pd.Timedelta(days=max_hold_days),
                    "EntryFee":entry_fee,
                })

        marked=sum(
            p["Shares"]*float(canonical_data[p["Symbol"]].loc[dt,"Close"])
            for p in open_pos if dt in canonical_data[p["Symbol"]].index
        )
        equity=cash+marked
        equity_points.append({
            "Date":dt,"Equity":equity,"Exposure":marked,
            "OpenPositions":len(open_pos),
        })

    # Explicit EOD censoring at last available close.
    for p in list(open_pos):
        df=canonical_data[p["Symbol"]]
        fut=df[(df.index>=p["EntryDate"])&(df.index<=end_date)]
        if fut.empty: continue
        dt=pd.Timestamp(fut.index[-1])
        raw_px=float(fut.iloc[-1]["Close"])
        px=raw_px*(1.0-slippage_pct)
        exit_value=px*p["Shares"]
        exit_fee=exit_value*commission_pct
        cash+=exit_value-exit_fee
        net_pnl=(px-p["EntryExec"])*p["Shares"]-p["EntryFee"]-exit_fee
        gross_pnl=(raw_px-p["EntryRaw"])*p["Shares"]
        trades.append({
            "Symbol":p["Symbol"],"SignalDate":p["SignalDate"],
            "EntryDate":p["EntryDate"],"ExitDate":dt,
            "EntryPrice":p["EntryExec"],"ExitPrice":px,
            "Shares":p["Shares"],"PnL":net_pnl,"GrossPnL":gross_pnl,
            "Reason":"EOD","GapLoss":False,
            "EntryValue":p["EntryExec"]*p["Shares"],
        })

    td=pd.DataFrame(trades)
    ed=pd.DataFrame(equity_points)
    if td.empty:
        return {"PnL":0.0,"Trades":0,"WinRate":np.nan,"MaxDD":0.0,
                "EndEquity":cash,"PF":np.nan,"GapRate":np.nan,
                "ExposurePct":0.0,"Skipped":skipped}

    gross_profit=td.loc[td.PnL>0,"PnL"].sum()
    gross_loss=-td.loc[td.PnL<0,"PnL"].sum()
    peak=ed["Equity"].cummax()
    dd=(ed["Equity"]/peak-1)*100

    return {
        "PnL":float(td.PnL.sum()),
        "Trades":int(len(td)),
        "WinRate":float((td.PnL>0).mean()*100),
        "MaxDD":float(dd.min()),
        "EndEquity":float(cash),
        "PF":float(gross_profit/gross_loss) if gross_loss>0 else np.inf,
        "GapRate":float(td["GapLoss"].mean()*100),
        "ExposurePct":float(ed["Exposure"].mean()/INITIAL_CAPITAL*100),
        "Skipped":int(skipped),
    }

# Use the exact three OOS windows from the True Global Portfolio test.
stress_folds=[
    ("Fold1","2024-05-27","2025-12-08"),
    ("Fold2","2025-09-17","2026-06-25"),
    ("Fold3","2026-04-28","2026-10-02"),
]

SCENARIOS=[
    ("NO_FRICTION",0.0000,0.0000,0.0000),
    ("COST_015_SLIP_010_GAP_025",0.0015,0.0010,0.0025),
    ("COST_020_SLIP_025_GAP_050",0.0020,0.0025,0.0050),
    ("COST_025_SLIP_050_GAP_100",0.0025,0.0050,0.0100),
    ("GAP_ONLY_050",0.0000,0.0000,0.0050),
    ("SLIP_ONLY_025",0.0000,0.0025,0.0000),
]

rows=[]
for scenario,comm,slip,gap in SCENARIOS:
    fold_results=[]
    for fold,start,end in stress_folds:
        r=_true_portfolio_stress(
            ps,start,end,
            risk_pct=0.020,target2_pct=0.15,max_hold_days=20,max_positions=5,
            commission_pct=comm,slippage_pct=slip,gap_extra_pct=gap,
        )
        fold_results.append(r)
        rows.append({
            "Scenario":scenario,"Fold":fold,
            "PnL":r["PnL"],"Trades":r["Trades"],"WinRate":r["WinRate"],
            "MaxDD":r["MaxDD"],"PF":r["PF"],"GapRate":r["GapRate"],
            "ExposurePct":r["ExposurePct"],"Skipped":r["Skipped"],
        })

stress_df=pd.DataFrame(rows)

summary=[]
for scenario in SCENARIOS:
    name=scenario[0]
    x=stress_df[stress_df["Scenario"]==name]
    summary.append({
        "Scenario":name,
        "TotalOOSPnL":x["PnL"].sum(),
        "PositiveFolds":int((x["PnL"]>0).sum()),
        "WorstFoldPnL":x["PnL"].min(),
        "WorstMaxDD":x["MaxDD"].min(),
        "MinPF":x["PF"].min(),
        "AvgGapRate":x["GapRate"].mean(),
        "AvgExposurePct":x["ExposurePct"].mean(),
    })

stress_summary=pd.DataFrame(summary)
display(stress_summary)

print("\nFROZEN BASE 2% / 5-POSITION STRESS TEST")
print(stress_summary.to_string(index=False, float_format=lambda x:f"{x:,.2f}"))

# Break-even friction search: equal commission/slippage per side,
# with no extra gap shock, where total OOS P&L crosses zero.
def _total_pnl_equal_friction(rate):
    total=0.0
    for _,start,end in stress_folds:
        total += _true_portfolio_stress(
            ps,start,end,0.020,0.15,20,5,
            commission_pct=rate,slippage_pct=rate,gap_extra_pct=0.0
        )["PnL"]
    return total

lo,hi=0.0,0.01
for _ in range(35):
    mid=(lo+hi)/2
    if _total_pnl_equal_friction(mid)>0:
        lo=mid
    else:
        hi=mid
break_even=(lo+hi)/2

print(f"\nBREAK-EVEN EQUAL COMMISSION+SLIPPAGE: ~{break_even*100:.3f}% per side")
print(f"At that point combined round-trip friction is ~{break_even*200:.3f}% (excluding gap shock).")

# Promotion gate: stress must remain positive in >=2/3 folds and retain PF > 1
# under the realistic scenario. This cell is research-only and never changes production.
realistic=stress_summary[stress_summary.Scenario=="COST_015_SLIP_010_GAP_025"].iloc[0]
moderate=stress_summary[stress_summary.Scenario=="COST_020_SLIP_025_GAP_050"].iloc[0]

if realistic.TotalOOSPnL>0 and realistic.PositiveFolds>=2 and realistic.MinPF>1.0:
    decision="ROBUST_TO_REALISTIC_FRICTION"
else:
    decision="FRAGILE_TO_REALISTIC_FRICTION"

print("\nSTRESS DECISION:",decision)
print("PRODUCTION CONFIGURATION UNCHANGED")
print("TRANSACTION COST + SLIPPAGE + GAP STRESS TEST COMPLETE")



In [ ]:
# ============================================================
# CELL 12.14 — RESEARCH ONLY: EXECUTION-AWARE ENTRY FILTER STUDY
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — EXECUTION-AWARE ENTRY FILTER / TURNOVER STUDY")
print("=" * 90)

assert "ps" in globals()
assert "SELECTED_SYMBOLS" in globals()
assert "_true_portfolio_stress" in globals()

# Freeze the current production signal set. This study NEVER changes
# PRODUCTION_* variables or SELECTED_SYMBOLS.
base_exec_signals = ps.copy()

# Attach signal-day features without changing the signal-generation logic.
# All filters below are evaluated using information known on SignalDate.
def _attach_execution_features(signals):
    out = signals.copy()
    scores = []
    vols = []
    atr_pct = []
    trend = []
    weekly = []

    for _, row in out.iterrows():
        sym = row["Symbol"]
        dt = pd.Timestamp(row["SignalDate"])
        df = canonical_data[sym]
        if dt not in df.index:
            scores.append(np.nan)
            vols.append(np.nan)
            atr_pct.append(np.nan)
            trend.append(False)
            weekly.append(False)
            continue
        bar = df.loc[dt]
        close = float(bar["Close"])
        atr = float(bar["ATR14"]) if pd.notna(bar["ATR14"]) else np.nan
        scores.append(float(bar["CANONICAL_SCORE"]))
        vols.append(float(bar["VOLUME_RATIO"]))
        atr_pct.append((atr / close) if close > 0 and np.isfinite(atr) else np.nan)
        trend.append(bool(bar["CANONICAL_TREND_BULL"]))
        weekly.append(bool(bar["CANONICAL_WEEKLY_CONFIRM"]))

    out["SignalScore"] = scores
    out["SignalVolumeRatio"] = vols
    out["SignalATRpct"] = atr_pct
    out["SignalTrendBull"] = trend
    out["SignalWeeklyConfirm"] = weekly
    return out

exec_signals = _attach_execution_features(base_exec_signals)

assert exec_signals["SignalScore"].notna().all()
assert exec_signals["SignalVolumeRatio"].notna().all()
assert exec_signals["SignalATRpct"].notna().all()

# Small, pre-declared filter family. The goal is to reduce low-quality
# entries/turnover, not to optimize dozens of parameters.
FILTERS = [
    {
        "name": "BASE",
        "mask": lambda d: d["SignalScore"] >= 4,
    },
    {
        "name": "SCORE5",
        "mask": lambda d: d["SignalScore"] >= 5,
    },
    {
        "name": "SCORE4_VOL14",
        "mask": lambda d: (d["SignalScore"] >= 4) & (d["SignalVolumeRatio"] >= 1.4),
    },
    {
        "name": "SCORE4_ATR15",
        "mask": lambda d: (d["SignalScore"] >= 4) & (d["SignalATRpct"] >= 0.015),
    },
    {
        "name": "SCORE4_ATR20",
        "mask": lambda d: (d["SignalScore"] >= 4) & (d["SignalATRpct"] >= 0.020),
    },
]

FILTERED_SIGNALS = {
    f["name"]: exec_signals.loc[f["mask"](exec_signals)].copy()
    for f in FILTERS
}

print("\nFILTER COUNTS")
for name, df in FILTERED_SIGNALS.items():
    print(f"{name:<18} signals={len(df):>5}")

# Use the same three nested OOS fold geometry as the True Global Portfolio.
portfolio_dates = pd.to_datetime(exec_signals["SignalDate"]).sort_values().unique()
n_dates = len(portfolio_dates)
nested_folds = [
    (0.00, 0.50, 0.65, 0.80),
    (0.15, 0.65, 0.77, 0.90),
    (0.25, 0.75, 0.85, 1.00),
]

# Realistic execution assumptions from Cell 12.13.
FRICTION = {
    "commission_pct": 0.0015,
    "slippage_pct": 0.0010,
    "gap_extra_pct": 0.0025,
}

nested_rows = []

for fold_no, (a, b, c, d) in enumerate(nested_folds, 1):
    train_start = portfolio_dates[int(n_dates * a)]
    train_end = portfolio_dates[int(n_dates * b) - 1]
    valid_start = portfolio_dates[int(n_dates * b)]
    valid_end = portfolio_dates[int(n_dates * c) - 1]
    test_start_n = portfolio_dates[int(n_dates * c)]
    test_end_n = portfolio_dates[int(n_dates * d) - 1]

    validation_rows = []

    for name, sigs in FILTERED_SIGNALS.items():
        val = _true_portfolio_stress(
            sigs,
            valid_start,
            valid_end,
            risk_pct=0.020,
            target2_pct=0.15,
            max_hold_days=20,
            max_positions=5,
            **FRICTION,
        )
        validation_rows.append((name, val))

    # Require at least 10 validation trades when possible.
    eligible = [x for x in validation_rows if x[1]["Trades"] >= 10]
    if not eligible:
        eligible = validation_rows

    chosen_name, chosen_val = max(
        eligible,
        key=lambda x: x[1]["PnL"],
    )

    test = _true_portfolio_stress(
        FILTERED_SIGNALS[chosen_name],
        test_start_n,
        test_end_n,
        risk_pct=0.020,
        target2_pct=0.15,
        max_hold_days=20,
        max_positions=5,
        **FRICTION,
    )

    base_test = _true_portfolio_stress(
        FILTERED_SIGNALS["BASE"],
        test_start_n,
        test_end_n,
        risk_pct=0.020,
        target2_pct=0.15,
        max_hold_days=20,
        max_positions=5,
        **FRICTION,
    )

    nested_rows.append({
        "Fold": fold_no,
        "TrainEnd": pd.Timestamp(train_end).date(),
        "ValidEnd": pd.Timestamp(valid_end).date(),
        "TestStart": pd.Timestamp(test_start_n).date(),
        "TestEnd": pd.Timestamp(test_end_n).date(),
        "Selected": chosen_name,
        "ValidPnL": chosen_val["PnL"],
        "ValidTrades": chosen_val["Trades"],
        "TestPnL": test["PnL"],
        "TestTrades": test["Trades"],
        "TestWinRate": test["WinRate"],
        "TestMaxDD": test["MaxDD"],
        "TestPF": test["PF"],
        "BaseTestPnL": base_test["PnL"],
        "DeltaVsBase": test["PnL"] - base_test["PnL"],
    })

entry_filter_nested_df = pd.DataFrame(nested_rows)
display(entry_filter_nested_df)

print("\nNESTED EXECUTION-AWARE FILTER SUMMARY")
print(f"Positive OOS folds: {int((entry_filter_nested_df['TestPnL'] > 0).sum())}/{len(entry_filter_nested_df)}")
print(f"Selected net OOS P&L: {entry_filter_nested_df['TestPnL'].sum():,.2f} THB")
print(f"Frozen BASE net OOS P&L: {entry_filter_nested_df['BaseTestPnL'].sum():,.2f} THB")
print(f"Delta vs BASE: {entry_filter_nested_df['DeltaVsBase'].sum():,.2f} THB")
print(f"Selected filters: {', '.join(entry_filter_nested_df['Selected'].tolist())}")

# Final unseen audit: evaluate the selected fold choices only on their
# corresponding unseen windows. No test data is used for selection.
positive_oos = int((entry_filter_nested_df["TestPnL"] > 0).sum())
total_delta = float(entry_filter_nested_df["DeltaVsBase"].sum())

if positive_oos >= 2 and total_delta > 0:
    decision = "EXECUTION_AWARE_FILTER_CANDIDATE"
else:
    decision = "KEEP_CURRENT_ENTRY_RULES"

print("\nRESEARCH DECISION:", decision)
print("PRODUCTION CONFIGURATION UNCHANGED")

# Also report the frozen BASE under the same realistic friction so the
# comparison is apples-to-apples.
base_realistic_total = float(entry_filter_nested_df["BaseTestPnL"].sum())
print(f"Frozen BASE realistic-friction OOS: {base_realistic_total:,.2f} THB")
print("EXECUTION-AWARE ENTRY FILTER STUDY COMPLETE")


In [ ]:
# ============================================================
# CELL 12.15 — RESEARCH ONLY: CANDIDATE FRICTION ROBUSTNESS
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — EXECUTION-AWARE CANDIDATE FRICTION ROBUSTNESS")
print("=" * 90)

assert "FILTERED_SIGNALS" in globals()
assert "entry_filter_nested_df" in globals()
assert "exec_signals" in globals()

candidate = FILTERED_SIGNALS["SCORE4_VOL14"].copy()
base = FILTERED_SIGNALS["BASE"].copy()

stress_folds = [
    ("Fold1","2024-05-27","2025-12-08"),
    ("Fold2","2025-09-17","2026-06-25"),
    ("Fold3","2026-04-28","2026-10-02"),
]

SCENARIOS_1215 = [
    ("NO_FRICTION",0.0000,0.0000,0.0000),
    ("REALISTIC",0.0015,0.0010,0.0025),
    ("MODERATE",0.0020,0.0025,0.0050),
    ("HARSH",0.0025,0.0050,0.0100),
    ("SLIP_ONLY_025",0.0000,0.0025,0.0000),
]

rows=[]
for scenario,comm,slip,gap in SCENARIOS_1215:
    for fold,start,end in stress_folds:
        cand=_true_portfolio_stress(
            candidate,start,end,
            risk_pct=0.020,target2_pct=0.15,
            max_hold_days=20,max_positions=5,
            commission_pct=comm,slippage_pct=slip,gap_extra_pct=gap,
        )
        bas=_true_portfolio_stress(
            base,start,end,
            risk_pct=0.020,target2_pct=0.15,
            max_hold_days=20,max_positions=5,
            commission_pct=comm,slippage_pct=slip,gap_extra_pct=gap,
        )
        rows.append({
            "Scenario":scenario,"Fold":fold,
            "CandidatePnL":cand["PnL"],"BasePnL":bas["PnL"],
            "Delta":cand["PnL"]-bas["PnL"],
            "CandidateTrades":cand["Trades"],"BaseTrades":bas["Trades"],
            "CandidateWinRate":cand["WinRate"],
            "CandidateMaxDD":cand["MaxDD"],"CandidatePF":cand["PF"],
        })

candidate_stress_df=pd.DataFrame(rows)

summary=[]
for scenario,_,_,_ in SCENARIOS_1215:
    x=candidate_stress_df[candidate_stress_df["Scenario"]==scenario]
    summary.append({
        "Scenario":scenario,
        "CandidateOOS":x["CandidatePnL"].sum(),
        "BaseOOS":x["BasePnL"].sum(),
        "DeltaVsBase":x["Delta"].sum(),
        "CandidatePositiveFolds":int((x["CandidatePnL"]>0).sum()),
        "BasePositiveFolds":int((x["BasePnL"]>0).sum()),
        "WorstCandidateFold":x["CandidatePnL"].min(),
        "WorstCandidateDD":x["CandidateMaxDD"].min(),
        "MinCandidatePF":x["CandidatePF"].min(),
        "CandidateTrades":int(x["CandidateTrades"].sum()),
        "BaseTrades":int(x["BaseTrades"].sum()),
    })

candidate_stress_summary=pd.DataFrame(summary)
display(candidate_stress_summary)

print("\nCANDIDATE VS FROZEN BASE — FRICTION STRESS")
print(candidate_stress_summary.to_string(index=False,float_format=lambda x:f"{x:,.2f}"))

realistic_row=candidate_stress_summary[candidate_stress_summary.Scenario=="REALISTIC"].iloc[0]
moderate_row=candidate_stress_summary[candidate_stress_summary.Scenario=="MODERATE"].iloc[0]
harsh_row=candidate_stress_summary[candidate_stress_summary.Scenario=="HARSH"].iloc[0]

trade_reduction=(1-float(realistic_row["CandidateTrades"])/float(realistic_row["BaseTrades"]))*100
print(f"\nRealistic-friction trade reduction: {trade_reduction:.2f}%")

if (
    realistic_row["CandidateOOS"] > 0
    and realistic_row["CandidatePositiveFolds"] >= 2
    and realistic_row["DeltaVsBase"] > 0
    and realistic_row["MinCandidatePF"] > 1.0
):
    candidate_decision="CANDIDATE_SURVIVES_REALISTIC_FRICTION"
else:
    candidate_decision="CANDIDATE_FAILS_REALISTIC_FRICTION"

if (
    moderate_row["CandidateOOS"] > moderate_row["BaseOOS"]
    and moderate_row["DeltaVsBase"] > 0
):
    moderate_decision="MODERATE_ADVANTAGE"
else:
    moderate_decision="MODERATE_NOT_ROBUST"

print("Realistic decision:",candidate_decision)
print("Moderate decision :",moderate_decision)
print("PRODUCTION CONFIGURATION UNCHANGED")
print("CANDIDATE FRICTION ROBUSTNESS COMPLETE")


In [ ]:
# ============================================================
# CELL 12.16 — RESEARCH ONLY: COMPOSITE ENTRY QUALITY STUDY
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — COMPOSITE ENTRY QUALITY / NET-PROFIT STUDY")
print("=" * 90)

assert "exec_signals" in globals()
assert "_true_portfolio_stress" in globals()

# Deliberately compact candidate family. Every feature is known at SignalDate.
COMPOSITE_FILTERS = {
    "BASE": lambda d: d["SignalScore"] >= 4,
    "SCORE5": lambda d: d["SignalScore"] >= 5,
    "SCORE5_VOL14": lambda d: (d["SignalScore"] >= 5) & (d["SignalVolumeRatio"] >= 1.4),
    "SCORE5_ATR15": lambda d: (d["SignalScore"] >= 5) & (d["SignalATRpct"] >= 0.015),
    "SCORE4_VOL16": lambda d: (d["SignalScore"] >= 4) & (d["SignalVolumeRatio"] >= 1.6),
    "SCORE4_VOL14_ATR15": lambda d: (d["SignalScore"] >= 4) & (d["SignalVolumeRatio"] >= 1.4) & (d["SignalATRpct"] >= 0.015),
    "SCORE4_TRENDBULL_VOL14": lambda d: (d["SignalScore"] >= 4) & d["SignalTrendBull"] & (d["SignalVolumeRatio"] >= 1.4),
}

COMPOSITE_SIGNALS = {
    name: exec_signals.loc[mask(exec_signals)].copy()
    for name, mask in COMPOSITE_FILTERS.items()
}

print("\nFILTER COUNTS")
for name, df in COMPOSITE_SIGNALS.items():
    print(f"{name:<28} signals={len(df):>5}")

FRICTION_COMPOSITE = {
    "commission_pct": 0.0015,
    "slippage_pct": 0.0010,
    "gap_extra_pct": 0.0025,
}

nested_rows=[]
for fold_no,(a,b,c,d) in enumerate(nested_folds,1):
    train_start=portfolio_dates[int(n_dates*a)]
    train_end=portfolio_dates[int(n_dates*b)-1]
    valid_start=portfolio_dates[int(n_dates*b)]
    valid_end=portfolio_dates[int(n_dates*c)-1]
    test_start_n=portfolio_dates[int(n_dates*c)]
    test_end_n=portfolio_dates[int(n_dates*d)-1]

    vals=[]
    for name,sigs in COMPOSITE_SIGNALS.items():
        vr=_true_portfolio_stress(
            sigs,valid_start,valid_end,
            risk_pct=0.020,target2_pct=0.15,max_hold_days=20,max_positions=5,
            **FRICTION_COMPOSITE,
        )
        vals.append((name,vr))

    # Prefer validation candidates with PF > 1 and enough trades.
    robust=[x for x in vals if x[1]["Trades"]>=10 and np.isfinite(x[1]["PF"]) and x[1]["PF"]>1.0]
    if not robust:
        robust=[x for x in vals if x[1]["Trades"]>=10]
    chosen_name,chosen_val=max(robust,key=lambda x:x[1]["PnL"])

    te=_true_portfolio_stress(
        COMPOSITE_SIGNALS[chosen_name],test_start_n,test_end_n,
        risk_pct=0.020,target2_pct=0.15,max_hold_days=20,max_positions=5,
        **FRICTION_COMPOSITE,
    )
    base_te=_true_portfolio_stress(
        COMPOSITE_SIGNALS["BASE"],test_start_n,test_end_n,
        risk_pct=0.020,target2_pct=0.15,max_hold_days=20,max_positions=5,
        **FRICTION_COMPOSITE,
    )

    nested_rows.append({
        "Fold":fold_no,
        "TrainEnd":pd.Timestamp(train_end).date(),
        "ValidEnd":pd.Timestamp(valid_end).date(),
        "TestStart":pd.Timestamp(test_start_n).date(),
        "TestEnd":pd.Timestamp(test_end_n).date(),
        "Selected":chosen_name,
        "ValidPnL":chosen_val["PnL"],
        "ValidPF":chosen_val["PF"],
        "TestPnL":te["PnL"],
        "TestTrades":te["Trades"],
        "TestWinRate":te["WinRate"],
        "TestMaxDD":te["MaxDD"],
        "TestPF":te["PF"],
        "BaseTestPnL":base_te["PnL"],
        "DeltaVsBase":te["PnL"]-base_te["PnL"],
    })

composite_nested_df=pd.DataFrame(nested_rows)
display(composite_nested_df)

print("\nCOMPOSITE NESTED SUMMARY")
print(f"Positive OOS folds: {int((composite_nested_df.TestPnL>0).sum())}/{len(composite_nested_df)}")
print(f"Selected net OOS P&L: {composite_nested_df.TestPnL.sum():,.2f} THB")
print(f"Frozen BASE net OOS P&L: {composite_nested_df.BaseTestPnL.sum():,.2f} THB")
print(f"Delta vs BASE: {composite_nested_df.DeltaVsBase.sum():,.2f} THB")
print(f"Selected filters: {', '.join(composite_nested_df.Selected.tolist())}")

if (
    int((composite_nested_df.TestPnL>0).sum()) >= 2
    and composite_nested_df.DeltaVsBase.sum() > 0
    and (composite_nested_df.TestPF > 1.0).all()
):
    composite_decision="COMPOSITE_ENTRY_CANDIDATE"
else:
    composite_decision="KEEP_CURRENT_ENTRY_RULES"

print("RESEARCH DECISION:",composite_decision)
print("PRODUCTION CONFIGURATION UNCHANGED")
print("COMPOSITE ENTRY QUALITY STUDY COMPLETE")


In [ ]:
# ============================================================
# CELL 12.17 — RESEARCH ONLY: SCORE-4 COMPONENT QUALITY AUDIT
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — SCORE-4 COMPONENT QUALITY AUDIT")
print("=" * 90)

assert "exec_signals" in globals()
assert "_true_portfolio_stress" in globals()

COMPONENT_COLS = {
    "MACD": "CANONICAL_MACD_BULL",
    "STOCH": "CANONICAL_STOCH_BULL",
    "TREND": "CANONICAL_TREND_BULL",
    "EARLY": "CANONICAL_EARLY_ENTRY",
    "VOLUME": "CANONICAL_VOLUME_CONFIRM",
}

def _attach_components(signals):
    out=signals.copy()
    for label,col in COMPONENT_COLS.items():
        vals=[]
        for _,row in out.iterrows():
            df=canonical_data[row["Symbol"]]
            dt=pd.Timestamp(row["SignalDate"])
            vals.append(bool(df.loc[dt,col]) if dt in df.index else False)
        out[f"Comp_{label}"]=vals
    return out

component_signals=_attach_components(exec_signals)

COMPONENT_FILTERS={
    "BASE": lambda d: d["SignalScore"]>=4,
    "SCORE5": lambda d: d["SignalScore"]>=5,
    "SCORE4_NO_MACD_MISSING": lambda d: (d["SignalScore"]>=4)&d["Comp_MACD"],
    "SCORE4_NO_STOCH_MISSING": lambda d: (d["SignalScore"]>=4)&d["Comp_STOCH"],
    "SCORE4_NO_TREND_MISSING": lambda d: (d["SignalScore"]>=4)&d["Comp_TREND"],
    "SCORE4_NO_EARLY_MISSING": lambda d: (d["SignalScore"]>=4)&d["Comp_EARLY"],
    "SCORE4_NO_VOLUME_MISSING": lambda d: (d["SignalScore"]>=4)&d["Comp_VOLUME"],
}

COMPONENT_SIGNALS={
    name:component_signals.loc[mask(component_signals)].copy()
    for name,mask in COMPONENT_FILTERS.items()
}

print("\nFILTER COUNTS")
for name,df in COMPONENT_SIGNALS.items():
    print(f"{name:<30} signals={len(df):>5}")

FRICTION_COMPONENT={
    "commission_pct":0.0015,
    "slippage_pct":0.0010,
    "gap_extra_pct":0.0025,
}

rows=[]
for fold_no,(a,b,c,d) in enumerate(nested_folds,1):
    valid_start=portfolio_dates[int(n_dates*b)]
    valid_end=portfolio_dates[int(n_dates*c)-1]
    test_start_n=portfolio_dates[int(n_dates*c)]
    test_end_n=portfolio_dates[int(n_dates*d)-1]

    vals=[]
    for name,sigs in COMPONENT_SIGNALS.items():
        vr=_true_portfolio_stress(
            sigs,valid_start,valid_end,
            risk_pct=0.020,target2_pct=0.15,max_hold_days=20,max_positions=5,
            **FRICTION_COMPONENT,
        )
        vals.append((name,vr))

    eligible=[x for x in vals if x[1]["Trades"]>=10]
    if not eligible:
        eligible=vals

    chosen_name,chosen_val=max(eligible,key=lambda x:x[1]["PnL"])

    te=_true_portfolio_stress(
        COMPONENT_SIGNALS[chosen_name],test_start_n,test_end_n,
        risk_pct=0.020,target2_pct=0.15,max_hold_days=20,max_positions=5,
        **FRICTION_COMPONENT,
    )
    base_te=_true_portfolio_stress(
        COMPONENT_SIGNALS["BASE"],test_start_n,test_end_n,
        risk_pct=0.020,target2_pct=0.15,max_hold_days=20,max_positions=5,
        **FRICTION_COMPONENT,
    )

    rows.append({
        "Fold":fold_no,
        "Selected":chosen_name,
        "ValidPnL":chosen_val["PnL"],
        "ValidPF":chosen_val["PF"],
        "TestPnL":te["PnL"],
        "TestTrades":te["Trades"],
        "TestWinRate":te["WinRate"],
        "TestMaxDD":te["MaxDD"],
        "TestPF":te["PF"],
        "BaseTestPnL":base_te["PnL"],
        "DeltaVsBase":te["PnL"]-base_te["PnL"],
    })

component_nested_df=pd.DataFrame(rows)
display(component_nested_df)

print("\nSCORE-4 COMPONENT NESTED SUMMARY")
print(f"Positive OOS folds: {int((component_nested_df.TestPnL>0).sum())}/{len(component_nested_df)}")
print(f"Selected net OOS P&L: {component_nested_df.TestPnL.sum():,.2f} THB")
print(f"Frozen BASE net OOS P&L: {component_nested_df.BaseTestPnL.sum():,.2f} THB")
print(f"Delta vs BASE: {component_nested_df.DeltaVsBase.sum():,.2f} THB")
print(f"Selected filters: {', '.join(component_nested_df.Selected.tolist())}")

if (
    int((component_nested_df.TestPnL>0).sum())>=2
    and component_nested_df.DeltaVsBase.sum()>0
    and (component_nested_df.TestPF>1.0).all()
):
    decision="SCORE_COMPONENT_CANDIDATE"
else:
    decision="KEEP_CURRENT_ENTRY_RULES"

print("RESEARCH DECISION:",decision)
print("PRODUCTION CONFIGURATION UNCHANGED")
print("SCORE-4 COMPONENT QUALITY AUDIT COMPLETE")


In [ ]:
# ============================================================
# CELL 12.18 — RESEARCH ONLY: UNIVERSE BREADTH / REGIME FILTER
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — UNIVERSE BREADTH / REGIME FILTER")
print("=" * 90)

assert "exec_signals" in globals()
assert "_true_portfolio_stress" in globals()

# Build breadth from the frozen 8-symbol production universe using only
# completed SignalDate information. No future bars are used.
breadth_rows=[]
for dt in sorted(pd.to_datetime(exec_signals["SignalDate"]).unique()):
    trend_vals=[]
    weekly_vals=[]
    for sym in SELECTED_SYMBOLS:
        df=canonical_data[sym]
        if dt in df.index:
            trend_vals.append(bool(df.loc[dt,"CANONICAL_TREND_BULL"]))
            weekly_vals.append(bool(df.loc[dt,"CANONICAL_WEEKLY_CONFIRM"]))
    breadth_rows.append({
        "SignalDate":pd.Timestamp(dt),
        "TrendBreadth":np.mean(trend_vals) if trend_vals else np.nan,
        "WeeklyBreadth":np.mean(weekly_vals) if weekly_vals else np.nan,
    })

breadth_df=pd.DataFrame(breadth_rows)

regime_signals=exec_signals.merge(breadth_df,on="SignalDate",how="left")
assert regime_signals["TrendBreadth"].notna().all()

REGIME_FILTERS={
    "BASE":lambda d: d["SignalScore"]>=4,
    "TREND_BREADTH25":lambda d:(d["SignalScore"]>=4)&(d["TrendBreadth"]>=0.25),
    "TREND_BREADTH375":lambda d:(d["SignalScore"]>=4)&(d["TrendBreadth"]>=0.375),
    "TREND_BREADTH50":lambda d:(d["SignalScore"]>=4)&(d["TrendBreadth"]>=0.50),
    "WEEKLY_BREADTH25":lambda d:(d["SignalScore"]>=4)&(d["WeeklyBreadth"]>=0.25),
    "TREND25_VOL14":lambda d:(d["SignalScore"]>=4)&(d["TrendBreadth"]>=0.25)&(d["SignalVolumeRatio"]>=1.4),
}

REGIME_SIGNALS={
    name:regime_signals.loc[mask(regime_signals)].copy()
    for name,mask in REGIME_FILTERS.items()
}

print("\nFILTER COUNTS")
for name,df in REGIME_SIGNALS.items():
    print(f"{name:<24} signals={len(df):>5}")

FRICTION_REGIME={
    "commission_pct":0.0015,
    "slippage_pct":0.0010,
    "gap_extra_pct":0.0025,
}

rows=[]
for fold_no,(a,b,c,d) in enumerate(nested_folds,1):
    valid_start=portfolio_dates[int(n_dates*b)]
    valid_end=portfolio_dates[int(n_dates*c)-1]
    test_start_n=portfolio_dates[int(n_dates*c)]
    test_end_n=portfolio_dates[int(n_dates*d)-1]

    vals=[]
    for name,sigs in REGIME_SIGNALS.items():
        vr=_true_portfolio_stress(
            sigs,valid_start,valid_end,
            risk_pct=0.020,target2_pct=0.15,max_hold_days=20,max_positions=5,
            **FRICTION_REGIME,
        )
        vals.append((name,vr))

    eligible=[x for x in vals if x[1]["Trades"]>=10]
    if not eligible:
        eligible=vals
    chosen_name,chosen_val=max(eligible,key=lambda x:x[1]["PnL"])

    te=_true_portfolio_stress(
        REGIME_SIGNALS[chosen_name],test_start_n,test_end_n,
        risk_pct=0.020,target2_pct=0.15,max_hold_days=20,max_positions=5,
        **FRICTION_REGIME,
    )
    base_te=_true_portfolio_stress(
        REGIME_SIGNALS["BASE"],test_start_n,test_end_n,
        risk_pct=0.020,target2_pct=0.15,max_hold_days=20,max_positions=5,
        **FRICTION_REGIME,
    )

    rows.append({
        "Fold":fold_no,
        "Selected":chosen_name,
        "ValidPnL":chosen_val["PnL"],
        "ValidPF":chosen_val["PF"],
        "TestPnL":te["PnL"],
        "TestTrades":te["Trades"],
        "TestWinRate":te["WinRate"],
        "TestMaxDD":te["MaxDD"],
        "TestPF":te["PF"],
        "BaseTestPnL":base_te["PnL"],
        "DeltaVsBase":te["PnL"]-base_te["PnL"],
    })

regime_nested_df=pd.DataFrame(rows)
display(regime_nested_df)

print("\nREGIME NESTED SUMMARY")
print(f"Positive OOS folds: {int((regime_nested_df.TestPnL>0).sum())}/{len(regime_nested_df)}")
print(f"Selected net OOS P&L: {regime_nested_df.TestPnL.sum():,.2f} THB")
print(f"Frozen BASE net OOS P&L: {regime_nested_df.BaseTestPnL.sum():,.2f} THB")
print(f"Delta vs BASE: {regime_nested_df.DeltaVsBase.sum():,.2f} THB")
print(f"Selected filters: {', '.join(regime_nested_df.Selected.tolist())}")

if (
    int((regime_nested_df.TestPnL>0).sum())>=2
    and regime_nested_df.DeltaVsBase.sum()>0
    and (regime_nested_df.TestPF>1.0).all()
):
    decision="REGIME_FILTER_CANDIDATE"
else:
    decision="KEEP_CURRENT_ENTRY_RULES"

print("RESEARCH DECISION:",decision)
print("PRODUCTION CONFIGURATION UNCHANGED")
print("UNIVERSE BREADTH / REGIME FILTER COMPLETE")


In [ ]:
# ============================================================
# CELL 12.19 — RESEARCH ONLY: TRADE-LEVEL EDGE DECOMPOSITION
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — TRADE-LEVEL EDGE DECOMPOSITION")
print("=" * 90)

assert "ps" in globals()
assert "_true_portfolio_stress" in globals()

# Reconstruct independent trade records for the frozen production signal set.
# This is diagnostic only; no production variable is changed.
diag_rows=[]
for _,sig in ps.iterrows():
    sym=sig["Symbol"]
    dt=pd.Timestamp(sig["SignalDate"])
    entry_date=pd.Timestamp(sig["EntryDate"])
    if sym not in canonical_data or dt not in canonical_data[sym].index:
        continue
    df=canonical_data[sym]
    end_date=entry_date+pd.Timedelta(days=20)
    future=df.loc[(df.index>=entry_date)&(df.index<=end_date)].copy()
    if future.empty:
        continue

    entry=float(sig["EntryPrice"])
    stop=float(sig["StopPrice"])
    target=entry*1.15
    exit_date=pd.Timestamp(future.index[-1])
    exit_price=float(future.iloc[-1]["Close"])
    reason="EOD"

    mae=0.0
    mfe=0.0
    for cur,bar in future.iterrows():
        op,hi,lo,cl=map(float,[bar["Open"],bar["High"],bar["Low"],bar["Close"]])
        mae=min(mae,(lo/entry-1)*100)
        mfe=max(mfe,(hi/entry-1)*100)

        if op<=stop:
            exit_date=pd.Timestamp(cur); exit_price=op; reason="STOP_GAP"; break
        if lo<=stop:
            exit_date=pd.Timestamp(cur); exit_price=stop; reason="STOP"; break
        if hi>=target:
            if op>=target:
                exit_price=op; reason="TARGET2_GAP"
            else:
                exit_price=target; reason="TARGET2"
            exit_date=pd.Timestamp(cur); break

        if pd.Timestamp(cur)>=end_date:
            exit_date=pd.Timestamp(cur); exit_price=cl; reason="MAX_HOLD"; break

    ret=(exit_price/entry-1)*100
    hold=(exit_date-entry_date).days

    diag_rows.append({
        "Symbol":sym,"SignalDate":dt,"EntryDate":entry_date,
        "Score":float(sig.get("CANONICAL_SCORE",np.nan)),
        "VolumeRatio":float(sig.get("VOLUME_RATIO",np.nan)),
        "ATRpct":float(sig.get("ATR14",np.nan))/entry if float(sig.get("EntryPrice",0))>0 else np.nan,
        "ReturnPct":ret,"HoldingDays":hold,"MAE":mae,"MFE":mfe,
        "ExitReason":reason,
    })

trade_diag=pd.DataFrame(diag_rows)
assert not trade_diag.empty

# Segment using only pre-declared, economically meaningful buckets.
trade_diag["ScoreBand"]=pd.cut(
    trade_diag["Score"],bins=[-np.inf,3.5,4.5,np.inf],
    labels=["SCORE3","SCORE4","SCORE5+"],
)
trade_diag["HoldBand"]=pd.cut(
    trade_diag["HoldingDays"],bins=[-1,5,10,15,20,100],
    labels=["0-5","6-10","11-15","16-20","21+"],
)
trade_diag["VolBand"]=pd.cut(
    trade_diag["VolumeRatio"],bins=[-np.inf,1.0,1.2,1.4,2.0,np.inf],
    labels=["<1.0","1.0-1.2","1.2-1.4","1.4-2.0",">2.0"],
)

def _diag_summary(g):
    wins=g[g.ReturnPct>0]["ReturnPct"]
    losses=g[g.ReturnPct<0]["ReturnPct"]
    gross_win=wins.sum()
    gross_loss=abs(losses.sum())
    return pd.Series({
        "Trades":len(g),
        "PnL_pct_sum":g.ReturnPct.sum(),
        "AvgReturnPct":g.ReturnPct.mean(),
        "WinRate":(g.ReturnPct>0).mean()*100,
        "PF":gross_win/gross_loss if gross_loss>0 else np.nan,
        "AvgMAE":g.MAE.mean(),
        "AvgMFE":g.MFE.mean(),
        "AvgHold":g.HoldingDays.mean(),
    })

print("\nBY SCORE")
display(trade_diag.groupby("ScoreBand",observed=False).apply(_diag_summary,include_groups=False))

print("\nBY VOLUME")
display(trade_diag.groupby("VolBand",observed=False).apply(_diag_summary,include_groups=False))

print("\nBY HOLDING TIME")
display(trade_diag.groupby("HoldBand",observed=False).apply(_diag_summary,include_groups=False))

print("\nBY EXIT REASON")
display(trade_diag.groupby("ExitReason",observed=False).apply(_diag_summary,include_groups=False))

# MAE/MFE efficiency: winners should ideally show materially larger MFE
# and manageable MAE. This helps identify whether entries are early enough.
trade_diag["MFE_MAE_Ratio"]=trade_diag["MFE"]/trade_diag["MAE"].abs().replace(0,np.nan)
print("\nOVERALL TRADE DIAGNOSTICS")
display(_diag_summary(trade_diag).to_frame("Value"))
print(f"Median MFE/|MAE|: {trade_diag['MFE_MAE_Ratio'].replace([np.inf,-np.inf],np.nan).median():.3f}")

print("\nTRADE-LEVEL EDGE DECOMPOSITION COMPLETE")


In [ ]:
# ============================================================
# CELL 12.20 — RESEARCH ONLY: STOP GEOMETRY NESTED OOS
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — STOP GEOMETRY NESTED OOS")
print("=" * 90)

assert "signal_cache" in globals()
assert "_true_portfolio_stress" in globals()
assert "SELECTED_SYMBOLS" in globals()

STOP_VARIANTS=[1.0,1.2,1.5,1.8,2.0]

# Build signal sets from the already cached strategy family.
# Cache keys are (min_score, weekly_confirm, atr_mult).
stop_signals={}
for mult in STOP_VARIANTS:
    key=(BEST_STRATEGY["min_score"],BEST_STRATEGY["weekly_confirm"],mult)
    if key in signal_cache:
        x=signal_cache[key].copy()
    else:
        x=signal_cache[
            (BEST_STRATEGY["min_score"],BEST_STRATEGY["weekly_confirm"],BEST_STRATEGY["atr_mult"])
        ].copy()
        # Rebuild only the stop/sizing fields from signal-date ATR/recent-low.
        rows=[]
        for _,sig in x.iterrows():
            sym=sig["Symbol"]; dt=pd.Timestamp(sig["SignalDate"])
            df=canonical_data[sym]
            if dt not in df.index: continue
            bar=df.loc[dt]
            entry=float(sig["EntryPrice"])
            atr=float(bar["ATR14"])
            recent=float(bar["RECENT_LOW_5"])
            stop=min(entry-atr*mult,recent)
            if stop<=0 or stop>=entry: continue
            risk=entry-stop
            shares=int((INITIAL_CAPITAL*0.02/risk)//LOT_SIZE*LOT_SIZE)
            if shares<=0: continue
            z=sig.copy()
            z["StopPrice"]=stop
            z["Shares"]=shares
            z["RiskAmount"]=risk*shares
            rows.append(z)
        x=pd.DataFrame(rows)
    stop_signals[mult]=x

print("\nSIGNAL COUNTS")
for mult,x in stop_signals.items():
    print(f"ATR stop {mult:.1f}x : {len(x):>5}")

FRICTION_STOP={"commission_pct":0.0015,"slippage_pct":0.0010,"gap_extra_pct":0.0025}

rows=[]
for fold_no,(a,b,c,d) in enumerate(nested_folds,1):
    valid_start=portfolio_dates[int(n_dates*b)]
    valid_end=portfolio_dates[int(n_dates*c)-1]
    test_start_n=portfolio_dates[int(n_dates*c)]
    test_end_n=portfolio_dates[int(n_dates*d)-1]

    vals=[]
    for mult,x in stop_signals.items():
        vr=_true_portfolio_stress(
            x,valid_start,valid_end,
            risk_pct=0.020,target2_pct=0.15,max_hold_days=20,max_positions=5,
            **FRICTION_STOP,
        )
        vals.append((mult,vr))

    eligible=[z for z in vals if z[1]["Trades"]>=10]
    chosen_mult,chosen_val=max(eligible,key=lambda z:z[1]["PnL"])

    te=_true_portfolio_stress(
        stop_signals[chosen_mult],test_start_n,test_end_n,
        risk_pct=0.020,target2_pct=0.15,max_hold_days=20,max_positions=5,
        **FRICTION_STOP,
    )
    base_te=_true_portfolio_stress(
        stop_signals[1.2],test_start_n,test_end_n,
        risk_pct=0.020,target2_pct=0.15,max_hold_days=20,max_positions=5,
        **FRICTION_STOP,
    )

    rows.append({
        "Fold":fold_no,
        "SelectedATR":chosen_mult,
        "ValidPnL":chosen_val["PnL"],
        "ValidPF":chosen_val["PF"],
        "TestPnL":te["PnL"],
        "TestTrades":te["Trades"],
        "TestWinRate":te["WinRate"],
        "TestMaxDD":te["MaxDD"],
        "TestPF":te["PF"],
        "Frozen1.2PnL":base_te["PnL"],
        "DeltaVs1.2":te["PnL"]-base_te["PnL"],
    })

stop_nested_df=pd.DataFrame(rows)
display(stop_nested_df)

print("\nSTOP GEOMETRY NESTED SUMMARY")
print(f"Positive OOS folds: {int((stop_nested_df.TestPnL>0).sum())}/{len(stop_nested_df)}")
print(f"Selected OOS P&L: {stop_nested_df.TestPnL.sum():,.2f} THB")
print(f"Frozen ATR1.2 OOS P&L: {stop_nested_df['Frozen1.2PnL'].sum():,.2f} THB")
print(f"Delta vs ATR1.2: {stop_nested_df['DeltaVs1.2'].sum():,.2f} THB")
print(f"Selected stops: {', '.join(str(x) for x in stop_nested_df.SelectedATR.tolist())}")

if (
    int((stop_nested_df.TestPnL>0).sum())>=2
    and stop_nested_df['DeltaVs1.2'].sum()>0
    and (stop_nested_df.TestPF>1.0).all()
):
    stop_decision="STOP_GEOMETRY_CANDIDATE"
else:
    stop_decision="KEEP_CURRENT_STOP"

print("RESEARCH DECISION:",stop_decision)
print("PRODUCTION CONFIGURATION UNCHANGED")
print("STOP GEOMETRY STUDY COMPLETE")


In [ ]:
# ============================================================
# CELL 12.21 — RESEARCH ONLY: ENTRY TIMING / EXTENSION FILTER
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — ENTRY TIMING / EXTENSION / EARLY-FAILURE AVOIDANCE")
print("=" * 90)

assert "exec_signals" in globals()
assert "_true_portfolio_stress" in globals()
assert "nested_folds" in globals()
assert "portfolio_dates" in globals()
assert "n_dates" in globals()

# Freeze the current production signal set. Every feature below is known
# on SignalDate; no next-day information is used for candidate selection.
timing_signals = exec_signals.copy()

def _attach_timing_features(signals):
    out = signals.copy()
    ema5_ext = []
    ema10_ext = []
    atr_ext = []
    close_pos = []
    candle_body = []

    for _, row in out.iterrows():
        sym = row["Symbol"]
        dt = pd.Timestamp(row["SignalDate"])
        df = canonical_data[sym]

        if dt not in df.index:
            ema5_ext.append(np.nan)
            ema10_ext.append(np.nan)
            atr_ext.append(np.nan)
            close_pos.append(np.nan)
            candle_body.append(np.nan)
            continue

        bar = df.loc[dt]
        close = float(bar["Close"])
        high = float(bar["High"])
        low = float(bar["Low"])
        ema5 = float(bar["EMA5"]) if pd.notna(bar["EMA5"]) else np.nan
        ema10 = float(bar["EMA10"]) if pd.notna(bar["EMA10"]) else np.nan
        atr = float(bar["ATR14"]) if pd.notna(bar["ATR14"]) else np.nan

        ema5_ext.append((close / ema5) - 1.0 if np.isfinite(ema5) and ema5 > 0 else np.nan)
        ema10_ext.append((close / ema10) - 1.0 if np.isfinite(ema10) and ema10 > 0 else np.nan)
        atr_ext.append((close - ema5) / atr if np.isfinite(ema5) and np.isfinite(atr) and atr > 0 else np.nan)

        rng = high - low
        close_pos.append((close - low) / rng if rng > 0 else 0.5)
        candle_body.append(abs(close - float(bar["Open"])) / rng if rng > 0 else 0.0)

    out["SignalEMA5Extension"] = ema5_ext
    out["SignalEMA10Extension"] = ema10_ext
    out["SignalEMA5_ATR_Extension"] = atr_ext
    out["SignalClosePosition"] = close_pos
    out["SignalBodyPctRange"] = candle_body
    return out

timing_signals = _attach_timing_features(timing_signals)

required_timing_cols = [
    "SignalEMA5Extension",
    "SignalEMA10Extension",
    "SignalEMA5_ATR_Extension",
    "SignalClosePosition",
]
assert timing_signals[required_timing_cols].notna().all().all()

# Small, pre-declared family. The intent is specifically to reject entries
# that are already extended at SignalDate, while preserving normal momentum.
TIMING_FILTERS = {
    "BASE": lambda d: d["SignalScore"] >= 4,
    "NO_EXT_3PCT": lambda d: (d["SignalScore"] >= 4) & (d["SignalEMA5Extension"] <= 0.03),
    "NO_EXT_5PCT": lambda d: (d["SignalScore"] >= 4) & (d["SignalEMA5Extension"] <= 0.05),
    "NO_EXT_ATR1": lambda d: (d["SignalScore"] >= 4) & (d["SignalEMA5_ATR_Extension"] <= 1.0),
    "CLOSE_POS60": lambda d: (d["SignalScore"] >= 4) & (d["SignalClosePosition"] >= 0.60),
    "NO_EXT3_POS60": lambda d: (
        (d["SignalScore"] >= 4)
        & (d["SignalEMA5Extension"] <= 0.03)
        & (d["SignalClosePosition"] >= 0.60)
    ),
    "NO_EXT5_POS60": lambda d: (
        (d["SignalScore"] >= 4)
        & (d["SignalEMA5Extension"] <= 0.05)
        & (d["SignalClosePosition"] >= 0.60)
    ),
}

TIMING_SIGNALS = {
    name: timing_signals.loc[mask(timing_signals)].copy()
    for name, mask in TIMING_FILTERS.items()
}

print("\nFILTER COUNTS")
for name, df in TIMING_SIGNALS.items():
    print(f"{name:<18} signals={len(df):>5}")

FRICTION_TIMING = {
    "commission_pct": 0.0015,
    "slippage_pct": 0.0010,
    "gap_extra_pct": 0.0025,
}

nested_rows = []

for fold_no, (a, b, c, d) in enumerate(nested_folds, 1):
    valid_start = portfolio_dates[int(n_dates * b)]
    valid_end = portfolio_dates[int(n_dates * c) - 1]
    test_start = portfolio_dates[int(n_dates * c)]
    test_end = portfolio_dates[int(n_dates * d) - 1]

    validation = []
    for name, sigs in TIMING_SIGNALS.items():
        val = _true_portfolio_stress(
            sigs,
            valid_start,
            valid_end,
            risk_pct=0.020,
            target2_pct=0.15,
            max_hold_days=20,
            max_positions=5,
            **FRICTION_TIMING,
        )
        validation.append((name, val))

    eligible = [x for x in validation if x[1]["Trades"] >= 10]
    if not eligible:
        eligible = validation

    chosen_name, chosen_val = max(
        eligible,
        key=lambda x: x[1]["PnL"],
    )

    test = _true_portfolio_stress(
        TIMING_SIGNALS[chosen_name],
        test_start,
        test_end,
        risk_pct=0.020,
        target2_pct=0.15,
        max_hold_days=20,
        max_positions=5,
        **FRICTION_TIMING,
    )

    base_test = _true_portfolio_stress(
        TIMING_SIGNALS["BASE"],
        test_start,
        test_end,
        risk_pct=0.020,
        target2_pct=0.15,
        max_hold_days=20,
        max_positions=5,
        **FRICTION_TIMING,
    )

    nested_rows.append({
        "Fold": fold_no,
        "Selected": chosen_name,
        "ValidPnL": chosen_val["PnL"],
        "ValidPF": chosen_val["PF"],
        "TestPnL": test["PnL"],
        "TestTrades": test["Trades"],
        "TestWinRate": test["WinRate"],
        "TestMaxDD": test["MaxDD"],
        "TestPF": test["PF"],
        "BaseTestPnL": base_test["PnL"],
        "DeltaVsBase": test["PnL"] - base_test["PnL"],
    })

timing_nested_df = pd.DataFrame(nested_rows)
display(timing_nested_df)

print("\nENTRY TIMING NESTED SUMMARY")
print(f"Positive OOS folds: {int((timing_nested_df.TestPnL > 0).sum())}/{len(timing_nested_df)}")
print(f"Selected net OOS P&L: {timing_nested_df.TestPnL.sum():,.2f} THB")
print(f"Frozen BASE net OOS P&L: {timing_nested_df.BaseTestPnL.sum():,.2f} THB")
print(f"Delta vs BASE: {timing_nested_df.DeltaVsBase.sum():,.2f} THB")
print(f"Selected filters: {', '.join(timing_nested_df.Selected.tolist())}")

# Promotion is intentionally strict:
# 1) >=2/3 positive OOS folds
# 2) beats frozen BASE under realistic friction
# 3) PF > 1 in every unseen fold
# Otherwise production remains unchanged.
if (
    int((timing_nested_df.TestPnL > 0).sum()) >= 2
    and timing_nested_df.DeltaVsBase.sum() > 0
    and (timing_nested_df.TestPF > 1.0).all()
):
    timing_decision = "ENTRY_TIMING_CANDIDATE"
else:
    timing_decision = "KEEP_CURRENT_ENTRY_RULES"

print("RESEARCH DECISION:", timing_decision)
print("PRODUCTION CONFIGURATION UNCHANGED")
print("ENTRY TIMING / EXTENSION STUDY COMPLETE")


In [ ]:
# ============================================================
# CELL 12.22 — RESEARCH ONLY: ENTRY GAP / OPENING EXECUTION FILTER
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — ENTRY GAP / OPENING EXECUTION FILTER")
print("=" * 90)

assert "exec_signals" in globals()
assert "_true_portfolio_stress" in globals()
assert "nested_folds" in globals()
assert "portfolio_dates" in globals()
assert "n_dates" in globals()

# The next-day OPEN is the actual planned entry price. Using it here is
# intentional: this is an execution-time cancellation test, not a
# signal-date look-ahead. No data after EntryDate is used.
gap_signals = exec_signals.copy()

gap_pct = []
gap_atr = []
signal_close = []
signal_atr = []

for _, row in gap_signals.iterrows():
    sym = row["Symbol"]
    dt = pd.Timestamp(row["SignalDate"])
    df = canonical_data[sym]

    if dt not in df.index:
        gap_pct.append(np.nan)
        gap_atr.append(np.nan)
        signal_close.append(np.nan)
        signal_atr.append(np.nan)
        continue

    bar = df.loc[dt]
    close = float(bar["Close"])
    atr = float(bar["ATR14"]) if pd.notna(bar["ATR14"]) else np.nan
    entry = float(row["EntryPrice"])

    signal_close.append(close)
    signal_atr.append(atr)
    gap_pct.append(entry / close - 1.0 if close > 0 else np.nan)
    gap_atr.append((entry - close) / atr if np.isfinite(atr) and atr > 0 else np.nan)

gap_signals["SignalClose"] = signal_close
gap_signals["SignalATR"] = signal_atr
gap_signals["EntryGapPct"] = gap_pct
gap_signals["EntryGapATR"] = gap_atr

assert gap_signals[["EntryGapPct","EntryGapATR"]].notna().all().all()

# Compact, pre-declared family. Negative gaps are allowed; only excessive
# opening premiums are rejected.
GAP_FILTERS = {
    "BASE": lambda d: d["SignalScore"] >= 4,
    "GAP_LE_1PCT": lambda d: (d["SignalScore"] >= 4) & (d["EntryGapPct"] <= 0.01),
    "GAP_LE_2PCT": lambda d: (d["SignalScore"] >= 4) & (d["EntryGapPct"] <= 0.02),
    "GAP_LE_3PCT": lambda d: (d["SignalScore"] >= 4) & (d["EntryGapPct"] <= 0.03),
    "GAP_ATR_LE_0P5": lambda d: (d["SignalScore"] >= 4) & (d["EntryGapATR"] <= 0.50),
    "GAP_ATR_LE_1P0": lambda d: (d["SignalScore"] >= 4) & (d["EntryGapATR"] <= 1.00),
    "GAP2_AND_ATR1": lambda d: (
        (d["SignalScore"] >= 4)
        & (d["EntryGapPct"] <= 0.02)
        & (d["EntryGapATR"] <= 1.00)
    ),
}

GAP_SIGNALS = {
    name: gap_signals.loc[mask(gap_signals)].copy()
    for name, mask in GAP_FILTERS.items()
}

print("\nFILTER COUNTS")
for name, df in GAP_SIGNALS.items():
    print(f"{name:<20} signals={len(df):>5}")

FRICTION_GAP = {
    "commission_pct": 0.0015,
    "slippage_pct": 0.0010,
    "gap_extra_pct": 0.0025,
}

nested_rows = []

for fold_no, (a, b, c, d) in enumerate(nested_folds, 1):
    valid_start = portfolio_dates[int(n_dates * b)]
    valid_end = portfolio_dates[int(n_dates * c) - 1]
    test_start = portfolio_dates[int(n_dates * c)]
    test_end = portfolio_dates[int(n_dates * d) - 1]

    validation = []
    for name, sigs in GAP_SIGNALS.items():
        val = _true_portfolio_stress(
            sigs,
            valid_start,
            valid_end,
            risk_pct=0.020,
            target2_pct=0.15,
            max_hold_days=20,
            max_positions=5,
            **FRICTION_GAP,
        )
        validation.append((name, val))

    eligible = [x for x in validation if x[1]["Trades"] >= 10]
    if not eligible:
        eligible = validation

    chosen_name, chosen_val = max(
        eligible,
        key=lambda x: x[1]["PnL"],
    )

    test = _true_portfolio_stress(
        GAP_SIGNALS[chosen_name],
        test_start,
        test_end,
        risk_pct=0.020,
        target2_pct=0.15,
        max_hold_days=20,
        max_positions=5,
        **FRICTION_GAP,
    )

    base_test = _true_portfolio_stress(
        GAP_SIGNALS["BASE"],
        test_start,
        test_end,
        risk_pct=0.020,
        target2_pct=0.15,
        max_hold_days=20,
        max_positions=5,
        **FRICTION_GAP,
    )

    nested_rows.append({
        "Fold": fold_no,
        "Selected": chosen_name,
        "ValidPnL": chosen_val["PnL"],
        "ValidPF": chosen_val["PF"],
        "TestPnL": test["PnL"],
        "TestTrades": test["Trades"],
        "TestWinRate": test["WinRate"],
        "TestMaxDD": test["MaxDD"],
        "TestPF": test["PF"],
        "BaseTestPnL": base_test["PnL"],
        "DeltaVsBase": test["PnL"] - base_test["PnL"],
    })

gap_nested_df = pd.DataFrame(nested_rows)
display(gap_nested_df)

print("\nENTRY GAP NESTED SUMMARY")
print(f"Positive OOS folds: {int((gap_nested_df.TestPnL > 0).sum())}/{len(gap_nested_df)}")
print(f"Selected net OOS P&L: {gap_nested_df.TestPnL.sum():,.2f} THB")
print(f"Frozen BASE net OOS P&L: {gap_nested_df.BaseTestPnL.sum():,.2f} THB")
print(f"Delta vs BASE: {gap_nested_df.DeltaVsBase.sum():,.2f} THB")
print(f"Selected filters: {', '.join(gap_nested_df.Selected.tolist())}")

if (
    int((gap_nested_df.TestPnL > 0).sum()) >= 2
    and gap_nested_df.DeltaVsBase.sum() > 0
    and (gap_nested_df.TestPF > 1.0).all()
):
    gap_decision = "ENTRY_GAP_CANDIDATE"
else:
    gap_decision = "KEEP_CURRENT_ENTRY_RULES"

print("RESEARCH DECISION:", gap_decision)
print("PRODUCTION CONFIGURATION UNCHANGED")
print("ENTRY GAP / OPENING EXECUTION STUDY COMPLETE")


In [ ]:
# ============================================================
# CELL 12.23 — RESEARCH ONLY: ONE-DAY ENTRY CONFIRMATION
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — ONE-DAY ENTRY CONFIRMATION / DELAYED ENTRY")
print("=" * 90)

assert "exec_signals" in globals()
assert "_true_portfolio_stress" in globals()
assert "nested_folds" in globals()
assert "portfolio_dates" in globals()
assert "n_dates" in globals()

# This study changes only the entry timing. Stop remains the frozen
# signal-date stop. Confirmation uses the first trading day after SignalDate,
# then enters on the following trading day OPEN. This is fully causal.
def _next_trading_date(df, dt):
    idx = pd.DatetimeIndex(df.index)
    pos = idx.searchsorted(pd.Timestamp(dt), side="right")
    return pd.Timestamp(idx[pos]) if pos < len(idx) else None

def _build_delayed_entry_signals(base, mode):
    rows = []

    for _, sig in base.iterrows():
        sym = sig["Symbol"]
        sdt = pd.Timestamp(sig["SignalDate"])
        df = canonical_data[sym]

        if sdt not in df.index:
            continue

        confirm_dt = _next_trading_date(df, sdt)
        if confirm_dt is None or confirm_dt not in df.index:
            continue

        confirm_bar = df.loc[confirm_dt]
        signal_close = float(df.loc[sdt, "Close"])
        signal_atr = float(df.loc[sdt, "ATR14"]) if pd.notna(df.loc[sdt, "ATR14"]) else np.nan
        confirm_close = float(confirm_bar["Close"])

        if mode == "WAIT1":
            pass
        elif mode == "CONFIRM_CLOSE":
            if not (confirm_close > signal_close):
                continue
        elif mode == "CONFIRM_CLOSE_ATR025":
            if not (np.isfinite(signal_atr) and signal_atr > 0):
                continue
            if not (confirm_close >= signal_close + 0.25 * signal_atr):
                continue
        else:
            raise ValueError(mode)

        entry_dt = _next_trading_date(df, confirm_dt)
        if entry_dt is None or entry_dt not in df.index:
            continue

        z = sig.copy()
        z["EntryDate"] = entry_dt
        z["EntryPrice"] = float(df.loc[entry_dt, "Open"])
        rows.append(z)

    return pd.DataFrame(rows)

DELAYED_SIGNALS = {
    "BASE": exec_signals.copy(),
    "CONFIRM_CLOSE": _build_delayed_entry_signals(exec_signals, "CONFIRM_CLOSE"),
    "CONFIRM_CLOSE_ATR025": _build_delayed_entry_signals(exec_signals, "CONFIRM_CLOSE_ATR025"),
}

print("\nSIGNAL COUNTS")
for name, df in DELAYED_SIGNALS.items():
    print(f"{name:<24} signals={len(df):>5}")

FRICTION_DELAY = {
    "commission_pct": 0.0015,
    "slippage_pct": 0.0010,
    "gap_extra_pct": 0.0025,
}

nested_rows = []

for fold_no, (a, b, c, d) in enumerate(nested_folds, 1):
    valid_start = portfolio_dates[int(n_dates * b)]
    valid_end = portfolio_dates[int(n_dates * c) - 1]
    test_start = portfolio_dates[int(n_dates * c)]
    test_end = portfolio_dates[int(n_dates * d) - 1]

    validation = []
    for name, sigs in DELAYED_SIGNALS.items():
        val = _true_portfolio_stress(
            sigs,
            valid_start,
            valid_end,
            risk_pct=0.020,
            target2_pct=0.15,
            max_hold_days=20,
            max_positions=5,
            **FRICTION_DELAY,
        )
        validation.append((name, val))

    eligible = [x for x in validation if x[1]["Trades"] >= 10]
    if not eligible:
        eligible = validation

    chosen_name, chosen_val = max(
        eligible,
        key=lambda x: x[1]["PnL"],
    )

    test = _true_portfolio_stress(
        DELAYED_SIGNALS[chosen_name],
        test_start,
        test_end,
        risk_pct=0.020,
        target2_pct=0.15,
        max_hold_days=20,
        max_positions=5,
        **FRICTION_DELAY,
    )

    base_test = _true_portfolio_stress(
        DELAYED_SIGNALS["BASE"],
        test_start,
        test_end,
        risk_pct=0.020,
        target2_pct=0.15,
        max_hold_days=20,
        max_positions=5,
        **FRICTION_DELAY,
    )

    nested_rows.append({
        "Fold": fold_no,
        "Selected": chosen_name,
        "ValidPnL": chosen_val["PnL"],
        "ValidPF": chosen_val["PF"],
        "TestPnL": test["PnL"],
        "TestTrades": test["Trades"],
        "TestWinRate": test["WinRate"],
        "TestMaxDD": test["MaxDD"],
        "TestPF": test["PF"],
        "BaseTestPnL": base_test["PnL"],
        "DeltaVsBase": test["PnL"] - base_test["PnL"],
    })

delay_nested_df = pd.DataFrame(nested_rows)
display(delay_nested_df)

print("\nONE-DAY CONFIRMATION NESTED SUMMARY")
print(f"Positive OOS folds: {int((delay_nested_df.TestPnL > 0).sum())}/{len(delay_nested_df)}")
print(f"Selected net OOS P&L: {delay_nested_df.TestPnL.sum():,.2f} THB")
print(f"Frozen BASE net OOS P&L: {delay_nested_df.BaseTestPnL.sum():,.2f} THB")
print(f"Delta vs BASE: {delay_nested_df.DeltaVsBase.sum():,.2f} THB")
print(f"Selected entry modes: {', '.join(delay_nested_df.Selected.tolist())}")

if (
    int((delay_nested_df.TestPnL > 0).sum()) >= 2
    and delay_nested_df.DeltaVsBase.sum() > 0
    and (delay_nested_df.TestPF > 1.0).all()
):
    delay_decision = "ENTRY_CONFIRMATION_CANDIDATE"
else:
    delay_decision = "KEEP_CURRENT_ENTRY_RULES"

print("RESEARCH DECISION:", delay_decision)
print("PRODUCTION CONFIGURATION UNCHANGED")
print("ONE-DAY ENTRY CONFIRMATION STUDY COMPLETE")


In [ ]:
# ============================================================
# CELL 12.24 — RESEARCH ONLY: SIGNAL-DAY EARLY-FAILURE QUALITY SCORE
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — SIGNAL-DAY EARLY-FAILURE QUALITY SCORE")
print("=" * 90)

assert "exec_signals" in globals()
assert "_true_portfolio_stress" in globals()
assert "nested_folds" in globals()
assert "portfolio_dates" in globals()
assert "n_dates" in globals()

# Causal diagnostic: all features are known at SignalDate.
# The goal is NOT a broad parameter sweep. We test a compact quality
# score built from four independent dimensions of signal strength:
# 1) MACD histogram accelerating
# 2) stochastic K rising
# 3) positive 3-day price acceleration
# 4) EMA5 above EMA10 by at least 1%
quality = exec_signals.copy()
q_rows = []

for _, row in quality.iterrows():
    sym = row["Symbol"]
    dt = pd.Timestamp(row["SignalDate"])
    df = canonical_data[sym]

    if dt not in df.index:
        q_rows.append((np.nan,)*5)
        continue

    pos = df.index.get_loc(dt)
    if isinstance(pos, slice) or isinstance(pos, np.ndarray) or pos < 3:
        q_rows.append((np.nan,)*5)
        continue

    hist = float(df.iloc[pos]["MACD_HIST"])
    hist_prev = float(df.iloc[pos-1]["MACD_HIST"])
    k = float(df.iloc[pos]["STOCH_K"])
    k_prev = float(df.iloc[pos-1]["STOCH_K"])
    close = float(df.iloc[pos]["Close"])
    close_3 = float(df.iloc[pos-3]["Close"])
    ema5 = float(df.iloc[pos]["EMA5"])
    ema10 = float(df.iloc[pos]["EMA10"])

    q1 = np.isfinite(hist) and np.isfinite(hist_prev) and hist > hist_prev
    q2 = np.isfinite(k) and np.isfinite(k_prev) and k > k_prev
    q3 = np.isfinite(close) and np.isfinite(close_3) and close > close_3
    q4 = np.isfinite(ema5) and np.isfinite(ema10) and ema5 > ema10 * 1.01

    q_rows.append((int(q1), int(q2), int(q3), int(q4), int(q1)+int(q2)+int(q3)+int(q4)))

quality[[
    "Q_MACD_ACCEL",
    "Q_STOCH_RISE",
    "Q_RET3_POS",
    "Q_EMA5_GT_EMA10_1PCT",
    "SignalQualityScore",
]] = pd.DataFrame(
    q_rows, index=quality.index,
    columns=[
        "Q_MACD_ACCEL",
        "Q_STOCH_RISE",
        "Q_RET3_POS",
        "Q_EMA5_GT_EMA10_1PCT",
        "SignalQualityScore",
    ],
)

quality = quality.dropna(subset=["SignalQualityScore"]).copy()
quality["SignalQualityScore"] = quality["SignalQualityScore"].astype(int)

QUALITY_FILTERS = {
    "BASE": quality["SignalScore"] >= 4,
    "QUALITY2": (quality["SignalScore"] >= 4) & (quality["SignalQualityScore"] >= 2),
    "QUALITY3": (quality["SignalScore"] >= 4) & (quality["SignalQualityScore"] >= 3),
    "QUALITY4": (quality["SignalScore"] >= 4) & (quality["SignalQualityScore"] >= 4),
    "MACD_ACCEL": (quality["SignalScore"] >= 4) & (quality["Q_MACD_ACCEL"] == 1),
    "STOCH_RISE": (quality["SignalScore"] >= 4) & (quality["Q_STOCH_RISE"] == 1),
}

QUALITY_SIGNALS = {
    name: quality.loc[mask].copy()
    for name, mask in QUALITY_FILTERS.items()
}

print("\nFILTER COUNTS")
for name, df in QUALITY_SIGNALS.items():
    print(f"{name:<18} signals={len(df):>5}")

FRICTION_QUALITY = {
    "commission_pct": 0.0015,
    "slippage_pct": 0.0010,
    "gap_extra_pct": 0.0025,
}

rows = []

for fold_no, (a, b, c, d) in enumerate(nested_folds, 1):
    valid_start = portfolio_dates[int(n_dates * b)]
    valid_end = portfolio_dates[int(n_dates * c) - 1]
    test_start = portfolio_dates[int(n_dates * c)]
    test_end = portfolio_dates[int(n_dates * d) - 1]

    validation = []
    for name, sigs in QUALITY_SIGNALS.items():
        val = _true_portfolio_stress(
            sigs, valid_start, valid_end,
            risk_pct=0.020, target2_pct=0.15,
            max_hold_days=20, max_positions=5,
            **FRICTION_QUALITY,
        )
        validation.append((name, val))

    eligible = [x for x in validation if x[1]["Trades"] >= 10]
    if not eligible:
        eligible = validation

    chosen_name, chosen_val = max(
        eligible, key=lambda x: x[1]["PnL"]
    )

    test = _true_portfolio_stress(
        QUALITY_SIGNALS[chosen_name], test_start, test_end,
        risk_pct=0.020, target2_pct=0.15,
        max_hold_days=20, max_positions=5,
        **FRICTION_QUALITY,
    )

    base_test = _true_portfolio_stress(
        QUALITY_SIGNALS["BASE"], test_start, test_end,
        risk_pct=0.020, target2_pct=0.15,
        max_hold_days=20, max_positions=5,
        **FRICTION_QUALITY,
    )

    rows.append({
        "Fold": fold_no,
        "Selected": chosen_name,
        "ValidPnL": chosen_val["PnL"],
        "ValidPF": chosen_val["PF"],
        "TestPnL": test["PnL"],
        "TestTrades": test["Trades"],
        "TestWinRate": test["WinRate"],
        "TestMaxDD": test["MaxDD"],
        "TestPF": test["PF"],
        "BaseTestPnL": base_test["PnL"],
        "DeltaVsBase": test["PnL"] - base_test["PnL"],
    })

quality_nested_df = pd.DataFrame(rows)
display(quality_nested_df)

print("\nSIGNAL-DAY QUALITY NESTED SUMMARY")
print(f"Positive OOS folds: {int((quality_nested_df.TestPnL > 0).sum())}/{len(quality_nested_df)}")
print(f"Selected net OOS P&L: {quality_nested_df.TestPnL.sum():,.2f} THB")
print(f"Frozen BASE net OOS P&L: {quality_nested_df.BaseTestPnL.sum():,.2f} THB")
print(f"Delta vs BASE: {quality_nested_df.DeltaVsBase.sum():,.2f} THB")
print(f"Selected filters: {', '.join(quality_nested_df.Selected.tolist())}")

if (
    int((quality_nested_df.TestPnL > 0).sum()) >= 2
    and quality_nested_df.DeltaVsBase.sum() > 0
    and (quality_nested_df.TestPF > 1.0).all()
):
    quality_decision = "SIGNAL_QUALITY_CANDIDATE"
else:
    quality_decision = "KEEP_CURRENT_ENTRY_RULES"

print("RESEARCH DECISION:", quality_decision)
print("PRODUCTION CONFIGURATION UNCHANGED")
print("SIGNAL-DAY EARLY-FAILURE QUALITY STUDY COMPLETE")


In [ ]:
# ============================================================
# CELL 12.25 — RESEARCH ONLY: CAUSAL EARLY-FAILURE RISK MODEL
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — CAUSAL EARLY-FAILURE RISK MODEL")
print("=" * 90)

assert "exec_signals" in globals()
assert "_true_portfolio_stress" in globals()
assert "nested_folds" in globals()
assert "portfolio_dates" in globals()
assert "n_dates" in globals()

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

MODEL_FEATURES = [
    "CloseEMA5Pct", "CloseEMA10Pct", "EMA5ATR_Ext",
    "ClosePosition", "BodyPctRange", "MACDHistSlope",
    "StochKSlope", "VolumeRatio",
]

def _early_fail_label(sig):
    sym = sig["Symbol"]
    ed = pd.Timestamp(sig["EntryDate"])
    df = canonical_data[sym]
    pos = df.index.searchsorted(ed, side="left")
    if pos >= len(df.index):
        return np.nan, pd.NaT

    future = df.iloc[pos:pos + 5]
    if future.empty or len(future) < 5:
        return np.nan, pd.NaT

    entry = float(sig["EntryPrice"])
    stop = float(sig["StopPrice"])
    target = entry * 1.15

    for _, bar in future.iterrows():
        op, hi, lo = map(float, [bar["Open"], bar["High"], bar["Low"]])
        if op <= stop:
            return 1, pd.Timestamp(bar.name)
        if op >= target:
            return 0, pd.Timestamp(bar.name)
        if lo <= stop:
            return 1, pd.Timestamp(bar.name)
        if hi >= target:
            return 0, pd.Timestamp(bar.name)

    return 0, pd.Timestamp(future.index[-1])

model_df = exec_signals.copy()
feat_rows = []
labels = []
label_end_dates = []

for _, row in model_df.iterrows():
    sym = row["Symbol"]
    dt = pd.Timestamp(row["SignalDate"])
    df = canonical_data[sym]

    if dt not in df.index:
        feat_rows.append([np.nan] * len(MODEL_FEATURES))
        labels.append(np.nan)
        label_end_dates.append(pd.NaT)
        continue

    pos = df.index.get_loc(dt)
    if isinstance(pos, slice) or isinstance(pos, np.ndarray) or pos < 1:
        feat_rows.append([np.nan] * len(MODEL_FEATURES))
        labels.append(np.nan)
        label_end_dates.append(pd.NaT)
        continue

    b = df.iloc[pos]
    p = df.iloc[pos - 1]
    close = float(b["Close"])
    ema5 = float(b["EMA5"])
    ema10 = float(b["EMA10"])
    atr = float(b["ATR14"]) if pd.notna(b["ATR14"]) else np.nan
    high = float(b["High"])
    low = float(b["Low"])
    op = float(b["Open"])
    rng = max(high - low, 1e-12)

    feat_rows.append([
        close / ema5 - 1 if ema5 else np.nan,
        close / ema10 - 1 if ema10 else np.nan,
        (close - ema5) / atr if pd.notna(atr) and atr else np.nan,
        (close - low) / rng,
        abs(close - op) / rng,
        float(b["MACD_HIST"]) - float(p["MACD_HIST"]),
        float(b["STOCH_K"]) - float(p["STOCH_K"]),
        float(b["VOLUME_RATIO"]) if pd.notna(b["VOLUME_RATIO"]) else np.nan,
    ])

    label, label_end = _early_fail_label(row)
    labels.append(label)
    label_end_dates.append(label_end)

model_df[MODEL_FEATURES] = pd.DataFrame(
    feat_rows, index=model_df.index, columns=MODEL_FEATURES
)
model_df["EarlyFail5"] = labels
model_df["LabelEndDate"] = pd.to_datetime(label_end_dates)
model_df = model_df.dropna(subset=["EarlyFail5", "LabelEndDate"]).copy()
model_df["EarlyFail5"] = model_df["EarlyFail5"].astype(int)

print("\nMODEL LABEL")
print(model_df["EarlyFail5"].value_counts().sort_index().to_string())
print(f"Usable labeled signals: {len(model_df)}")

stress = {
    "commission_pct": 0.0015,
    "slippage_pct": 0.0010,
    "gap_extra_pct": 0.0025,
}
thresholds = [0.35, 0.45, 0.55]
selected_rows = []

# nested_folds contains four fractional boundaries:
# train_start, train_end, validation_end, test_end.
# Build each split by SignalDate rather than positional indices so that
# dropping unlabeled rows cannot misalign the fold boundaries.
for fold_no, (a, b, c, d) in enumerate(nested_folds, 1):
    train_start = pd.Timestamp(portfolio_dates[int(n_dates * a)])
    train_end = pd.Timestamp(portfolio_dates[int(n_dates * b) - 1])
    valid_start = pd.Timestamp(portfolio_dates[int(n_dates * b)])
    valid_end = pd.Timestamp(portfolio_dates[int(n_dates * c) - 1])
    test_start = pd.Timestamp(portfolio_dates[int(n_dates * c)])
    test_end = pd.Timestamp(portfolio_dates[int(n_dates * d) - 1])

    # Purge training observations whose 5-bar label horizon reaches into
    # validation. This prevents outcome leakage across the fold boundary.
    tr = model_df.loc[
        (model_df["SignalDate"] >= train_start)
        & (model_df["SignalDate"] <= train_end)
        & (model_df["LabelEndDate"] < valid_start)
    ].copy()

    va = model_df.loc[
        (model_df["SignalDate"] >= valid_start)
        & (model_df["SignalDate"] <= valid_end)
    ].copy()

    te = model_df.loc[
        (model_df["SignalDate"] >= test_start)
        & (model_df["SignalDate"] <= test_end)
    ].copy()

    if len(tr) < 20 or tr["EarlyFail5"].nunique() < 2:
        raise RuntimeError(
            f"Fold {fold_no}: insufficient training data for causal model "
            f"(rows={len(tr)}, classes={tr['EarlyFail5'].nunique()})"
        )
    if va.empty or te.empty:
        raise RuntimeError(
            f"Fold {fold_no}: empty validation/test set "
            f"(validation={len(va)}, test={len(te)})"
        )

    model = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("clf", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ])
    model.fit(tr[MODEL_FEATURES], tr["EarlyFail5"])

    va = va.copy()
    va["FailProb"] = model.predict_proba(va[MODEL_FEATURES])[:, 1]

    base_val = _true_portfolio_stress(
        va, valid_start, valid_end,
        risk_pct=0.020, target2_pct=0.15,
        max_hold_days=20, max_positions=5,
        **stress,
    )

    best_name = "BASE"
    best_pnl = float(base_val["PnL"])

    for th in thresholds:
        cand = va.loc[va["FailProb"] <= th].copy()
        if cand.empty:
            continue
        rr = _true_portfolio_stress(
            cand, valid_start, valid_end,
            risk_pct=0.020, target2_pct=0.15,
            max_hold_days=20, max_positions=5,
            **stress,
        )
        if float(rr["PnL"]) > best_pnl:
            best_name = f"LOWFAIL_{int(th * 100)}"
            best_pnl = float(rr["PnL"])

    # Refit only with observations whose labels are fully known before the
    # unseen test period. Validation is included for model fitting after
    # model selection, but its label horizon is purged at test_start.
    fit = model_df.loc[
        (model_df["SignalDate"] >= train_start)
        & (model_df["SignalDate"] <= valid_end)
        & (model_df["LabelEndDate"] < test_start)
    ].copy()

    if len(fit) < 20 or fit["EarlyFail5"].nunique() < 2:
        raise RuntimeError(
            f"Fold {fold_no}: insufficient refit data "
            f"(rows={len(fit)}, classes={fit['EarlyFail5'].nunique()})"
        )

    final_model = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("clf", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ])
    final_model.fit(fit[MODEL_FEATURES], fit["EarlyFail5"])

    te = te.copy()
    te["FailProb"] = final_model.predict_proba(te[MODEL_FEATURES])[:, 1]

    if best_name == "BASE":
        test_signals = te
    else:
        cutoff = float(best_name.split("_")[-1]) / 100.0
        test_signals = te.loc[te["FailProb"] <= cutoff].copy()

    sel = _true_portfolio_stress(
        test_signals, test_start, test_end,
        risk_pct=0.020, target2_pct=0.15,
        max_hold_days=20, max_positions=5,
        **stress,
    )
    bas = _true_portfolio_stress(
        te, test_start, test_end,
        risk_pct=0.020, target2_pct=0.15,
        max_hold_days=20, max_positions=5,
        **stress,
    )

    selected_rows.append({
        "Fold": fold_no,
        "Selected": best_name,
        "TrainRows": len(tr),
        "ValidRows": len(va),
        "TestRows": len(te),
        "TestPnL": sel["PnL"],
        "BasePnL": bas["PnL"],
        "Delta": sel["PnL"] - bas["PnL"],
        "PF": sel["PF"],
        "Trades": sel["Trades"],
    })

    print(
        f"Fold {fold_no}: selected {best_name}, "
        f"train={len(tr)}, valid={len(va)}, test={len(te)}, "
        f"test {sel['PnL']:.2f}, base {bas['PnL']:.2f}, "
        f"delta {sel['PnL'] - bas['PnL']:.2f}, PF {sel['PF']:.3f}"
    )

sel_df = pd.DataFrame(selected_rows)
positive = int((sel_df["TestPnL"] > 0).sum())
total = float(sel_df["TestPnL"].sum())
base_total = float(sel_df["BasePnL"].sum())
robust = bool((sel_df["PF"].fillna(0) > 1).all())

decision = (
    "PROMOTE_RESEARCH_CANDIDATE"
    if positive >= 2 and total > base_total and robust
    else "KEEP_CURRENT_ENTRY_RULES"
)

print("\nEARLY-FAILURE MODEL SUMMARY")
print(sel_df.to_string(index=False))
print(f"Positive OOS folds: {positive}/3")
print(f"Selected net OOS P&L: {total:.2f} THB")
print(f"Frozen BASE net OOS P&L: {base_total:.2f} THB")
print(f"Delta vs BASE: {total - base_total:.2f} THB")
print(f"RESEARCH DECISION: {decision}")
print("PRODUCTION CONFIGURATION UNCHANGED")
print("CAUSAL EARLY-FAILURE RISK MODEL STUDY COMPLETE")


In [ ]:
# ============================================================
# CELL 12.26 — RESEARCH ONLY: SETUP PERSISTENCE / SIGNAL MATURITY
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — SETUP PERSISTENCE / SIGNAL MATURITY")
print("=" * 90)

assert "exec_signals" in globals()
assert "_true_portfolio_stress" in globals()
assert "nested_folds" in globals()
assert "portfolio_dates" in globals()
assert "n_dates" in globals()

# Causal execution study:
# require evidence that the setup was already present on the prior
# completed bar. Entry timing remains the original NEXT-DAY OPEN.
# No future information is used for candidate construction.
maturity = exec_signals.copy()

prev_score = []
prev_macd = []
prev_trend = []
prev_early = []

for _, row in maturity.iterrows():
    sym = row["Symbol"]
    dt = pd.Timestamp(row["SignalDate"])
    df = canonical_data[sym]

    if dt not in df.index:
        prev_score.append(np.nan)
        prev_macd.append(False)
        prev_trend.append(False)
        prev_early.append(False)
        continue

    pos = df.index.get_loc(dt)
    if isinstance(pos, slice) or isinstance(pos, np.ndarray) or pos < 1:
        prev_score.append(np.nan)
        prev_macd.append(False)
        prev_trend.append(False)
        prev_early.append(False)
        continue

    p = df.iloc[pos - 1]
    prev_score.append(float(p["CANONICAL_SCORE"]))
    prev_macd.append(bool(p["CANONICAL_MACD_BULL"]))
    prev_trend.append(bool(p["CANONICAL_TREND_BULL"]))
    prev_early.append(bool(p["CANONICAL_EARLY_ENTRY"]))

maturity["PrevScore"] = prev_score
maturity["PrevMACDBull"] = prev_macd
maturity["PrevTrendBull"] = prev_trend
maturity["PrevEarlyEntry"] = prev_early

assert maturity[["PrevScore"]].notna().all().all()

MATURITY_FILTERS = {
    "BASE": maturity["SignalScore"] >= 4,
    "PREV_SCORE3": (
        (maturity["SignalScore"] >= 4)
        & (maturity["PrevScore"] >= 3)
    ),
    "PREV_SCORE4": (
        (maturity["SignalScore"] >= 4)
        & (maturity["PrevScore"] >= 4)
    ),
    "PREV_MACD": (
        (maturity["SignalScore"] >= 4)
        & maturity["PrevMACDBull"]
    ),
    "PREV_TREND": (
        (maturity["SignalScore"] >= 4)
        & maturity["PrevTrendBull"]
    ),
    "PREV_MACD_TREND": (
        (maturity["SignalScore"] >= 4)
        & maturity["PrevMACDBull"]
        & maturity["PrevTrendBull"]
    ),
}

MATURITY_SIGNALS = {
    name: maturity.loc[mask].copy()
    for name, mask in MATURITY_FILTERS.items()
}

print("\nFILTER COUNTS")
for name, df in MATURITY_SIGNALS.items():
    print(f"{name:<20} signals={len(df):>5}")

FRICTION_MATURITY = {
    "commission_pct": 0.0015,
    "slippage_pct": 0.0010,
    "gap_extra_pct": 0.0025,
}

rows = []

for fold_no, (a, b, c, d) in enumerate(nested_folds, 1):
    valid_start = portfolio_dates[int(n_dates * b)]
    valid_end = portfolio_dates[int(n_dates * c) - 1]
    test_start = portfolio_dates[int(n_dates * c)]
    test_end = portfolio_dates[int(n_dates * d) - 1]

    validation = []
    for name, sigs in MATURITY_SIGNALS.items():
        val = _true_portfolio_stress(
            sigs, valid_start, valid_end,
            risk_pct=0.020, target2_pct=0.15,
            max_hold_days=20, max_positions=5,
            **FRICTION_MATURITY,
        )
        validation.append((name, val))

    eligible = [x for x in validation if x[1]["Trades"] >= 10]
    if not eligible:
        eligible = validation

    chosen_name, chosen_val = max(
        eligible, key=lambda x: x[1]["PnL"]
    )

    test = _true_portfolio_stress(
        MATURITY_SIGNALS[chosen_name], test_start, test_end,
        risk_pct=0.020, target2_pct=0.15,
        max_hold_days=20, max_positions=5,
        **FRICTION_MATURITY,
    )

    base_test = _true_portfolio_stress(
        MATURITY_SIGNALS["BASE"], test_start, test_end,
        risk_pct=0.020, target2_pct=0.15,
        max_hold_days=20, max_positions=5,
        **FRICTION_MATURITY,
    )

    rows.append({
        "Fold": fold_no,
        "Selected": chosen_name,
        "ValidPnL": chosen_val["PnL"],
        "ValidPF": chosen_val["PF"],
        "TestPnL": test["PnL"],
        "TestTrades": test["Trades"],
        "TestWinRate": test["WinRate"],
        "TestMaxDD": test["MaxDD"],
        "TestPF": test["PF"],
        "BaseTestPnL": base_test["PnL"],
        "DeltaVsBase": test["PnL"] - base_test["PnL"],
    })

maturity_nested_df = pd.DataFrame(rows)
display(maturity_nested_df)

positive = int((maturity_nested_df.TestPnL > 0).sum())
selected_total = float(maturity_nested_df.TestPnL.sum())
base_total = float(maturity_nested_df.BaseTestPnL.sum())
delta = selected_total - base_total
robust = bool((maturity_nested_df.TestPF.fillna(0) > 1.0).all())

print("\nSETUP MATURITY NESTED SUMMARY")
print(f"Positive OOS folds: {positive}/{len(maturity_nested_df)}")
print(f"Selected net OOS P&L: {selected_total:,.2f} THB")
print(f"Frozen BASE net OOS P&L: {base_total:,.2f} THB")
print(f"Delta vs BASE: {delta:,.2f} THB")
print(f"Selected filters: {', '.join(maturity_nested_df.Selected.tolist())}")

if positive >= 2 and delta > 0 and robust:
    maturity_decision = "SETUP_MATURITY_CANDIDATE"
else:
    maturity_decision = "KEEP_CURRENT_ENTRY_RULES"

print("RESEARCH DECISION:", maturity_decision)
print("PRODUCTION CONFIGURATION UNCHANGED")
print("SETUP PERSISTENCE / SIGNAL MATURITY STUDY COMPLETE")



In [ ]:
# ============================================================
# CELL 12.27 — RESEARCH ONLY: SIGNAL COOLDOWN / TURNOVER CONTROL
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — SIGNAL COOLDOWN / TURNOVER CONTROL")
print("=" * 90)

assert "exec_signals" in globals()
assert "_true_portfolio_stress" in globals()
assert "nested_folds" in globals()
assert "portfolio_dates" in globals()
assert "n_dates" in globals()

# Causal execution study:
# suppress repeated signals for the same symbol when they occur too close
# together. The filter uses only SignalDate ordering and never future prices.
# This is intended to reduce unnecessary turnover and therefore execution
# friction, which has been the dominant weakness in the current OOS audit.
def _apply_signal_cooldown(signals, trading_days):
    if trading_days <= 0:
        return signals.copy()

    kept = []
    for sym, grp in signals.sort_values(["Symbol", "SignalDate"]).groupby("Symbol"):
        last_pos = None
        idx = pd.DatetimeIndex(canonical_data[sym].index)
        for _, row in grp.iterrows():
            dt = pd.Timestamp(row["SignalDate"])
            pos = idx.searchsorted(dt, side="left")
            if pos >= len(idx):
                continue
            if last_pos is None or (pos - last_pos) >= trading_days:
                kept.append(row)
                last_pos = pos

    if not kept:
        return signals.iloc[0:0].copy()
    return pd.DataFrame(kept).reset_index(drop=True)

COOLDOWN_DAYS = [0, 2, 3, 5, 8, 10]
COOLDOWN_SIGNALS = {
    f"COOLDOWN_{d}D": _apply_signal_cooldown(exec_signals, d)
    for d in COOLDOWN_DAYS
}

print("\nSIGNAL COUNTS")
for name, df in COOLDOWN_SIGNALS.items():
    print(f"{name:<18} signals={len(df):>5}")

FRICTION_COOLDOWN = {
    "commission_pct": 0.0015,
    "slippage_pct": 0.0010,
    "gap_extra_pct": 0.0025,
}

rows = []

for fold_no, (a, b, c, d) in enumerate(nested_folds, 1):
    valid_start = portfolio_dates[int(n_dates * b)]
    valid_end = portfolio_dates[int(n_dates * c) - 1]
    test_start = portfolio_dates[int(n_dates * c)]
    test_end = portfolio_dates[int(n_dates * d) - 1]

    validation = []
    for name, sigs in COOLDOWN_SIGNALS.items():
        val = _true_portfolio_stress(
            sigs, valid_start, valid_end,
            risk_pct=0.020, target2_pct=0.15,
            max_hold_days=20, max_positions=5,
            **FRICTION_COOLDOWN,
        )
        validation.append((name, val))

    eligible = [x for x in validation if x[1]["Trades"] >= 10]
    if not eligible:
        eligible = validation

    chosen_name, chosen_val = max(
        eligible, key=lambda x: x[1]["PnL"]
    )

    test = _true_portfolio_stress(
        COOLDOWN_SIGNALS[chosen_name], test_start, test_end,
        risk_pct=0.020, target2_pct=0.15,
        max_hold_days=20, max_positions=5,
        **FRICTION_COOLDOWN,
    )
    base_test = _true_portfolio_stress(
        COOLDOWN_SIGNALS["COOLDOWN_0D"], test_start, test_end,
        risk_pct=0.020, target2_pct=0.15,
        max_hold_days=20, max_positions=5,
        **FRICTION_COOLDOWN,
    )

    rows.append({
        "Fold": fold_no,
        "Selected": chosen_name,
        "ValidPnL": chosen_val["PnL"],
        "ValidPF": chosen_val["PF"],
        "TestPnL": test["PnL"],
        "TestTrades": test["Trades"],
        "TestWinRate": test["WinRate"],
        "TestMaxDD": test["MaxDD"],
        "TestPF": test["PF"],
        "BaseTestPnL": base_test["PnL"],
        "DeltaVsBase": test["PnL"] - base_test["PnL"],
    })

cooldown_nested_df = pd.DataFrame(rows)
display(cooldown_nested_df)

positive = int((cooldown_nested_df.TestPnL > 0).sum())
selected_total = float(cooldown_nested_df.TestPnL.sum())
base_total = float(cooldown_nested_df.BaseTestPnL.sum())
delta = selected_total - base_total
robust = bool((cooldown_nested_df.TestPF.fillna(0) > 1.0).all())

print("\nSIGNAL COOLDOWN NESTED SUMMARY")
print(f"Positive OOS folds: {positive}/{len(cooldown_nested_df)}")
print(f"Selected net OOS P&L: {selected_total:,.2f} THB")
print(f"Frozen BASE net OOS P&L: {base_total:,.2f} THB")
print(f"Delta vs BASE: {delta:,.2f} THB")
print(f"Selected cooldowns: {', '.join(cooldown_nested_df.Selected.tolist())}")

if positive >= 2 and delta > 0 and robust:
    cooldown_decision = "SIGNAL_COOLDOWN_CANDIDATE"
else:
    cooldown_decision = "KEEP_CURRENT_ENTRY_RULES"

print("RESEARCH DECISION:", cooldown_decision)
print("PRODUCTION CONFIGURATION UNCHANGED")
print("SIGNAL COOLDOWN / TURNOVER STUDY COMPLETE")


In [ ]:
# ============================================================
# CELL 12.28 — RESEARCH ONLY: ENTRY QUALITY + SIGNAL COOLDOWN
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — ENTRY QUALITY + SIGNAL COOLDOWN")
print("=" * 90)

assert "exec_signals" in globals()
assert "_true_portfolio_stress" in globals()
assert "nested_folds" in globals()
assert "portfolio_dates" in globals()
assert "n_dates" in globals()

# Causal composite study:
# combine a small number of already-supported entry-quality filters with
# signal cooldown. Both operate only on information available no later than
# SignalDate. No production configuration is modified.
base = exec_signals.copy()

def _cooldown_quality(signals, trading_days):
    return _apply_signal_cooldown(signals, trading_days)

QUALITY_FILTERS = {
    "BASE": lambda d: d["SignalScore"] >= 4,
    "SCORE5": lambda d: d["SignalScore"] >= 5,
    "SCORE4_VOL14": lambda d: (d["SignalScore"] >= 4) & (d["SignalVolumeRatio"] >= 1.4),
    "SCORE4_VOL16": lambda d: (d["SignalScore"] >= 4) & (d["SignalVolumeRatio"] >= 1.6),
    "SCORE4_TRENDBULL_VOL14": lambda d: (
        (d["SignalScore"] >= 4)
        & d["SignalTrendBull"]
        & (d["SignalVolumeRatio"] >= 1.4)
    ),
}

COMPOSITE_CANDIDATES = {}
for qname, qmask in QUALITY_FILTERS.items():
    qsig = base.loc[qmask(base)].copy()
    for cd in [0, 3, 5, 8]:
        name = f"{qname}_CD{cd}D"
        COMPOSITE_CANDIDATES[name] = _cooldown_quality(qsig, cd)

print("\nCANDIDATE COUNTS")
for name, sigs in COMPOSITE_CANDIDATES.items():
    print(f"{name:<30} signals={len(sigs):>5}")

FRICTION_COMPOSITE = {
    "commission_pct": 0.0015,
    "slippage_pct": 0.0010,
    "gap_extra_pct": 0.0025,
}

rows = []
for fold_no, (a, b, c, d) in enumerate(nested_folds, 1):
    valid_start = portfolio_dates[int(n_dates * b)]
    valid_end = portfolio_dates[int(n_dates * c) - 1]
    test_start = portfolio_dates[int(n_dates * c)]
    test_end = portfolio_dates[int(n_dates * d) - 1]

    validation = []
    for name, sigs in COMPOSITE_CANDIDATES.items():
        val = _true_portfolio_stress(
            sigs, valid_start, valid_end,
            risk_pct=0.020, target2_pct=0.15,
            max_hold_days=20, max_positions=5,
            **FRICTION_COMPOSITE,
        )
        validation.append((name, val))

    eligible = [x for x in validation if x[1]["Trades"] >= 10]
    if not eligible:
        eligible = validation

    chosen_name, chosen_val = max(eligible, key=lambda x: x[1]["PnL"])

    test = _true_portfolio_stress(
        COMPOSITE_CANDIDATES[chosen_name], test_start, test_end,
        risk_pct=0.020, target2_pct=0.15,
        max_hold_days=20, max_positions=5,
        **FRICTION_COMPOSITE,
    )
    base_test = _true_portfolio_stress(
        COMPOSITE_CANDIDATES["BASE_CD0D"], test_start, test_end,
        risk_pct=0.020, target2_pct=0.15,
        max_hold_days=20, max_positions=5,
        **FRICTION_COMPOSITE,
    )

    rows.append({
        "Fold": fold_no,
        "Selected": chosen_name,
        "ValidPnL": chosen_val["PnL"],
        "ValidPF": chosen_val["PF"],
        "TestPnL": test["PnL"],
        "TestTrades": test["Trades"],
        "TestWinRate": test["WinRate"],
        "TestMaxDD": test["MaxDD"],
        "TestPF": test["PF"],
        "BaseTestPnL": base_test["PnL"],
        "DeltaVsBase": test["PnL"] - base_test["PnL"],
    })

composite_cd_df = pd.DataFrame(rows)
display(composite_cd_df)

positive = int((composite_cd_df.TestPnL > 0).sum())
selected_total = float(composite_cd_df.TestPnL.sum())
base_total = float(composite_cd_df.BaseTestPnL.sum())
delta = selected_total - base_total
robust = bool((composite_cd_df.TestPF.fillna(0) > 1.0).all())

print("\nENTRY QUALITY + COOLDOWN NESTED SUMMARY")
print(f"Positive OOS folds: {positive}/{len(composite_cd_df)}")
print(f"Selected net OOS P&L: {selected_total:,.2f} THB")
print(f"Frozen BASE net OOS P&L: {base_total:,.2f} THB")
print(f"Delta vs BASE: {delta:,.2f} THB")
print(f"Selected candidates: {', '.join(composite_cd_df.Selected.tolist())}")

if positive >= 2 and delta > 0 and robust:
    composite_cd_decision = "COMPOSITE_ENTRY_CANDIDATE"
else:
    composite_cd_decision = "KEEP_CURRENT_ENTRY_RULES"

print("RESEARCH DECISION:", composite_cd_decision)
print("PRODUCTION CONFIGURATION UNCHANGED")
print("ENTRY QUALITY + SIGNAL COOLDOWN STUDY COMPLETE")


In [ ]:
# ============================================================
# CELL 12.29 — RESEARCH ONLY: ENTRY EFFICIENCY / ATR + VOLUME + COOLDOWN
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — ENTRY EFFICIENCY / ATR + VOLUME + COOLDOWN")
print("=" * 90)

assert "exec_signals" in globals()
assert "_true_portfolio_stress" in globals()
assert "_apply_signal_cooldown" in globals()
assert "nested_folds" in globals()
assert "portfolio_dates" in globals()
assert "n_dates" in globals()

EFFICIENCY_FILTERS = {
    "BASE": lambda d: d["SignalScore"] >= 4,
    "VOL14_ATR15": lambda d: (d["SignalScore"] >= 4) & (d["SignalVolumeRatio"] >= 1.4) & (d["SignalATRpct"] >= 0.015),
    "VOL14_ATR20": lambda d: (d["SignalScore"] >= 4) & (d["SignalVolumeRatio"] >= 1.4) & (d["SignalATRpct"] >= 0.020),
    "VOL16_ATR15": lambda d: (d["SignalScore"] >= 4) & (d["SignalVolumeRatio"] >= 1.6) & (d["SignalATRpct"] >= 0.015),
    "VOL16_ATR20": lambda d: (d["SignalScore"] >= 4) & (d["SignalVolumeRatio"] >= 1.6) & (d["SignalATRpct"] >= 0.020),
}

EFFICIENCY_CANDIDATES = {}
for qname, qmask in EFFICIENCY_FILTERS.items():
    qsig = exec_signals.loc[qmask(exec_signals)].copy()
    for cd in [0, 3, 5]:
        EFFICIENCY_CANDIDATES[f"{qname}_CD{cd}D"] = _apply_signal_cooldown(qsig, cd)

print("\nCANDIDATE COUNTS")
for name, sigs in EFFICIENCY_CANDIDATES.items():
    print(f"{name:<24} signals={len(sigs):>5}")

FRICTION_EFFICIENCY = {"commission_pct": 0.0015, "slippage_pct": 0.0010, "gap_extra_pct": 0.0025}

rows = []
for fold_no, (a, b, c, d) in enumerate(nested_folds, 1):
    valid_start = portfolio_dates[int(n_dates * b)]
    valid_end = portfolio_dates[int(n_dates * c) - 1]
    test_start = portfolio_dates[int(n_dates * c)]
    test_end = portfolio_dates[int(n_dates * d) - 1]

    validation = []
    for name, sigs in EFFICIENCY_CANDIDATES.items():
        val = _true_portfolio_stress(sigs, valid_start, valid_end, risk_pct=0.020, target2_pct=0.15, max_hold_days=20, max_positions=5, **FRICTION_EFFICIENCY)
        validation.append((name, val))

    eligible = [x for x in validation if x[1]["Trades"] >= 10]
    if not eligible:
        eligible = validation
    chosen_name, chosen_val = max(eligible, key=lambda x: x[1]["PnL"])

    test = _true_portfolio_stress(EFFICIENCY_CANDIDATES[chosen_name], test_start, test_end, risk_pct=0.020, target2_pct=0.15, max_hold_days=20, max_positions=5, **FRICTION_EFFICIENCY)
    base_test = _true_portfolio_stress(EFFICIENCY_CANDIDATES["BASE_CD0D"], test_start, test_end, risk_pct=0.020, target2_pct=0.15, max_hold_days=20, max_positions=5, **FRICTION_EFFICIENCY)

    rows.append({"Fold":fold_no,"Selected":chosen_name,"ValidPnL":chosen_val["PnL"],"ValidPF":chosen_val["PF"],"TestPnL":test["PnL"],"TestTrades":test["Trades"],"TestWinRate":test["WinRate"],"TestMaxDD":test["MaxDD"],"TestPF":test["PF"],"BaseTestPnL":base_test["PnL"],"DeltaVsBase":test["PnL"]-base_test["PnL"]})

efficiency_nested_df = pd.DataFrame(rows)
display(efficiency_nested_df)
positive = int((efficiency_nested_df.TestPnL > 0).sum())
selected_total = float(efficiency_nested_df.TestPnL.sum())
base_total = float(efficiency_nested_df.BaseTestPnL.sum())
delta = selected_total - base_total
robust = bool((efficiency_nested_df.TestPF.fillna(0) > 1.0).all())

print("\nENTRY EFFICIENCY NESTED SUMMARY")
print(f"Positive OOS folds: {positive}/{len(efficiency_nested_df)}")
print(f"Selected net OOS P&L: {selected_total:,.2f} THB")
print(f"Frozen BASE net OOS P&L: {base_total:,.2f} THB")
print(f"Delta vs BASE: {delta:,.2f} THB")
print(f"Selected candidates: {', '.join(efficiency_nested_df.Selected.tolist())}")

efficiency_decision = "ENTRY_EFFICIENCY_CANDIDATE" if positive >= 2 and delta > 0 and robust else "KEEP_CURRENT_ENTRY_RULES"
print("RESEARCH DECISION:", efficiency_decision)
print("PRODUCTION CONFIGURATION UNCHANGED")
print("ENTRY EFFICIENCY / ATR + VOLUME + COOLDOWN STUDY COMPLETE")



In [ ]:
# ============================================================
# CELL 12.30 — RESEARCH ONLY: DAILY SIGNAL RANKING / PORTFOLIO SELECTIVITY
# ============================================================
print("=" * 90)
print("RESEARCH ONLY — DAILY SIGNAL RANKING / PORTFOLIO SELECTIVITY")
print("=" * 90)

assert "exec_signals" in globals()
assert "_true_portfolio_stress" in globals()
assert "nested_folds" in globals()
assert "portfolio_dates" in globals()
assert "n_dates" in globals()

# Causal study:
# When several valid signals occur on the same signal date, keep only the
# strongest ranked opportunities. Ranking uses signal-day information only.
# Production configuration is NOT modified.

rank_base = exec_signals.loc[exec_signals["SignalScore"] >= 4].copy()

# Composite ranking deliberately uses only already-available signal features.
rank_base["EntryRankScore"] = (
    rank_base["SignalScore"].astype(float) * 10.0
    + rank_base["SignalVolumeRatio"].clip(lower=0, upper=3).astype(float) * 2.0
    + rank_base["SignalATRpct"].clip(lower=0, upper=0.10).astype(float) * 20.0
    + rank_base["SignalTrendBull"].astype(float) * 1.0
    + rank_base["SignalWeeklyConfirm"].astype(float) * 0.5
)

RANK_CANDIDATES = {}
for top_n in [1, 2, 3, 5]:
    ranked = rank_base.sort_values(
        ["SignalDate", "EntryRankScore", "SignalScore"],
        ascending=[True, False, False],
        kind="mergesort",
    ).copy()
    ranked = ranked.groupby("SignalDate", group_keys=False).head(top_n).copy()
    RANK_CANDIDATES[f"TOP{top_n}"] = ranked

print("\nCANDIDATE COUNTS")
for name, sigs in RANK_CANDIDATES.items():
    print(f"{name:<12} signals={len(sigs):>5}")

FRICTION_RANK = {
    "commission_pct": 0.0015,
    "slippage_pct": 0.0010,
    "gap_extra_pct": 0.0025,
}

rows = []
for fold_no, (a, b, c, d) in enumerate(nested_folds, 1):
    valid_start = portfolio_dates[int(n_dates * b)]
    valid_end = portfolio_dates[int(n_dates * c) - 1]
    test_start = portfolio_dates[int(n_dates * c)]
    test_end = portfolio_dates[int(n_dates * d) - 1]

    validation = []
    for name, sigs in RANK_CANDIDATES.items():
        val = _true_portfolio_stress(
            sigs, valid_start, valid_end,
            risk_pct=0.020, target2_pct=0.15,
            max_hold_days=20, max_positions=5,
            **FRICTION_RANK
        )
        validation.append((name, val))

    eligible = [x for x in validation if x[1]["Trades"] >= 10]
    if not eligible:
        eligible = validation
    chosen_name, chosen_val = max(eligible, key=lambda x: x[1]["PnL"])

    test = _true_portfolio_stress(
        RANK_CANDIDATES[chosen_name], test_start, test_end,
        risk_pct=0.020, target2_pct=0.15,
        max_hold_days=20, max_positions=5,
        **FRICTION_RANK
    )
    base_test = _true_portfolio_stress(
        RANK_CANDIDATES["TOP5"], test_start, test_end,
        risk_pct=0.020, target2_pct=0.15,
        max_hold_days=20, max_positions=5,
        **FRICTION_RANK
    )

    rows.append({
        "Fold": fold_no,
        "Selected": chosen_name,
        "ValidPnL": chosen_val["PnL"],
        "ValidPF": chosen_val["PF"],
        "TestPnL": test["PnL"],
        "TestTrades": test["Trades"],
        "TestWinRate": test["WinRate"],
        "TestMaxDD": test["MaxDD"],
        "TestPF": test["PF"],
        "BaseTestPnL": base_test["PnL"],
        "DeltaVsBase": test["PnL"] - base_test["PnL"],
    })

ranking_nested_df = pd.DataFrame(rows)
display(ranking_nested_df)

positive = int((ranking_nested_df.TestPnL > 0).sum())
selected_total = float(ranking_nested_df.TestPnL.sum())
base_total = float(ranking_nested_df.BaseTestPnL.sum())
delta = selected_total - base_total
robust = bool((ranking_nested_df.TestPF.fillna(0) > 1.0).all())

print("\nDAILY SIGNAL RANKING NESTED SUMMARY")
print(f"Positive OOS folds: {positive}/{len(ranking_nested_df)}")
print(f"Selected net OOS P&L: {selected_total:,.2f} THB")
print(f"Frozen TOP5 net OOS P&L: {base_total:,.2f} THB")
print(f"Delta vs TOP5: {delta:,.2f} THB")
print(f"Selected candidates: {', '.join(ranking_nested_df.Selected.tolist())}")

ranking_decision = (
    "DAILY_RANKING_CANDIDATE"
    if positive >= 2 and delta > 0 and robust
    else "KEEP_CURRENT_ENTRY_RULES"
)
print("RESEARCH DECISION:", ranking_decision)
print("PRODUCTION CONFIGURATION UNCHANGED")
print("DAILY SIGNAL RANKING / PORTFOLIO SELECTIVITY STUDY COMPLETE")

